# ARC-Jeni-2026 — Hybrid DSA + Program Synthesis + Chakra-Ensemble LLM

**Competition:** ARC Prize 2026 – ARC-AGI-2  
**Author:** joyjeni  
**Repo:** https://github.com/joyjeni/arc-agi2-hybrid-solver  

## Architecture (3-tier)
| Tier | Method | Time Budget | Expected Coverage |
|------|--------|-------------|-------------------|
| 1 | DSA + Vedic deterministic solvers | 300 s total / 5 s task | 5–10% |
| 2 | LLM Program Synthesis (write+run Python) | 40% remaining / 45 s task | 15–25% |
| 3 | Chakra-8 Ensemble direct prediction | remainder / 90 s task | 10–20% |

**Model:** Qwen3-4B fine-tuned on ARC grids (`qwen3_4b_grids15_sft139`)  
**Hard time control:** Tier-1 ThreadPoolExecutor wall-clock; Tier-2/3 `signal.SIGALRM` per task  
**Total wall-clock target:** ≤5 h (18 000 s)

In [ ]:
# ── Dependency check (internet DISABLED — all packages pre-installed on Kaggle) ─
import yaml  # pyyaml
print('pyyaml OK:', yaml.__version__)


In [ ]:
# ── Setup & Imports ─────────────────────────────────────────────────────────
import os, sys, time, json, logging, signal, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, wait as _fut_wait

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('arc_jeni')

IS_KAGGLE = Path('/kaggle').exists()
IS_RERUN  = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))

DATA_DIR   = Path('/kaggle/input/arc-prize-2026-arc-agi-2') if IS_KAGGLE else Path('data')
MODEL_DIR  = Path('/kaggle/input/qwen3-4b-grids15-sft139/transformers/sft139/1') if IS_KAGGLE else None
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

T0 = time.monotonic()   # competition start timestamp

print(f'IS_KAGGLE={IS_KAGGLE}  IS_RERUN={IS_RERUN}')
print(f'DATA_DIR={DATA_DIR}  MODEL_DIR={MODEL_DIR}')
# ── Budget constants & time helper ──────────────────────────────────────────
TOTAL_BUDGET_S    = 9 * 3600 - 120   # 9h Kaggle limit minus 2-min safety margin
COMPETITION_LIMIT_S = 9 * 3600          # full 9-hour competition window
T1_BUDGET_S       = 300               # tier-1 DSA pass budget (5 min)
T1_PER_TASK_S     = 3.0               # per-task limit for tier-1
T2_SHARE          = 0.50              # fraction of remaining budget for tier-2
T2_MAX_BUDGET_S   = 4 * 3600         # hard cap for tier-2 total
T2_MAX_PER_TASK_S = 120              # max 2 min per task in tier-2
T2_MIN_PER_TASK_S = 5                # abort tier-2 if fair share drops below 5s
T3_MAX_PER_TASK_S = 60               # max 1 min per task in tier-3
T3_MIN_PER_TASK_S = 10               # abort tier-3 if fair share drops below 10s

def _remaining() -> float:
    """Seconds remaining in the competition budget."""
    return max(0.0, TOTAL_BUDGET_S - (time.monotonic() - T0))


In [ ]:
# Cell 4: Embed source files from zlib+base64 bundle
import json, zlib, base64, os, sys, textwrap

BUNDLE_B64 = (
    "..."
)

_raw = zlib.decompress(base64.b64decode(BUNDLE_B64))
_bundle = json.loads(_raw.decode('utf-8'))

# Write files to /kaggle/working/src/
for rel_path, content in _bundle.items():
    abs_path = os.path.join('/kaggle/working', rel_path)
    os.makedirs(os.path.dirname(abs_path), exist_ok=True)
    with open(abs_path, 'w', encoding='utf-8') as f:
        f.write(content)

# Add to sys.path
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

print(f'Extracted {len(_bundle)} source files')


In [ ]:
# ── Embed all source files (zlib+base64, fits Kaggle 1MB limit) ──────────────
import os, sys, base64 as _b64, zlib as _zl

os.makedirs('/kaggle/working/arc_src/src', exist_ok=True)
os.makedirs('/kaggle/working/arc_src/src/core', exist_ok=True)
os.makedirs('/kaggle/working/arc_src/src/llm', exist_ok=True)
os.makedirs('/kaggle/working/arc_src/src/solvers', exist_ok=True)

_arc_src = '/kaggle/working/arc_src'
if _arc_src not in sys.path:
    sys.path.insert(0, _arc_src)

# src/__init__.py
open('/kaggle/working/arc_src/src/__init__.py','wb').write(_zl.decompress(_b64.b64decode('eNpTUlJyLErW9UrNy9Q1MjAys1LwqEwqykxRcAl2VNBWCCjKTy9KzFUIrswryUgtziwGivn4+CoU5+eUpRYppOUXKTgGOes6unvqGukpKSlxccXHAyWKM/Pz4uMVbBWUDPUM9AyUgKKJpSUZ+UUQwaz8yiyghUpcAEdFJYE=')))
# src/core/__init__.py
open('/kaggle/working/arc_src/src/core/__init__.py','wb').write(_zl.decompress(_b64.b64decode('eNptUj1vhDAM3fkVEVMrRVXXDl3aoeup162qohwYzrqQ0Dhw7b8vDhcIqCz4PX/EfnbjXSce9NB2YIMO6KzArnc+CMD2HNSIcCUp0I7gCSIsmpjTeqxV5WqoUsZdIabvpKtL691g2Wucl5GNpjojBdd63SWy/1VcZ4aNwV6dM3uc7fhSjR1lEL4HbTIcnArwE2bGu2kUeHqcEfPsXx8KXlvqHYEs7m/ToA3gG10BpWleNMHRmWluKd44VRw0TvbBQ40VKyXF7H8HGkyQ4kPT5dVZfu9WlYZTh0SZqqcBTa1WXgqqnIcNM2qD9TTAhrx63DBFoZQ2RinxLD7jVCV3WUpRcp/8z/qJ9NI4o7x1xuu45SxSmevKEbmOjJe1LCAuJaUvy2V32gjb85oXa4y100bW7M29cMz+sFJodqgclp9qCtmrznH/qMz0Xmfm9iuayn4Vf/6VFUQ=')))
# src/core/augmentation.py
open('/kaggle/working/arc_src/src/core/augmentation.py','wb').write(_zl.decompress(_b64.b64decode('eNqNVcmOnDAQvfMVpZ4LJHSrt9mQeq5RriPlECGCGDC01WCQMTNpjUbKMR+QQ74vX5LyAriXWZC6ZZerXr16uIrJZPKF0wySrqgIE4mgNYN/v/7ADbT7qiKC7+GRkqcWPkNalzWHhvCq047tbDKZOE7O6wriOO9Ex0kcA62amgtIGKuNn/FBgJKkOjJ5SHvHr4LwRNTceM0KJBSndUYGD0nRx/BmHxdqmZe0ibc+cJmB3M4dx8lIDoQWWxErvq50DFSkB9M7KGkrQrmLAgfwQeb3BAkzSMoSq83olmQ8KacFr7sGBE9Ym9e8aqHOIQGJpqqVsQVsRjIqkafsfIEHPSW3MMalbeQLY10dWJfGqhmFBVa2wN8Sf6u+WgQclggzrpfWeuVFRgvKHglviVLDlX8xzX4GaBY+HGkjF4Mq31hWg9gSoFOxVe9+FGNQ4EIdxDssos9savnh7qZrD7l6gBGww4TwvPYv/Sv/+sXEGmoBjEFTFebBv99/DSKAaxa0hZaU+dSEaalwE1dJgwSe1V4+8wDKpHrI8HUFUPiDfWHbR9FPFkgapRyeC+O7vJ4PUMu3oYoDgFOoxc0ItToLVZwHOIW6HZHWNpJ1WeA9JCOwre4AevlB0YZ8WjxLNOtlDqhXb6NaYP4r+p1DvT6LOoL570kJx6gvdj+auxb2XRSZnteNljRNuY/VcIyt4WgNIF8NzUDPIGzA6LTr7om8zAqkha6lrMChY6GpWHClJazLLNow8uQN7djPjVCdp5FqvVS2Hq+f9A4Xci9J9RNiT0mZHRAnWdwklLeuQsW2p0wbDPeMpiLSfcVUSKsGCjbh2ndUUf0sD0XXlCS0osbqo3EEf5cUwDVfH0xv5fTt+vU0wTmU6e+I3EjJGlSqHXQwnwvtpixG0Y3K7uJBQdzF3NMzpCWEBfgvNNsI3XDj6kOZL1YSqiBTL3wCDA+GmyeN6msg04RBNBxoDrN22+V5SdSL84bDHdljkEp6dEJzdYhpFTnHvq5pzQRl3dii0mWWZJmLISMEiong4UhFViIVlaj2Oz0Ax6gZyklY5j47x40yoazpxCR49a5LwNB4RfrFef4pTN2Jj+AYt/NAL2Ol6gJL5trRFrEkzJXyeHC3Ge7qAc4DJ8nO+Q9vD6tr')))
# src/core/interfaces.py
open('/kaggle/working/arc_src/src/core/interfaces.py','wb').write(_zl.decompress(_b64.b64decode('eNqlVsFu4zYQvesrBt6LXdiCk/TkhRebuLtFgbRNY98WhkBLI5uIRAoktYmAPfQj+oX9kg5JiZJsby+rgxyRw5k3b94MM5lMNlIhcGFQ5SxFDUxkwA7aKJYaODCNkBZMa9TxZDKJolzJEpIkr02tMEmAl5VUhk4JaZjhUugoatcML9Hbs0PaGd4/bObBf4nmJDNvkzHD2kidbViaQ86xaA1NU3FxDP5EE0W/Kp7BGgquzRf3onz2++iJcUXLGU/NFwo4B2u3j6LoY/AcuTfsmH7ZSCLhzawioIdy/a0sa8MOBRHgdyCXCqRAuH/egKETjhFrbT8Snq2AovgFxbhIKoqvVx6WxbL3e6hNwkVVm27Pw7J772Br2BEXd3CiFGj/QMaLY42E8UhWGhwFf72iuFv8/AC73W7uIFWonGNwjuPOGVIJG1C1SLT1ezedvW8j3IKWxVdUGmoqsTkhvRlV3xgsK5PcQs6K4sDSF+2d+fNJC6uHDd/gDwtg7X6ucfuk0FaApBGo3b3KRRsJqrAd+B0k0jHc4bpZuSKO1m4Haz6pRLASXTUI16QWL0K+Cu+IapnzDEVK+3khmSGLZbx0e1iwSmOWlPpyj6TKbGargZxIe3sycdqcZpizujAJdZGRqllbs9k1PrYO4jNqsg6MfK6LApRb62kgVcG0ZA2c2FeEkvZ4VQzp0bPvS7DnNUgwrOwBvEAupdOzSL4OUhaU4GdWaPRROKqVnRaWGbj6vIOb9S/b+zncrp+UPCpWbhthTnO4Wz8+/k6EeBYeaLJ4JqY0EmaBiC21jsFj048k+PfvfwDJsGmrS0xYtCVxUSJBMSeuXY/akeKnlHX2sVKScjON/zqbOXaNSgZWKVONRT6DxQdL3griOL7mwVpXikvFTdOfIJgee4v/Ub6iVR31HDUr9b+J4U8CTQKlykmPk1jFN0xrp3qpMlShjvZRSMNVwM1yGf0vdkeHgzKH1LythmPMgRtKrW3UEdjw933bjATQObUDwY+4YPLsQI3lCzwPDWXm1D80RGyQ+GqQwKvFntjbIUt+LAWzJKato5jKlCeprK1oprMBk+7cGmyE2AejML1B2/JkMb3iBxYUYgY/2VIswxlKWg26t3ts21akh3YzHjZgdN4lVdwPGwrefpwrwHsKPfNJaCzpQmr7pm+hvn02kq5FzQ1NtxLVke5S76O9OMIMSU+8yLoroO8ZVxm6nLhJkq4q1lKhaIdIH3Q/69Ny9CadqeWb7mbMpt3KHF6wWResPGQMaB7puOuk2fda7bIxz9mZjPkYpPBjqrKFbCcNVXOc2riSRjWXpe0153zEI6WPxHchKPfPVyufdgZH12bsWB7DHXxLkfr4k/sJV+5IefYmOnPkru7/AP8fHoo=')))
# src/core/grid_codec.py
open('/kaggle/working/arc_src/src/core/grid_codec.py','wb').write(_zl.decompress(_b64.b64decode('eNqNVs1u4zYQvusppuphpa0sONmfdgW4QNsN9pJsgN1TYQgCJdE2E4lUKSqOtwnQh+gT9kk6Q1OyZCTx6mBRJOebb2Y+Du37/ictSuCyUKWQa2CyhJK7j86IShjB29j3fc9baVVDlq0602meZSDqRmmDJlIZZoSSredZtAVUojVL+yOkSVPP++P68vpL9vm3q4uvuPy3B/jME/DzihW3fgRndtxxHJ7jUPMSR29wtNacSz+yBm/xe8erSm1x8Z1dZDscvsdhIyTh/IxDpZlcc2fzC06wb0gYFz/guGZaKQJ8dJy+/nn1+/XlhFTsCDHHJndcCnwTh9JhEwXu/K+c8zW+yefG+RPW1d5JZj2Sq7sEbmGlNNxGcAdCwphLLAyv2yB89Dyv5CtYY04zozLD701AHwlQmiOoVckTaI1GSF92Ndei8EOY/UpziaWIhbug2nJgcD77aLGAtfhVqLphhaGtVGsic3l5BQ0WuTE4E3sW4Apt2z0WPc4LwH///Auv5nAG5/AAb+AtvHsFAbJlXWUi6CSrc7HuVNeGg227q3NVgbONkUOOtgWg/NDW8Ymg4sZwPSPS5cH4jutctdwZW9kACQZQKogSxzFibMR6g9lE80Zzw3KS7y7sE2HfWm1bTNcytV8UNc5QBWxiB3diZbMLi1FmD6s9UMyahssy8FEzN0rIAJMZFKGFLQgUN4WHGHg1ht2n43tQx+JYFunz8C3KAX7sc/W9wPZUPoOLaey0BB9T7GwIKHTKJEWSMil3AX1YNb4kTNsfHuCzknwQ6EfuBGrF6fSYU4GNcrLFmtqeEsMXS6i1CKAkrJio8HTHfX2N3h0y2hdbx4TaBPvKaAqR2PazcdugUAL/wQ/TwfZw0A5yOZIMwU/Lh/XFZkgL03l6CiXxXHXTshh1yyVxRBNHIzxGPCHEnmxfXWq5gdmHamyo1kc6xbVieRln3LSWJn0B0IU9PUIj+VCxhnnUJ6tEyQyd3cJgr+4qpoflLV0fXNpGt5xPnTC5C2hNh/DDAncO1aTN4WnXbo522zl+X/DGwIV94QWWeE9Zj7pwKbAxH5RhJW26puKU9Ihkmg6y3isVmxLHroSLW1GaTTgI1XnoQw2jSdQUrT0OVCeYjznwvzpWBay/BPIRlVypKhmjM9JN7qwL1ez2R3U9svk0FM3ZLFGKyySd9MbUQWi66fmH+XEODiCbaFLBo7AmfpZ2dgMzvEVmcJMuxd7pjT1bdIcHm3A/JQ5T27Ans6pEk22epzKJJ5mdpcftfgJ0dxLI0rVAfftDRm2DnfY0B2pe1DrDMYdv2Hxe2yylQ4kqpbMNblb436Y+xi1FYY6VRpuToxX6i/H48gU37vXTg0OAdBss7ChecxMUEcxD+AnOxjHRqqNdq9ZkeIHXSmY2hGPiSMqxRdingpwoo2b3Aerolu8WG/LfXzd0J6y16mR5wkuP8ySt0PsfidMrSA==')))
# src/core/submission.py
open('/kaggle/working/arc_src/src/core/submission.py','wb').write(_zl.decompress(_b64.b64decode('eNqlV11u4zYQftcppgoWkFBFdfLUCusF2n0timJ3gT64gkDLtKNEJg2SWsdwDfQQBXqg3qQn6QxJ69dON60QJIo4P99w/sMw/Ngst5XWlRSwbKp6xRX8/fsfsFeV4RpKud1xUxk8vl1LtWUGdMuQPmr8FYZhEKyV3EJRrBvTKF4UUG13UhlgQkjDiFsHgf9GTOf3Wm42ldg49h0zD3W1PPP+jP96wWklDFdrViIif/pR1p+5+sB1U5sEPjH99F4i0bMJAhKKVszP0tMNNz/ab1FRCLZFfHEQBCu+dhYXnUVRAPgYlKYzWFWlWWijBuLzxJIoq3hA1EfkqdasrpesfCoqsWtMBkspa8T1STU8CWK4fWfZM0tL10h/fyBIwOwJqmGrA+DF21tLV812B+iD8gHNAvPA+/4B558ssGLoObZvKJ6MKqpVmMGi9xmJQmYM3+5McUdnizRN8zyB9ut99/WUDDjxG0yeG5CCAxdGHWCHXsAgMmDN7/Hm7fvJof1ebXTWfvT330H02LO+I04teeuLCXnfJR392CvV2nkEL3N3cFhho6oV7B+4ACFhpzh5g+6YP1fa6HTgsS58XDigh4/eLvKcR4PizTNKd9almF1bHcXZyArk9eZQ1EaeNx5cTUF40NwaoSxIYY5ci7zzO2mtEnv1zkZSy0Wz5YoZHiGOtDvTPQz04G14KEysoIK3UHMRuU9pdxNjNnrotDWgT7uo8gkxu0NSoknb+JvS3I9p7gc0vEa0I3deVtTZe1nJlXNeaz6VeAOE5QB3f/15B8sadV9WuljM8vyyPnc0OOtcm7LdjotVNExNdjfKSnZ/ijsRXRQufNRQWHRCA1+3sECLHrEvhJ9ZXa0wOPq1EF9dQCcvV0RbyGww4lHeVrMPThUdgGw12CRSSioNkbvFuTuK03M+ueOsE+nCe5JPnW0U34h2mlSUcnOfcRfzCePHpqWGn7BuDV3tcJydsQ7PdeWUQSOehNy7XA7jAVeJt1KJhrcfRWFr4Nzm0Tj3+kDovLMphq/mnvclWJPwGuDkzzteGszKo5N06tUyncBGGjiO1J7Cgch/Mc62psRl/qDI9CQO4RPHE6c6C1EvvgexfaG24P0QF44TxEmCpzQvOm1xfDzlGdjgxuZ5RGmnkeuuWnl+bFdwFWmB/PkllBYhqtCGiZJHxJLYUI4BLadT+vRfwKcWc2ZlMCvytfg9PCyYUQ+iOuMj1yi6XkIY/x+ISu61VUWC9Wth7mc+WwgHFsop+wDpZaA+oZTNo/3sMtFrLl3cKswlJjZNzdQVkwhXSbjUdX3eCdEM3s6RGH99F1+n/oKMf6ECDEwoeV3DsTyBbGxFVmgLD18UF/ebhoPhG4bdDyaTc9cy7L800WeAJRx+s+M8ujUcrQ+hozz3n9GMPJrGUQyVaTymP36E7go3NpBfCNZ4RUlA7qjiYdwf2j6ENWBdKW3atlN1XdCOPk5v0HOCPveqaaP0LXJQz4lj6Fi3maR7pgTqj/q716Q9Zr8KgDeaCiO9hemjrEREMhfZPeaEU0W3GtE1x6nzCHXkqF0WNGGLE6zLpVyhynnYmPXttz56PZxKrGWENyfR8DfYlt9422PUTaITm0gk6Lw26VKqa6OClnVj24v7YF20riUbrTnvcXFpUCF/ZqW5tTsNsLJsFCsPwDaMqhNmt2zIE6rB2GkFp26Uaf9vF592ARhsOzhrUZWgpaZ7o90lh24nkOoqt2VpOZAgogZS81vfrwYo44GRBpdfCmZNewh1jtl0itmYwo5INMK0Fk4GGTfmzO2U0xtk0KRedfRz/2Y073sNoyLjsH09h7vx+F/BO1d+L7Xvq4X7hvRu2QGWHIsD1So7ETDamYxy/WqiqdcnDRZ6/PG9aKoUrfDt15J+wZje5wgu7CnWvPFqgqhse++NJjh/zltheMXD8/vB+RSF93170ecJ3H3+xvsB1boXMgVm6Sz4B3TtDpk=')))
# src/solvers/__init__.py
open('/kaggle/working/arc_src/src/solvers/__init__.py','wb').write(_zl.decompress(_b64.b64decode('eNqN0ktKBDEQxvF9TlHUbBTawcf4QBDcqMx2xo2IhNApYyCdaCXToisP4Qk9iW1Pt4+Ji9Qu8ONPFh8iLoNriaEJeuUowsfbO6hVCjtMxsZEHEE5B7FXEYIH2zwGTlNEFGIC8/5lvYH0QN+VxNaYL38+ZuS6AJrqwCoFjuKeQwPToQdbArrTUQ0yVvBzE/DhSZ3C5Wx3r3ctaVvnMnPKkE+5zFzHiJWzr6R/28x5ei76n9Kt8vXfWInbr8rcQaGbFbrDQndU6I4L3ck/bntYxno6/DIuZHFxNV9eL24q2BiVEFJ2I5USzuAWR4YV4AbEO/EJ1ZPieQ==')))
# src/solvers/registry.py
open('/kaggle/working/arc_src/src/solvers/registry.py','wb').write(_zl.decompress(_b64.b64decode('eNp1VNtu1DAQffdXjIJQE9H1BwQViUtBCMrDLjyg1SoyySSK8Nor29lqVSrxEXwhX9Kxneu2zUuiyZkzZ65JkrCNlkc0YLBprTMn+P/3H3wUpdP0fRDOoVFQ6wGAplUNCFVBq6wTyrXCeYsNLJYz9sOKBnMG9IycVxCjrHtDmi3+84E7vTl92LyN2AiJvEQwYht0aTLDJREopCx6EXP0zExBE8qX1UbvoSjqznUGiwLa/UEbRzkp7SgZrSxjvU3qpqHseh/OS22Qt4qU1qJEO7i+ExajGMa8SxDc+3q9X4MtLQol9hQxY4yVUlh7VpVYNNL4HpUzQo5ZvKYGSKlv7diYV7AhbolOq6FJlofsPEURe1NifhYB/sA3rZDU+VcEV1hTNRTeFkVaSpvB6s2TuvzT1kAQPvJDawPThPDPEnIFtjtQ9hmfR3negQ/9yqFqS7clBZfgTgfcTmXe7Yj27n4kMUi9VEueKbtxuizK+rIfqYLA+SPekP0yI+80atpOznxop9cymUe/OAi8wl9dkybrXgNWPTaHlzaZi+Gz8RiU+1mPov2vHKgWQeGkeNEbDwKaYtrNpexlf4xoLcIXPF0bo01aJ/0JuLjzBPcXgcKMgnmSnRd6WRPvtUtnsudL56FBsyS6eanzc9ItFSHNwq2hr0c58KOQHRJiNwXykc9DUI0ec/s/6ZLuN56IzO/iC7jRVSdxJfGIEuywWWx9/enz5vv651P3i7H5aPWs6fNDdW6clj1cggrptgja7RxE5/RqILbDBaRFj9cmrLn3HdRN13NcrGkf2AMDRvAz')))
# src/solvers/dsa_solvers.py
open('/kaggle/working/arc_src/src/solvers/dsa_solvers.py','wb').write(_zl.decompress(_b64.b64decode('eNrtWl2O28gRftcpKuMHk4mkHTnZBysrxz/7CxjZxYyTF0EgOFRL6gzJ1rIpaeQkwD7lAMECuUbOkNzEJ0lVdfOf1Mijmd1ZIIZtkc1mdf18VV1VzbOzs97nl6/gc5GKJJKx1KkM4FKFW5Fo+PDDj/BOimQwgoWvU9B2fKNlvIQg9LWWgR+CHy5VItNVpIe93hd+sLIzQWqQ8VysBf4Xp+DHc0jEEhcRiZiDv0lV5KdEItzDVvrwMnvqGQLD3nd+irexhkBt6aVxD2AAb1SoksEiEd9vRBzsQRJ5me4h2YSCZ3yVSFprEYoglSqGT0Dvo0ikyR4pRetQ0Kid6W/p1YUMQ3DU1V/wDY3y4l2q4EqlyKRrV41jfCjmAyKhYhIpEQHxAld70PK9WftC7T7B0U0Uox6SlMdeq008R60NrtQNiJs08Zmv3hkaoLdIVASet9ikm0R4HkiknpC6YpX6NE3bOcNhoBIxXKJwXqDmIsimOrgIsNB9vrryg+tlQmt6zJ8Z5UtvhRpWy8SPssH13lvmby5CufZWpeutueY15zLS5jYhzsTzc3OH4sR6rbTo99wKpzJG6y38QOiM09e+FgZfffgODSpZD32LuQuhN2Hah3e+vkZ1p6gpS88gA81n6dSQ0us9gQ8//oB/4U+pDGUqcUkz8Iv+2+vNxQK8q4VGQ1r4OWSLsbE2JGN0MdRYkP2SjfnahcELCFFJ03SDgJ/yY/xvNhuz0VZ92MEEQhEzQbefX07PZy5P2UotccExaJHiVPzfMQ/Q7zYCR6YOmjFwZ+XZQ38+d8y4mbxbyVCYV8zKDDuaECAJHh+u1dqSZuChR81xwhwBHoPjDEZ9OEcGnez3vA+Dkb0YuW5Blv7E+GZMpIMEfsN0cCG8CCqz5ALO4bMJzobPYMWxydwHeL/je8dQcgH9kBjJ1FGhY2mx3uJkNo2DGUwm1g6NmQ092SXc1plGN/6aAmhtaiIwWMRsX8cSdHsWLTlSvDxQ6QpoxE0QbubCu1qOMcKpEHX1LtmIAjGtsDkeN1dLnFCPQWY26apYHkSoBZryMNpICj1mzghzBm2EkYSsgrFnKZxVCQT0KCge7Wr4QA4MPGt2ZZuzHGhGtOKvUIZl04TEDbLR4pPojUi2X6LRNGtm/M16jgHUIWJu6xI6s3sxxdqcn2a2jvxr4a3zQFoxswmNXuxHApWaEnMqXtBmGeDAIlQ+6XM0PGfDF9F4XF6tGHZyPmlTjtapN5oUG0d5/FltvMTIpHRdTCj4mhSX5nGO6oT3BifFrcEjGVkgktxCY1pwOusDbgAmDJKELF95h6lIWH6QkZ/YX7OCoaonvJrV63xCLmMWmtB/xGi+CeUp1C9/CyrtRfX8rMc5IHxj0y8jtFNs8dbxMMn5dpOuN2SLb2L6dbQfU9J1hVNDGYvfA87eQ6h2qG9JyeTeHXJuRO+/XCdqLZJ0z3cEBkKPg68u2LKIhMJNrVHPqkx1UsqWK6ghZhrUnj9/DvAEQsqCEYeU1eUUWBX8OvpXejMuJy8N4MHf4I8YkIsFGFIU1RqeLOM1Qg3JDtlX4Hx4jkCm2IZPKGzhYsNU6NSTpFM9qzOdeQzPK8NZE1Q7bMmp9ZdZZn2B+fQBq/KvNS3m+j6M/vuvEYc/8ujUx4ICC4V0Jcx2iFcYcSis+egaO6wYwLcBPtbEC3prkdRTLg9+kCjkirPxxNJb+zLBSoNe/GK4HMJTlYErUjql5DxSsV0R1cTaeTrMeb47pLp0cyK4Ruf3BaYn8BZVG49hrjD9zdSC4JqSZjyjGbsbs1rc2YwRRfpltf6hsomSCbwFqRCmlU0q9KOruQ+YPiBp/8apVRbOErOCa7GfNMeHS0Gr9rvJyfgUcrPGXo+yOY0d1krmrKdnrIizmUuKwlujtbNZ4xVSSOF3BEaPVVaZ6DbzhW4Pz5hADtySqx/n5B/h7LWphJnuAGDr4S9RbQc8H2kMigQP1vIGM7m8aiaPN5UzFoo7wKSEaZ4a0husneh4z6zjvdRUZgeRSFdqntPwMFCF+1JCxXTooiB0RC58Sz5c4oldmpz1ajmDXyNhgoFXTnFnx6a4Aef001Iy20iX86KllOxW0cUo7MOW3hLxJhIJZayJoMRGUF4aui1wN3JMVzCAEf6TTH0C27ryzbz7inzsnOiNLf7ZCAjsaNa6NKMUAlALZiSLAuMuf2Mnut3Fy2vdyc1P2csvTP/ra3SF97Qdh8ckaKt8Nvd/QC3MDnqi73bwcurW+el9AchuFKb9dWBb6NwGaq7QDQm7xJ2D/n0HfGuZP6P6qRV7DEa2du5DIKTKx6PEx/bh8bF9RPiwrd5jgPH8/D//hiBUwfVOamG6xNR+vy+IVFh5bNjIeuIPio58kUeDj3dZ8/8YgOQnBfcFidrqjw0TubwPCopilUeDijdvLszR2Ov9pXx/axPBToa8oQpF6zo7XKPU9dq8FvrJEiWBD//4py30X5HU+Pa8NPa6DyINTOXPhbG2DyJMyKiTwEdLbX2FE/sEbbKfiMzfHq5VSDWmrEBwrAkGWRMYUVYqXwgxRUu/pYFwIb7fyESARjFxdiRiTa1PPgsO/PhpSsqDQIThIFUD+gWF5V5C8b4MfqqDCIyUUtM1suECqp4GudGOA/xrJ1K9VJpLtzX4tyXe3AqnZnzrkQetX+aJ2v3mLOE4wh41+hCKEzAX3IrXphWBfPbB1kLcCS75h9ppT3G4yyQv0Q1rz1hwZI4G+TTkvDzZ2HNct9sE/vr3ai1IhxIyrvBdkxKjGJ2J4QRcsl4TJfBiUjDOxeWLSc7tcVUQ/VEhWgCXIYtyYVk9mRM7fkpiN5+SEvh1GReiM0byuylP4GqVaR3PWIOEpVD3s3zeEa2Bkpd1mqqlaZBX+vnRexWptVI986SaO9Q6ACU36i7KE1uM59xS+8zJbdUvzOY+bLmeiAUbd2KL8Vr4qlquuimihPlO2m992OjgVYJAtrTUNZ660EMxkT4KKb4ywddNQBR8apHvKZp6XzIlkcCJFVLzQ5PxoLEizoTdeoJwTcnAlq15bXstlsOhTEWknWPC4BPAqkkumDPcT9R8Q19cBCpJiG/bB6YlXr19W9vvPr6DEnVZrdJL6dcaKW7d1SNyYivq8W78BL72EwwIKxFcj4HdMDtiINmjDR/QWBXwcC6uVYO48YM03FeoEh1pAn1Xb6if8erWcjIUNLDRvCZxv2Uwi/bV0XrgZ56Qsp9RtgxakvYuo5Vx30KEjroNi7y5Ooaoe7y+S+RP7Y1VZt/WKCvUXc5mm+3J1sy2OAy4pzZapLbiMkJ3/dZ8HnYgm70Q69APRHsiq4mG4HOwmMGZ5bF8FlY0iE/upLQzfGq7/NNH1C4/nPlV596a+hUbcfXFyL/xuPKY0CXvsYFb7LxMsJh9LcSaP15pfDlT+ZqlNWVrNKl5Lfryg/aHjI+mx9Ga7d+SfPx5AmeI+JTl6B1IH4pjg/839X/ypj4fQl/u/PWBIMTFLviwkDfCZs9YFVdrX8qrWQTMWIb3cC5esHRqD+bZwTATkmwnFbn24Iu/Zs1r1GKE6rXbUy4byqqEiuh0VNHW8fnaEed7LAaKXa60+h1llRVZMiLziooqIcmFlGr/SrErI8jEo7fx5Ub9lJdO91Il5ClZYfY71wP1MuD4EuBAIJqOxrOu/Ljg+W6p8XFxivRs997Ojbf07nSKlLnuC/gbyAJpamePidWO7kunwd2h8Oc62VS7S5Wkx3Scqalh+zfUSMSsC+24QMulmBRUPmLgZhYHyZOPKUrcnRgNf3e472cE80jIw6nXMTkVUbwWewepdYYjtWv9/Dno+lq2HBra4Gi//S2N2F6b+aCWWm34z/35Uoqyhn+axKK84kN51v8APnFmHA==')))
# src/solvers/vedic_solvers.py
open('/kaggle/working/arc_src/src/solvers/vedic_solvers.py','wb').write(_zl.decompress(_b64.b64decode('eNrNWm1v4zYS/u5fMeeiB+lqJ3EWd5ca8KHdLro9ICgWzWI/nGEYtERZRCTRR1FxjF6BfrofcCjQH9R/sr/kZki9S37pxttNkN3oZTicZzh8OBxqOBwO3nFfePBn+CYUCU/5WCTpRijuQyqjB65SeP/zL/BWcDWewIZpzVVSvLoYDL5nsUjWEMgoktsUtGK+0EImLII0wzu4xGdirVgMnkw8vtEprDINIt5EPOaJTgf0XHHN4dXd18CitVRCh3EKWeJjX5zpcAT3nG+on40SMVM7iOQajZarIEs9prl/MRgilEGgZAzLZZDpTPHlknqRSgNLEqkZmZXmMhcXnlT8Yq2Ev/Skz71C1BkA/rzG5yNztWLe/VpJtAXlIqnsU09udst1KRREYrMM7bWinviXV6OB2+hKJOi5gHk8Lbp6yVJ+Zxw5gjeKRoEsHIF99gNPs0iP4C1L77+R2PhR5/oUX4tUoxNyPfaeq6UdlcFg4PMAlvG9QxZOLRhIWMyngO1GNBDBFI2WTMMMJhdXLoz/UTNhaoFwdGJSe+zQ4McbvZzMDHQo7q/ze9v/knqa0X+2J+FzHPcZXbqFbYqnjibbjD0b7CKdQoQo5lV3C2NV3RkNu+ovHI1OWgp/psmMTakinRnVuWX+7K3K0CiNsTybkC2fwftffsZf+DrJVLbZsSk2luTUPII9FlHUWalP/jsYfNUeay9iaVqaf4f25iHlVNHlWr/RBKG/hTA4FVZ3CjLTG5qXKTCDGwnAyTYpo3nqu0DTHQVBBiASFLwwum45UwkGNIbnmivbDjDItVRwDyZckQNEYqYuE8QYhSnm4isygSu9M3cUGhQ3TsqjwIw+hoc1vjbwwx60e9UhXxCd7CqVaGtH5fVV3j4llvBirkPplzqWBlZjMt1PSY/RRw86Cudzkp6rxdxbIDcqQH5JQLFkzZ2IJ0bX/Grhuublsnp57y5KXcUPiahu+/7GpdUmQAxsnIb6cVonks7Mgv/A9zLhFY7ADGCp+XoEN+60YZm8R+6gCTVo20oDTU2x1wsz+ksz9NMOMIHELraohiCRzHxoYmu4cEfdZ+SvjgqJKmRThQ3klo7iYa8SEaAe+NMMDYK/IGzEgDrpfkv3XcNL/N+yKOW9r1eKs/u+nmhELvKQauDDqHKp06bBZ+qcIPYAMfSIqua0WtQtQ5vInpE1lyaljTZ8Xg4tT/XS2J52YzafB4bojbAl6Jzr3fZ8oeCr8/EbptgDw4mMjKx4EHFD55ciKWjomRJxZffdLkYaUbtjbFy1AAcnC+ZeMuWXihNQXtGyJ4mH0eWYywB7FOk4tR1gFpQnZZZaX6GQhznWNhReaCTB+e7y3eUrF1v6EIgoIr4GHXKIRZoSM4cMx/gMxLwP/VPZ+fogO4dLdEWNnXtYGUmi4AjDnCOok3CtP8OFsyq9s+KDPUwctiixRfRbtzvhYqEUSs2gWB+2MIYJ/vMWfWRhLcrXkdkMrswg5kqQK6766aHZLJdvu9UKHXTtwwmurfv1Y3mysWQec2qYe9Sg/5ROPc8qHCTkh7klZxPrOSnb0WktEAiPRZET4LpXLaYEclMtKXaR7lmh3SMLBGpFlZ9sUXi3Y2moxfiOxeaCcjBfPAg/Y9H7//6PrddIyrj3wgT/o2Xte5k/N+6tOJ6At3GA08CB20yz3j3U6B8RIfUTZxvfGjXocQbXv/16TfnKi99+fdFcCJ5G5R04T+Xwvx4kGsLXSLCV3KYmx6Y9ZJQeS7cboZbn3vB5RUw4Z2u3RCOt6DxAOJgCkgmtFvuScpQm26tgPmtKrk2Xc0zHXyy6jKm9fW/3J+y/O2mnH/64wQjFHeIspyIzfq1slnb22nN7FSBNlTpOTHlPSHsP5t1yTx7fkwIbNCYDzjF8MOOdgfVu5YYlmAZ7OPHgElZUikJ2G6/kI/oQh8qkxvAs0l9r68uX8vEYB1pJcBSP5QOLGvUHorkGTANdBuZFIpNxVZMztS95FsZr2/5EwpvcHCQ8gnQ4sVqtMSjb1cd2mkUlVwxdZenI0CbFJU+ymCtcCnNOoqwg2TkezbXVukZ2clujNWI50uYdrljkymoFjkprb5mi1gW2TaS2dqM4XRt6H/RMGGrcIfqyV2yF5kzN3/Fk8cVkUTMA1Z+NdvOMyjKDGbanZ1Y9nGM0f3B29USOecnGrzPkmJtxICMfzD4Uu47RFKZwG/lMd9svGVr9tjD2GOVYkODcFAcSKWC2BTfgi5D7ikUVbhwv5NaQZSmlYdEOqNqOUQA3RlO+9d7VGpgtRO5doil0MBW/U7I60bAVOvwIldA+BzyVt/42qKVlJT7bqJJGu3+wDRxb6Q8S1+IyMWt81efYi2I09mZvzlBJfTUc4WXE4pXPYD2FtTvqCn1ppIrzlj6JyQ2JVGoKWae8WCM99TS8/vuxhg0NHRV0IPQdWWdPhvrfL0sMVUdW/rB9ZeM2vHbjE2xc9oDdq6apr6bwvDluEVDEenlCVsWh29nvGnkiYhs43SwP56EWScYBPgM6jNI4CSDECUtnHKsd/DN/Zi37BNvpYPhjyfk/TX80gH4a/qF763+FEpOtW9xTIzM+kIMMZflC2dIr7p2YSv6gs7C9hG+svBWvjFXcf21NPUb8BTZwPBxo4iN0rEe76wKryPFhPFSInWxz6cttchnxQF/ikhFq9xykfQjEU3fZh5POHG5jo13iNWexH147PSldLaqC8/lqvcCd8rZ9ghUWJ1j55KtGg+Y3jcbwd5dc0RLqsnUg1yg+UlftfLb/HE6M4KGZYeelet/BfvqKk7XSYVGZFHkJ8aEU5lEHbLZ5dlAJ4UGAJwEzM6kH24FycL7X6cHWcMhZhpH2RO6xCvC2HMljaIk8nh9YuT0KsQXto9W3fTLNsXN7ZMJ+lHttVMSK21vjtqlBwWmbWuXJP9sS3ezDlIT8T1b/vpXjuzCbQszWwhun/86Y4nAJuFAISZ9w0dkePJe92a1EW9/kpn2Llh2vCBE6cAw6yNF5Oy/iaVUdsseXtfqPx+lBZo4xmdFTuiM/BfDx/gEXdvsxSreCdMai+R7MT1zUXxwunZNLznL6ecoK/hmOAPqMCqSYL8Q4DTHe0dANR9PQ2Wm2GjfqNiahDyvemowghC9g0nPep7cNsW2PWD77Q/iclNJRHVWRtnS7PXByV2wAOi8JR51np3rbWTJ12PNJzuFa+hFyP3ExrwHu1Nqo2tB8SFDMmYcO7WEHQtmv88Ri+v6Ceq2Yt6+w/kHV+PL7qT48Xa/1JDiL9vxpH0Gfce066dCk/ObHzNTmZ06nHH7UV4MDi5TR/kEFxGPL0v8BOVYSTQ==')))
# src/solvers/agentic_solvers.py
open('/kaggle/working/arc_src/src/solvers/agentic_solvers.py','wb').write(_zl.decompress(_b64.b64decode('...')))
# src/solvers/generalized_solvers.py
open('/kaggle/working/arc_src/src/solvers/generalized_solvers.py','wb').write(_zl.decompress(_b64.b64decode('...')))
# src/solvers/new_solvers.py
open('/kaggle/working/arc_src/src/solvers/new_solvers.py','wb').write(_zl.decompress(_b64.b64decode('eNrtPe2S2zaS/+cpsE5dhcxIiqTxR3YSpS5x4t3UOonL9m7uakqnoihoxIxEKiRlzWRvq/bXPsDVPuE+yfUHAIIkSHE+7Iy37EpGFAU0Go1Go7vRaDx48ODoB7kXr7YyjIJ19KtciFfJ+o1MM/Gvv/9TvI5k2h99OhaZerlMUrEN8lymcSaihYzzaBlBpWWabMQupmILkQfZRTY4OnqhS4YJ1JaL0yMh+uKbKDhP4mD9aiWD9Llc5tyi0P9OhQzClUiTvchW0TLPxBoKifmV8OZJniebGf7UxwI+QF5nBPXby20QL56lwUa+2gfbCkyEmsFrQBQK/Osf/4xiwCyC3gCABDp2LMI0ybK+JDDC2+zWedSn0mIZXfrUxrNovX4Zna/y18nXQVprAtpYQgnCPMVi+yBVpNkE6QWUzROxD6AENLrbxATzhyTKrv6QRouXchtEqQPvKF7Cm1RuZZBH8bnIo7XsAYg0lWEuYqgvhReEefRG9lN5HiWxCKBlyTj/OP8Zin2nuos9KDdxKrYwxmGy2SYxDKfCtD8PMhhIQyXsF4F7HqTnMssZ6ktJ1LMBngKi9FLkKynWXFo8fSqSpQjEOeAYM8kJ2tfrJLxQUL6+epFkUQ7oF/AKaMBZ0QK/5UF8vgO4Yo51M2QLpqbYquqKybBshHz2Og2idZWup2J8ORbz5LInFrooUGEBnT7p74GzRY7VMiG80XIYPvm9fMT0fJ1GgMFavrgC3ogWjvFS6GTyl52MQwkz6R//J5hzxZZrmdLeyeJx+FiOTxj40ySNZfoqpHlT5y+FtRkVIgGBPwcGQpaIcZIW0B8/lo/Dh4/mBft+k+zjZ8CRPwVrB02IfRdQpOBc4lfkaOBd5g3VgDc6eTI6efTZYwb+PIrld/EiAtxhLNfJvMIcp0CCdIHiRCCfZ2Ihl1BFrOFP9rkZZ+jSPBPecPHZk8U4ANgPQEQdESaz2XKX71I5m4kIuDXNRRDHSR7g0GVHR+rdJshXXB4ArnlgM13habJD2vWw8QBmOKCbc+H8aotzS5X7ccuso1oeDIC0coBEniGPhLqcd4SUwNnbo6d5EF6cp9AIloPu8Nsw2V7Nzk0hArOINlnvyC/Bp3FdBsgyCv7XMAuZgD3xAgRoRN3pKRn9UmbQh554DeL2aQKVL1VnBigGsjy90nD4u0xnLMePjj4S//rn3+E/8WoFkmIhdiBUYPIgs9L79+i/oyMYTDHbBBdytoW+eEjfUx4UEYP8Bsmfpygw4yVw+DoJcjERo8HQF/0vLaqe0uCkElgstl57OBc323w2mtAQCv19rL4zSWfY0gT/cEu4NoZygo/+kUIxpfHycHWcIYqEFqKcncI0yPKzotVpD8S8TFH4E7aEqz3qJWztHzT4ifrkFhhqNqHWFM6Lyet0J7mhCf4xiM6XWYmKjAd0TH8ib9Mz4UW457vtWp7Rz/BnOmUEYfZ+/ewVqQ1ZhGKzz9Mc6BLD3JQotNTKM8CZjnVWPbGHPptpQqj49NObCES8RNJJJMtfPRxX/2/0G0jbnYSXZ/xySi/3K1gs+SdGiOYjFgihKL0fbJOt55sfEdUFSgiY5LHwvP6oJ4Z+D8Sd+hz2RH+kHka+X4DFfzHUjBF0mIJKseCG4CEslYqWYii+mEBp8YVYCVQ4+HsI3/f03WNIPqzvOSKiek6/IT3O4nR6FodTMZmo4Si1YBFrECwWngbn10oxDYLtVsbVYoq7cHg9BcxmkRnpSNR4K7fMz6/HKut1skdZ/NXz59D7uD8/F6Fcw1LsBfEVd9bntYlH+gPj3IhxfjfBkXm3XAO6xMxM+LKMaeMSJ9dYfPOyaBL1TI/XXuYanwaGle6azMmEvAzXuwWy2/z8BowEfxULIMwMuYj5BxtNkf4paIvSW1nDjT+FxU/7CifAGDMjNo4gDGDz+LF8nVgl60VoNk0KOQ9zFhpUct1vZInddhHk0mOyOhoGAhiWKA1BiTGonGaIKJtpbXaGZTsLknmSrM344zqGdMNRfihA4VklZGGKWIIZNkf7Tl4icyAJQd9RYqXLeN98VpsZbSZ0eT4DvjSNh4AwPn05gRmNj6F5GeLLfXmMFRWfBetM2rCqi0KVN2r11Auk3ZGlEI4GjQb6vVT+/rOq24brIMua+uAVGrVv2Ic+f0CWMOq7SMjAFRlWB1tFQVtfkS+i7qF4/u2z12CKEiTtpWAQylkR7sBcj+mbbyzVbGBQoIf/3KYJWOM5w8EJggqll8n1kpgeFMZiUNX4PWjoaSPILU62KL8qwMKcqoE9GR6ZGjOiwgwM4jiD+bChmj1RTFWCom2mM3wzLQC2TC4ymc7hx6rdVCkDlJxpoaVW6bqArctVMy0sgTktzT8QsAa2c7oAVxSzzPI+TcDOvPRSxqLHzRpIFuI7XCTOzqBd8QmQAYvPbKSnpVWhBqmME/EaUqvkBSurEXI/Iy0CfqHibvWBSn1RFS0KYyQXlZi6lxFFGihpsQhwXLS8miXxDK17xSFRvNWyXF5uae21GKYQ4RZUrDioMRwA8lGoaShFwzTpVXNhfnlqG8M1o0n8Lw1oeSlGfJHoUHuAPh/uQVZblJFZGLtKX/HP2QNAcZc/ALuNvwJ16LtfJ7GLtcguK7QHjVsusxwWyK3BT33f5RX8mNHcxNOV/GqPsFKUVUjShqXBVC/zltUNwHqMAMosS2VUgLT1S92wTdPMt5ef8aDBk3uP/Q8NS5CzH80L0Le0pFgOTnY+q4VoFWQiUK9YxUOKo18YdFefFJuyS5uXFljNXxm39xf9L02ZfZSvgKnQSQurwXmE2tIcBTCpwsklV6fRyCWuhzv2ZhMa5CoHBHRFCy0fHbLQyoyWx0wjNtuTp56hPvvuv7795hS6FC/WMhPf//n56+9ePP8Wyi0kshW6oQlkhr5p9m168x1of3toXvkRFHXi9ZXPUP+CsxL3IpL4VJzM5XKxPJF3scA6x/G2y+sjS3aey5wsI+4zTNVlUlti6wZSAVNzEP6rmkIbkGewrOAHKPiwaNE3+IBv1J5yVNIgVfR/lEAS+npVYktkuMKCAg4KNHcuZI7l5mA4u3ixiuoBxaDR0mIykjyvCcwGk6uD2WWr0Za+wDa1yx6rV+9oft3EBOtuhhF1cOYBbS6p05c9cUULCFaYNmFNFa66VtBshfyEylAUe9io32O9CB8b6yk+DFU9bFvVo0dnvY/EqzCIC7YC1ralFdv4oKYvlxLVbOM3UjtN9X+F1Un7f6d6awCQUk+eGxNsKbLYjCgBlt1IUcN384WpGparhkXVsKUqjb9mqwj4KmpgLIuV31gcTM8NfsIDpDl7MxXHEzE6amiGZ0WZmo1thKCZRfFONo3JTM+fCsTBJsnwx80mib2Rfzacwn9u3ifJYPwQbfKvJvlqSgsDs0T1b2cDqQVRK3muNeNcrRN+1cbhQocNHNYizXZVFQNUOm6wniDDuzBgLWGihEifZQp8jmqllL01C1WpsF5qqcCBdUO+E66G31wuMgcTosXDGLBtN+wZhAiyXy9tCb8VYqTl4XFjDcK9BD9U8Pdu+FpI7gv4oYJfqQESMpdbMTptV/gKZQ/EZm19rMq1okcjh3CqrKgl2e6uURgeao0Fm87imma5YdUoOMtZWq6rDZjy3cBb+LgIPD7VKjFrwUhXUdaEHZUrxDXMppjszojbpSPVgdYj3LO4+i4H/CY4XZv5zPRS0+q3xMUQpGfN45vTp8Xfcls/y/32sNw7z8p76lM5Gbgj1+53VEeDV8XVk0NOlWSvLFQVhsdGqSAwqFjqkK0Iy3yMEU8fC28epL5tvj6j0L4OYX0D8Se5zZnLMBCSlwhTs09BVjqI8i4cFC6S3NY/MbZkDQUgzva8ebs+4JowKjBu0nbWgKvGOas7ffq/MtEBBWUU4Ky27PX6jkCr/19N2LUkE1SBJBm4cnoGsCA6IazCVHp02mCH8G7nkWvydzEoekJTvCAuvr/bnZWDGn+LR4W539qZCXsWvTtuyJB5GuKz7u7UtS7YbXXU6LH5DeoDbFMCHu1Aqqt4WIxAR1XWtR/vWuU3d7bAu5ik63pvde9trPxu9qvoAGjI6wW+4DuNl1l4KxKoBUjN/jWwnKuya0V+eypLK0O9DeWlSbFoUmcsBO9C8Xg4aAhvf//UDmc/mvWO73AwWJ/AAP0rCt/PadOfI+ExiF8rAZ8LZhB0N+UJ/KFtmXAtg5jKlXZNHIH+dFSDne9Z2S2aLMkT+qtME+VypgiY7XYdyYygEh40vTBelfZTir2hgCOHMeIYkHmDUBcyWHxK8H4FRs1Q6ABGjh2Y4ePhk1B+9vguFBwn8W+p4TwcVndgiLAz7nDdvVfablG7LF4KCkoI/6dj+Bz7HE2WAZgDw9BDFxXO9GxwjR0R2xLtVV3pK1aX9vjxNrdEho1LIQYzEYrNi6B2sKXNML7k/rXAUG63RhihwiNsx4PCJJphMB5hOx5lGFBPITeBQWhcaHq1v1XedDhZrW0CfLR4FxbjFHh3lu3mpMLV9ch0pAPp9EM61m/GTcqlQuWM4jjBgjqDyuFYaXVoUQHbMGeMTtPxdGphtIxguUfBcmAWPYNyYo7ngzwsPVv1SBzNQFF8E6Gx9jPN4/6bJMfd5ThDuRyHVyKj4xLXmDnYyCyHQvyARR3Upx8JtphUZxFYcGaewECsxKefirHDqQM8sBL/gcV/1933TA3sSw3smxrQag02sm+ZkI07LbqDQ6cCnFviIl816L1U0BIc+d5v2/LZxfk1Nth0A/PIElxIjUZ0SrV+tq1IrLX3D295MY5sTEK7n+D4HQMtpmcA7xMkNHwLpy0bYQVljycKnGPPajR1jSXX+9LiPzfGJf6kz+ZiZWbHb/m+OrerBa0pnEorjMuWJTxRlfTg2dokQTCgfxetwboLYPWLwmDN2obynPDkFji5P+cfUOvBE6XQPyXMrjPHCURdVbb5mXCvsANH851Nj1oZnMWSazp3Z243U7twamfoJlyamZnaKDE0wWhlaqCLNhwa2Xk4LfeSlFVVCQNda3GQw25hkLfSWGxDGxE6S1FUEgngrX7eH4hn5AkwW+7W6+tt9ZJ2SOYjK5RKgybN+nNSn0lzBkU3uMoEqoTX0gHLGmd5O7ikwJaroWo16m4Cw/QzoKv6hYpWKWu+RU3AP8MOpGMMTh0Rln0oaqOSrUBFG6MODEXh6TBCZf3A4GZpGrt5qbeK5zv3mMdbLmZ217UU3M17FQxuytpRT+tOMt5tZIqBPHbTDnb/uSfegOgsV0kaOX+E28PA6Pj5M84BqPwutpLsCdO8mTQzalyO4wYGknt2VRp6W0xfPkVxH7i+jNEt2R7bUm9gNLKV+b6n711Oltx4XmAsFoWr7DbeqCrWs3q4fsbuYYBZ2Jp248qpXGW+IVCSm/pCDAfjRzAJoZ/wZ/9vs4N5MwTqE62Ey2+we2pLiPdq//TRoDmLxnvjwGzqQbMPUzkcu6YHGRQbro6TltoHZg7z4oasvAQRu74SP/7wrcg4BY3ZopVrXra8IqiT93E4XQOYCotkE8UBvMecESpwlne6xccas4/N0WE8GkgAnccDi0OBPm/XYgOl3eKjIrpaCddzWCAkcvMVzzwhN3O5wAwiOk/Gko6SeXJwPqApA5ZQJpESw9+Hj04eyieu0HX9GyUgidGFSFP9LvyoTTxw283iz1wbmrMAN31WMrxo2TF2eFVhsAwMEP288BIlAeLVTO3hoYLgX0dxbt/pbA02N43DL7w63q2D9TbHfcngR/6fOM/k67Bye1OsJaYcIflHzYHts2Z7l81NNDDVVvwcWp5T5xlqg6dV7b8b2H5dNXK5tTAe74falFMB387aRlxMSn3pHllc2+7mHvqO3hq3uSYIjJ5uf9pGCbsNH2uNOpJCAg20OB2VhGhrX+zTCnbbTIZp7d1oOm3wVBRThE8UN/ozSoRq7l3UdDq8p0AQU7e79tAg0iMwKfX5yGESWVLmHuyBd9b6CHNYTWd61alFq1mSuKKPVsWC0iBtaHjYAkQnsGJZeX1n+mP3zhW72If71Yo+zqg/qNVdLINovUvlKa/gqLY4lvp3qY4+HrRlYbvvemgz6s2a6GsrlVyDbqlzy2XJLg31EUkwJFReMUxKmIiAt7wRktImP6ecEdkG/tLqYfKCwJAEfD4SpqAJ77ZUwdtpYs1UuK0u9rgWuKdIMEt3ZkfO8vIAUzYrZTWdqmSyUVanNJyZ+CpUm97YG4Q4xfHrG9KB8J05llSL6iulT+mkl3VQwDoqYQcVMehIEeZl+tyyNtvn+0gksR5mqjbvfXU/5lfLuuIoazQsKNagWjl3CvV0m6iDeVC9Jy7k1QSglVvIgs12LXG/Wj3hpriqXlWlgNnQV4QfyFeIGTz7Pf2EOwraCyBB0a5sUqJ/SLWGmVEIGvnTdMPq5b5jJ/Pz4sAZtq6BT/VTWItFtKq0MoKzveRCq0c1/gQC02KIhK5D045Ug22BBgYrUOXMjBlRzuDmnA860wRWbQnosIncRt1SB8sOxJqxksrg4nA05e8mlU6/hUZ1BpCLBrOqVgkxvGjUHDxD8l4xPv4t4n8L9lLbrRroDQOCHTL2gN37rozbw0K18cD0AYnaXZq2SVLFKY7ZaZJauf0K3cRohyhsNV8VvJqSXOx4nuezO4wuBuoWzHeTEGMAQHXvW4gxKkOV8OKSnlRWTtrijV0/FgG/83MDqbwxhe133491SBbcbQIg7yRs2SnZ7sP5qwbU3qU59mTQnsb6ntphbTg3W2KvmtJwgwlGUQ22+YVWVmF/kVvdOPNrlhgmhMYRQod3og2vSmZvh7d8NDyRy+Wj+V0YZW0kua1Z9qQaH0kJzVHm5NnBpb8pdWrxZ9r9qNWBvCoYQ373aVXeGy3g3y91ilGjs+hXXPK8cr4FjDAVnwivnF8B37Z67NlBba3pBL5ZSSfG6pCVw5WDA2rWnBpaJIA837Y6NSoz6bTG6lZIhT0ny4tLae1GAlAhvzV4qRJdDBgjsph9+G9dc2pQ7gxq6tRtuHYIPXxXeR0KnM6MQuqILYRitpraYf+Fre4D0qGZ7vgPNG7ifKsnFR2BB+cMCiotusqGqkRPPMADoESZBze04xSoU4Fp2QGdZME56h0WXbtlcICFz8ut+7fluoNU5LFV3RvA0u5BFd81mI0hHhSKzA2hhNOw4E3m+cp0CjbzRSAuTkUwz7wLjLxTYssH9uR3I3pXEShONM90i9PfZNZUbLd2q41FnxnQTs7cljPZIay/ES6ImdIDOvtuy+e2tfu3gFeViYclrWPxPiBpG71s4QoHYNHoLr4B519X5t6Mg27ORdYJLO1TVeeqzQvgsgO7pR0FuC66CdvkeJOzjWte88hKsWZswoYNeiVX2rOcKc6g3Pbok3B6yrkM6TYF+zUuOE5fS8XHd31ni1lscG24icelAuB++V6KPaOyA6YQbG/b+2LtWnV1wZQoWkG8pIzeFe51B36pC1qdvBfH2i0nTJnz7p2PqIzeu/QTfTZoupTs/cqr7+pCSxApnvEM6NoyFf6JB6JR0qvgxpEJjNJXA4FpCUvdCYvW/pht6VJ0afFeyBgv5KLssxozwcGp8BDlmboUzbiQhEp6DFhoHsqTbZ9uGESZ9+cXfcrlj4Gn6QRPVC9C+PCtspTJR5V9+d0f/vi6VPbYlJ0neQH3mx9/+oEhY9njKlwsa+BSWYJsldVwieocBCtr1PDSXuhrT5smBwDpLUIf1GWY1JjC7mIy7I16gwHTlNIReenxxSd0SQV+hnyjRfld/Q2gpNrSoXVWfMQrIJXYr2RMERYnOv1AKmm26vyt2QCxEXg95D6NcpX6YNyn3sDw4awjcH+SgDhCMrEYDrbJmC4kIly+QnWR393cuuCaB7fNSTCuegmBTbtGz9JMU5ca8rAU8YBmgOrzysQ0e1APNVT8AA1CV7lWkG3VV4hpnA74Cx1FaKKt25Mo56Uky6ga18vMCxhUwAVnnlbLuGHh7TLxFVhIlOanMIy8fN0TeI/bfI2Bib5/TdUWhOIs3ye4+L1phmpsr/G0KTWVAtQhJRV7TFV5W5duyFFwL9LfupmzbFTiZHHmwKXKhzUl88NHxTLHSdBZ0NJCApKLRBRPNGtvHX+F3lGM4l9LbeiZpfuBspjeq/xmPWzRtQq1Q2F7kKEcW1Bq65MDDKeTKCFzbCHjXrra4Wh0ji10GpY1A+hvNs1f5UGas+QCjsCB/Jw8XyT2D0j7G9+4cofJUSZFjPdhX48uWWK6NNiXbp9tWObLfjy+lA5XZ76nyue4HsOOA1hWYcrV81QQ6iEixPE242tkq7iwhfyxcBFnmerb7C5gJBaOOCTKnhLqAvVcLHkyI+UEY+2X0Lcl5X3GnvK34ol4zyEa4xlrGQRCX4y3pPu26HIuVAZVI+1X602b/Bmmia4xRJqCBQYHQCDXmHu9HHzzIT3sh/SwtzJMfz9ouNT6PUvR5uxEi2VaLOwUZEALj31E8VTfIE0LUOnwIi6zedLni7gIGqaXxezqC7zn3tz9PWD/MJTP8bjh54I5V8TAMxf/M+aUADBVzoHaF8pu2q4julAhEagObSm0AvdWM1xpT3riUU+A6dYT44v+SHjUAyr4cWYuH1nC9MkJVf1SHTTkfCwXfXTn66vIwSr8+sfXr3/8Hhvie8QlurTZhKYkcEwhzbC0FyC8YR/v67mUKhUukAQkzWQ4GABePkIlWFjYW/UvjiO/d2R2zAFaRtd09iMfA/PxEUuQMapjRbj7Z9HUNit/5Ex5mFlvvcYBkZchZtvVvVFGzVPVhhlUMMeWMJT97JddoPLmkI3Jh8DILs0uIphiMHgR6NM5ZuswJzBp2PBqc9QI4SNcwXDMAzz+Qc+uG4jUxe53YWw6Ofu2tubJvdDrr6GvQUdDmWWy5Sgm5fls0tg+Qq6g+dw8l/FSK5rCQIMA5l50HmPwUiXiXf6iLyI06l+ynvoHMhBDkYYcxLyy/dKif5Uvh1Ch81CljNgFxZnmq0GU/ZLmXlxbflDRukAk4ua2SBWHqXBKaAVCzRyhZk5lZSoPSf3o30fXE2iluqp47VBItLis5S8jSVqQ/sKx/Kv4lzHQIKIcetUC1J61M/HLGbSEZx4V7/C9Q4VwZV8iiCKqWNcQAc3jCTXrbsqs6VbDfpV8LLItYa0YVQnrXiGtWVBflyo0eTkJFrprLjB9TMulwkokO1XahWUu9aMeU7nBIcJ3XSfr+h3XFrtW77am7wbhL1yJum2tWZfUunPl5KeaeCUe7pIo6IO6/UHdvoW6PRqieoLm+auQ8v++tzmRHb1ouYlBOaMflldfHD/UnIPFzwFKMnKsmbwiFFxsb/2Q1M1Exm3yqU5KlpykQP1twvHDZVcdpuhgjca14YMXMQ2d2zv4y77v3s4BlXZk1Sv5ufg3U/Mr3LQAPLTTai7DZCMtvcihPD5+LB+HD+8mqtkxSrdVHR/+pqrjke07p4J3eSe09qVjJhj49WEHd7GKGc6SNKfc6nnlYmgFzj6Ita5UCNsr6FAoDA4mVx0eyFOZM9SbDivXEBOY4bERAFPgMsSUZrw6ZpYz8vsdaDlzWZubNCUrKdb6AJuSVdA5QfweDh3JK5zEU55KFRlbdjEjxuEQncpD9nsPjRFplxhZJXg/qVpqpOGoe9qGzhIjq0QVjuWpvqaL2cvwaGyDgxZjG53+WdRfFsCji+IW7gzPpXa7h/uDJvJBE+mgiYw4XdY3yT5+liabn4L3L62ZG/9mNeQrvqmJ3FJ0ZROtyRneJW3fGQsTBpN2YbEhuf8eZRhRGa7woBIpFeWMZyrhDuX+mcs1VMNC1BTMlSimCBXaZ+BMlZoH6NUC8C/ulPLoCg5oGcMrQL1RaWsI4yhW9wsYyWTd442Ns78P3ZnZFiQ3x48OBhzcClINFnc+YAWaGMDWapjyDgKArLCHRLYCTJI9sHSfjd4sgd7ILWGdAdrxx3kRuqGL86+u8IuTJ6OTR3dzI4R72G97SOukGn6hh+J6GcyKISzuU8F7UHExKLbVbxVYUU9iPXsTrF03YpW0n5rnCbDz9D3BdrA3QeMYYn3vlXnbFGJQhBfooq3BBVVaq0w6zaRWxATg9lfn2Z5iKMxF6KDa84jQlEjWKqgaH9TwLNngmGdqBiP4wV1dWF7kL2o+Wqe7x5vqbzuXnOOlRdD7fbn5jIb1usf0TK3uZ/V41Oxza55p3++ZQ3jmuzqJp783XCutk0bdBwe8S1SVw2qMEHTG1him7R5eU83lVZr+upGeE7NaSgZnOq+GoJ4XvLJZ6xoo55T6CZ4wpXaRuBOXuyq+A7TXPOsE0uZUbCgH/rUu0DbLefkKH31PnV9cVJedth1JKksLS0t4m5c1X/OGuw+2ygdb5Y5slfFAPAc9+rt4gfuwSYrK6v3IfNec6q4V32Yb5QVyFodI42akWk9pKxu0qp3EgF4yKkC7iYvDdmjE8M4+8ipyNVgr0a8Yar+mkrhDxJe6oS1Bpoe6bgVfEHzj7SRguD9jgAGeOV3iUgKFRUqg8IUNioLavnr+XKG8YJx1OmVebvDnIq8f62DJUqkbYzRU+HGE3PnVD/+tMj4rZzCbQATw+yAGpHKwfBYwHgHeaPcFuaUoelcsAJeY8fc4Ph7MILybA6b2ZiMXEage8Eb7u1TQBLB6P0924Qotp3Gf0fNMA19OxNin6zwwCGTsyui8+OzJYhzcieHTzlO3deyOqko5DVbHy/Cw7Iy0I1B9e0TmGSlD8NVn25bT+Wu+HvUt4xP/mYc/1hkXWfTlKWD6yy7CKxJH+NJ7ib48YIlv6PEnjDzByxDRT3/MDnluq2jjL1U2RoZ9WgM87ImnCvAf0Z3/lAGDFXGsNmCrgF/jTjAIrjeUP5L9e5/SNMowoJOWC6ZCmMqAzPBsu4Nx2WX2nLAFQQcVzxC9ZnEY+l83r5flIR/Soj8qmQrkGZ022IAGHTpjmTZf8NzQ5lArGlab5JE17xvaxI5Sm2FthTE4WTxpq944tWVGrK44fa5P6s/1gUwDg6w7CxB9d6goMIQYDEEmmc4TbWSVS04ZIYWgS2Ynku4lJf8ySNTM+HkGrQCdXvok8BqX7VLyQQT81ACm3jgBY9jt0+sALt0kch+MHBcLlM0PFnS1+2NQnSumGLKkeVOnl9PscJsE1ajvwlGoFsjOo9/xRqyXSjEfdR79DplVlcb/lAFbfdILe3XZvId5B8fXzDQ0vq3/Qp0kUtq4LXx0XnMXt1azzHMDp+03KJZLXydX+uiD6fbBdLuN6fb/dqo3mw==')))
# src/solvers/advanced_solvers.py
open('/kaggle/working/arc_src/src/solvers/advanced_solvers.py','wb').write(_zl.decompress(_b64.b64decode('eNrtfW2P4zaS8Pf+FUwPcrAysseevPek7y6ZTbYDbF6QCa4X1zAMtSzb2rYlR5Lt7gQB7tP9gAf3C/eXPPVCUqREyXK7Z9K9SLA7LUtkkSwWi1XFquLp6elJMN0GSRhNJ3m63EZZPljfiX/+z/+JN+sojINl/Gs0FeugKKIs6S+jIEviZC5kWTFLM/HlT6/7X/712/7LwcnJ99FOfyuCbB4VWPozsUno7VRcb8KbqMjPToQYDcSPQRasIgCNrbxOl2n2Zhes3xAAQb2Ari3SrD/N4m2UiHWU9ddBnIkQy4pVsF4j/N7s00+mn38afOwB2JcD8VO6+xFgptM4/CZeLhU89R/CpZGIdFOsNwWChcJilqUrkaRJ/9coS0WW7nIA9+FA/Df8RDhf3SmoP8fLSENFcDP4LLBaLjY5dmkKowoLGHABRRX+ANxHA4GVL+Ni8c0yXle7JsFRpeciXRdxmgRLsYqzLM2g+scD8cP1PwDyTxGh4Ku7N4tgHRlgsHrGH8Xr17m4vhM5FukXd+tIhMsgz+NZHAYIGQB+MhBv7lYwB9nd63S1XkK3K31CgKH8BAPJCiAKsXixfZGlxeizochldQD2KQCLltC7N5vreRZPv7r7LshuoqyEiMByKiLSJBL55rp/vUzDGxEnU+wUUsgdzCvWAoCfDcRfAQ7MJxDHmzQravgigPCBpkukRBk06ACmAIlmKm4i7NvnQBbROgoKBvYjz4gJjzGHRaiyOX8A/AUALudxNBwIotfX6SYpfk5/IEKyesd4g6+SWGdZ9MsmSsI4ysU///f/KdpDPAE8WAzfpXkQhw6KKOERmWENptUV1ZBd3CwjgPMSMRZs4+Luy+XyLzFQAk50XiGQORcRRboLsmkOq+xOpDPxkbhOM0Ab0P0I6B5p/uskXKZ5NL1Il1Fu9013KID/L/C7iGRpnIDZnAcOsIDoYeX8CAuXife7dGsShbUk9RJPqaiYxvl6GYTRKkoKAPUxL8efNolrZYvqcuxnwHloDWxgBW2AyBLZrfzkFJgfoXEymW2KTRZNJiJerZGYgiRJC1oi+cmJfLcKigWXh/pLiVdVgcggynygm1mwWRZAzAX+gBnnOrD6sGlZ/Ae5sE/442AQplk0wImdhOk0ClU5JH5fXAfhzZx6P6Gu+9CD9d2EKMesH2MXZoAr3a2vglwuZ1/w35+iHLrni5+D/OZ1ChVu4cePWYQdRo7A8LJoHuewphUc/h1lco84OTl5Jv75f/8D/xPAf6C22AARxgXRNr1//P87OYG5EpN80ZufEaI93JQEjBVoIQFqTHpzzxc9ergajj0Rz8RcRMs8EkNP1gZyTyINQPT/nR4sQFdLQF0v82ivzIDRiflY1r6eW1Vh/qya1XmH/siKmyQGyprAXlWCADqBJ4BBsPLIhvXb1mifHrf4mOGgtuK9c6j8uwSeEYn0wuL2zCYTJLjcl/v7JIGd+0wAlVB7JnVxw2sgi1yci6uSunrIPlfrYjI6n/tC/XiJPwyg58YzY21OvcbWx+aYzDZ7BfR0Ek/PodsD+exTH7jp/Jz6Ixuanv+cbSIfOGeUnY8UVmdBvKwNu2F09+rC1Vi3/00AdFTtwPUs/6iczjyj6YSH0DGvF764BOwS/Xr0BjeoczG/yrPxVR4yprYxTsFvvTxDMN7v9PIXeEWsqXelPowZxG6BW8kv3AINE3gNlP5lsE5BAJgVPU9/womZIscLcXZ6vf7IH3p+j/8d+v0R/Rl5XgkN/0ugSoIws+dYO3w+Da3vQI/DL86T7IsL4MJTeg6/uKTnHtRNQg9EtAKbxLHh6/lVAiNOwvH5OaDAbk3iYBBMpz1Z3XsFwwHBMUrKV+aUQnG1uNMkISFggvIPLPSkyMvpiW7D5WYa8dKkCYJBwahxlnDJN00TTwnMo8RlGNIq4fnSazQLknnUuzCQh5/C8tNlBa+At17mu7ADyAnHuMTtHtfwFOKsSBr0aeY9Jy436ykIar3Q8RnGolErGwYwFQRDIXMD2acG/KFbxH9Wdz4SoFs73Cs3XTlFKGgQLcB8oJSEvCGHtRZlkSgWUUWZgXlbm+BRIhIoEQ0IiBRcz0h+5po9VpI8gagPQPOC6f/yb3+DPyBgMhWwsAnKCZRE8pm+0uQWBeGC4PvYGegSdABh8x4DhArLYBqALMCNFYsAZaE1DCMVQ9GDV9ewTxI4bIn7okvg+CogeBzfRUXQR6EV2yPRL5JSbZEFMSqZr0QR5Up6jnFUsyjLdCn8RiNkeK9RpszPhFIFqTN5vIqXQVZBaBYpXNNMDPQk0cN/rrMUUF7c0S9kBLgR9UBrmTEHLoy1I0n6tI0gGuGuQZnMQAwvYWsJwID9yfBE15gQDiaMY6rmM7omOIG5wRJgON+ATmXQF2xEOXAwRRwKy5JKBopIJS9BLmJAtrmF7Nn3QCb6PTA04mPI2AxZY311Sg2cjg2hg3f1NT4bbYw1LGC3qzRhLpmD6GXxQCJv+jA6G9v90vXkw7+JvAJzAOpECCpPb+hV8UxiGpciUU/CIHmPRlrOQriIwhvcHnLkDUkxQTquT4cvFwPv3tbc6OdvzBV4plYP6od/98Xf6WEIzBiJXa/dga79E3cdJ3ml19M0ysMsJg2Dx6G6OXD2AOsYO5BCc3V2KrtNsvaRqUC9coZ9fGZOczqubk5QBhcxYhLVdKwrf9Z3Ihd9VTZTAOdVm8D3ANfDva6HZb3uoJ8JWi4gAMFc8IL5u5gjTWcR6Z8gfsjp8KqY4A2VBC2D8nHADjl7+LtVHfrrro4I2l8dtG2cA6MTfRMkDEs+EdsG4uctIppaUOYwyzDAc7Nq3wRageME8kwRb75IN0vgvUVwhzMNNJTkMJJVdbqQxWD/D5mkNzEaogQtwDMBzDWewa4SLZf967s+/i3JPbyr8Abo/wQYP+K5Mgc38A4VAuttiyjWUSTTY93CWtnyNCmhCDDNj+4asF5o4nWv3ZBlWV3qKt6SpJdum8vrEZMK8kpcZ1Fw4yyNrK8Zjt0otnni6BpOcnrjhlJvuaVCE00Q+1JCp+5TjbtTMYOFz2C1T2A7JuY20RKKZOJUugv7/hpZN3FdEngM9q1B+ob8Q/yjZMOvkYxhSZHIR2Id2tGmbFZHqxoPztkyNzVhI7uDf7NoJQdeUxVkR6WugGW6Y5ybRA0Oql0xpBq7V4XOFRI7g7cHpmaWf5bT+kz8DJueHEaw3AV3uZY5g9KIKIVYaYg0ONWUOZkWvVmctTAY+xqJUbJZRRlqPYTOyjov5ogNu99XcR0lUMxAuJCTdAXvadEOu+FIvqvMPwwKBRi04urBV+Ruk/wBq8s7U2xB64rSbA2yN4i3tEGUJi5JlF8iNCW8AI27aB9e1wUZk6BNTRktYifGFonvpanN+rSHR3fgz8iWEWYDM6Ztt4WIZf+kjn1eDvekzkwVrBKje8ENq7MOH6tMDBWhGgOjl7DdOKdUif66VgwcgsiHplcZp6+g8NitUVSVQhQVtT6G+qj6gPQO045/5Fo1BIhc1fMsQqC+O2UiNaqaYCSXVW+oRur9bqz27zbQtetIRLdBWCzvpHZ7k6S7xFZqC5Ar8lIzKk/t5I4HbCNG2wf2y+xln8w5LrR6VZ2qhHFWlT+R4aPJ94yOY/AcUnNpeSRps7G2LpnNogG7LOchCY6c+hxrQGVJU/MCfguzjV3U6DGGac6qmqiyCwnRKCmHpQbnwhb2VhOFHMm4OhQJrmUcsoTZfZOR6vVDJh25YJxmX70STDOvsSQM9QiVTjT7lm9atGlk/TgUU3UXX4iXzVp2ZdtnC+JyNrANAia8avOyYocmSPhgyouThnpSZ+Q+uNVhhyLsuYQ9h4DC4mVSxElFJjdpRbXeJMlZMpyzZZP9NbdfWab/xTqHtRLs+W5XKUCqWNsihduAo/7D8wM9VmPbNtVuHqNvDgjkD8+1mxG4985NTb1hG2pVEVoE9SbEkZBAJkGTOVgnReMatjTHR10I15f8vSkcUwbDt6misjWqyk6EObGFEButFN0UKRqckmJrs+jskuedVJEN7aDMSsAaxUTj5I7PnfC4C1ukk7QTNzMsDfGNjjOP8vC2wTLvHEGzSR7Nb1VzvHTMIIeSWbpcpjvWKAioEgpYKXtT4Aqeg+zLzgtYXZfE080lfLat2soUzxZzqJAIogcRF2pxKMN9j/vEQtUiyKWDEb+VvcTX22C5AQJ+CDu2E33HGrBHhsjKfjUTQO6EEVXTP+ydF2USWwg1bZ6A0VWwlF47iFyACyRPfjaRj9ssErmtZsCCpSP9miIhvcC0ujDyofCLF+KleC5G9dO2C/G+qnLuVN70/EsN44xLO5UM2E56Ug1BaKrqVaYbGdeVnQZzk0S7gtEuA6kJAWHLnhDApEsnaJyP+lxES3La0fNB0gTCtebjUs4HvN87HZct03G5dzoAHWi0SHd7ZwLLhDwRUOkqrEyCpVLumQSov2cCWIJTs9XpUCcwjPoGu+rXWE9Fr+KXhxv7Z6a4cLjZvtC7cn31G4BrMto9Tgj0GLXdyDQZfYlCm8bCSqqF8RTQGIfBsqppkFK3WeM2Tf+aPjwFPxesrUiQXotSogvtVUukfFChCNNAoz4ZC9S2yTTaUtQKYfVDgik/I7lOdpXPyvdJ8zN2gXo39hn0udLmGVrZJAiFoC5xZ/cbUVzM1LT1uEwrf5hq2EE/M/YVUwfU9NKkBep6zlXlbEqpO2cso9CRuYICwgodXNoHyWIeb00bCi/uA5iNQ9/RIzN1njrxHqLhOEer9RBn4ySl61ZZl3MqJjXK2ieXlyL4PmfzRyp0t/W5Wfb+C20GOexmpo88EQfIdtrohS7naNbGl4RgQ3JGP18lFl/fCVicEbv4YmGE+hAycdvojhWNX9asubHTJG+6dmrWg5+aDLQgii1RlSDUknUOn3IbtTibadLNCv8M/d3JcR+rX8/p1LXkZPByAi9h8fTIc6zO+uvcvsLcyQl1XOWNDHi//wnIn5MM2oe/vYxb96k1BuBZBUNZMGwtGNwyxOB2D0QoGMqCTogVKyrNSR7/GpXoW6PtebJgINBqX44HRN1KmZ0sE8oyIZWxGGyxsKRmBbsuNlPhnaPwzlGYFh32HOb4CiZKfIB1EcSkhFAsvLpUDfppPHXZw7SI4AsDX+4dvUD/NMRyT2PHg/28WPiip3FBb3bO+nsOetQeGeOBHBQpWNpo6o1CRllWbBvO0KtQgcxbTucVrlptSnjmgeXaVY94GXUTNN0sB1+72E9HSRMnptjJTwjL+4MExkaCgj5NAI0oIeL8ZERNY9L2il0rhVC199qpw5JAZZ2nK3LGy3evNhqkVQp+trh3kUKZtCzUqEpe4HRdpGgRusTHy7R7P2hbw1iNpl5IrmganuOKhd7cLgxCuoeCGxvuLrysqgouzdXDKbcIziOfulFHSpEUn9vnh6xJHEA/e/HuUhWqiMcXTvQ/gJpgL4YupxR6SDX3xcZjAhoR6x5qKN79FQx3+Omjj9Rq0DvqQ2nWNr4l7RSDIlg4JnS+UtazWKkhU3QqzNF1dgeARbBEnYTsmQRl0Z9BW77Yyr/o/Z4Wi6prPgM9l5GZZBEFyN/ffgdvVrA6q7BlgC88PYTCUkfLsWrKh4bQsMCRKzMlMqBacRnxFiEmoylwFpO1QI2xAW27F5oNrB5NRxDVkRmHUW3iJfPgqQEZnY/W+SSTf0NfYNOThfy7NVrWsguZwbBXvn5Ea1hV+oHJ/EBDh4ryR1iReK6g7geXVZH5wrP5SGHIPwzTJbWHlUKhS1qXFOiSkXjs7AiHEs9Lh+Wybr/luS+8Rohbhph1hrjdB5H76Abevfu0EGETwsVKwNbs75zF8zgxNscGORTUGjfsikBa7Lx2ibDIPigWz0kT+KDYPWfpsKKSOF29ioy3ggkMeWKsGeWIb7QbA/XGO5effApfUvUFK5lCQboQX0Bdcs7f4eNuv9LN1d7HaujAKOu+j83XHRqdzoy4HFGrAzAvXlDXAQA+7RyuRzhujMu4jhMZnG3Ow4zU3p6MrERl07FwZts9hZQQU0oWJjshdHOngW8g79g6SVdCII/6Vi2t1wTscXksdZD8KMYKoVoT/PbVBJZ99GzVF0pDgEpNIpSADu4ADoDH3lhV44bbuHp5Nka67rkoiCwGssJ7Zo2O4inDxMAcgvEQYmszR+H1+S0eQNCeV/F1yHCZ2nLuEDleSQumjsAsaogsYKh0umHjhGHB2CroVBKyjLjLkNnL0P5GnGfH33bDfQL5PkZQd7HpLpi3J3Z57IJ4c9f3mf+FDrIWZZC16HHyjxzkLpmx5k4nshEzkKY3mTSe9q5lVo/+dXorgnyNMInsgDryfv7LJsiA0aO1FffoG19ERTjwjDMElSyHDrN4OOXp+vLuIQTyZvQcK5h/ZIgIhJ3JTaR8wMlAf4auyVSf1HwDAro3ATewLNtUxYzDXFKZsKXMQlnIAZ7XJ+s7Pj0v7dY7ZRoHaFyCnowSMEWBlLYJuiGXgPiLM3ouelzoXCw+2Fnfc0xnoMpRmoAFFtvVttIerNKdT435qoLPEEztQRHcRNJgy2lMBzNoNdqfeNz1P1wBO8eaPd3h/5XzFbcwYob+0wjd0f8dMgCo8ZVpAHQHMBOAZH8GrWJ2gNpUIYSaiwzXKdKJjvIqZWA1MyhuNU4SHpCphS6zC3GcJnBvDv7UEUExGs+AIUW5jMW0DsnkBErv5yq9EEDT4sQdr4Uekit8FuqcPkj5gJ8bahFaqAYIvAko9lEaF7ifeK6HEd7WWVwtrlRmJ+qVNunqeib5B4R+XIIUQqucLOQLqOY5fajLdg6wYVK8VFlzsErJ3R/jrXsjDDuoSgXoikt4KVFJ4f3q11V+Q/QNkLt3w65Nlat0qIvUDk30Fwd70N/OBOeBaj4+aaekuSTkzkcn3cmJ50D3dDAHDpXfeK6YtUapVsZMU8qvvJC7M6xpSr8GXLYpHrTEj1tf566poZx0D0tVPeHD3XJw0Ke8B7IE/D1fBqvraSAwyuY6791cvRz3c/zX89o6o0lFNlH3VFdnmLzrnrQeB9UJ7cmcBGlMPMbjIJkDwFTy7pMLYO/BQ76q+F45tqYGrdN5FgHwDkZHORFaSVlZZ0HfRdk8kouS5qunzoRK8dbwYcDSU8cGtbLYbl4XTECkJptKvhrERbSCheaUTogPyVaQd/Pj1U3bgXjT2MsOX93w0XuVqPnrAR5ub/lcqtwuTLrgXjYfTxn5Mv6Q46rWgZCYo0agzrRax9GduC3lG38cceTVnOH0KR11uYfQ4bhLZmxd3pGzXDR9VQqQDCmXkSfAGGQq1ziU0S2bNWZ/BB1y0VdZXvHoq3zmFLD093P4M42DOe5LXP17HbbKEmYiPURBNOBkRNkWu8Pee/iSO8gmJNXGQ6jfbtwdq3p/bOqMEvIEuu0SCO9WZabEjv4zLZKeZkD2axlUX3W0AqZMYkBp3uIsf6o4MmMu84UYVUz2JUgymNuLm+o8Pzc81B42bcqWlYQWh60VCF2rUJqcffc+obbYuxVaBU4Xp2dmRfh72R/1w2bfLVVxCxV1tQuogwLf/moZrA9HzY6N4oI6pRm6OD+/NMCAIp3trx401qf2fe5Nc+4XdxCuxr6aoBXM0KrZp27LXli8628P8ZtabV81UXVtzCuzmU6tyG6j/NDYjPMsjuOdAclGzLMr+5e17K9TMzflg4vL9fVpbd4WgzJFEPh9lGuM3a58SSh8EurMM/FDAltjEIbRGhRT2Ev6aCp+h2pNqbMoWRJRX1Vjup79oMR+RyJ7D1idj2zLl0zIl+zEV3yhHkNnhvNXyLtK2A2R6kb4hEVyJCxitU5xEw8b17wnK/3jFvuaO75P9sPI4BWgLUajYfTLJlgSaZc591HoypjrUWQmNsHi1g92av5cnHLO/mh6yqn1OUG/kfxlEcnU8S+i6VxGXfygXas4ZQxDKAE/iGjXjJ9j5btPqqEZeYQnqgXo98s4iernAo5ATTwKeUGnKJwgBYRc9GNAPqNyt2qogqB63aIxoNZEneCMrbfqzKbz4YLhDypt9c4wRrNVpZllXtfsxJyb+so4jaj1atzUK6jb2iMcsU4AV1MUVZ99XdY8Mlsv42JyfVfOreu4pw7Cnuo3CIWvZQBKwrCasIBBbTDzrFppGAatJnqd5jG5r3Sba4yWpVPOyrwCJ9xacj1bgDk9Kve4zuEz8e9U0bGh6Gb02Q0WxDzkXs2+sSULrRWewoLDFrSIi8rG2wT4wjwxxwD07qMM1ShxNuqjDBtHWTZTHWXYMMqwYZSXNQp3A740RynJobY+exlq0SOPFoTGV93Y1guhXMjlyhbrw2TOba644Vk4qq062erYXb8cCRUMh57PH+oHdlzeXFnEmCf0Hvb3aXTrOkelWhzFvH+ZGQla9Z0x0S1nhjYujwlU0j/epLpGtxm5zdQ6tY8fZKdU8qVqKvy2hVdWpUT0NCeYpd90Jd1H1i4YQ0r0b1l9aRg88rMSFRyy59NfAFRu3GRd5NhKOYUWBfANEza5NlgfZG4jIhVJKTeeneiIW6jvDPVEb1DXfUeF1O7kIXeJEzuhq9F3KzNoXkcVUxL5Y1u4qkw+7Nrl5FfWrnwfJ2a79iDL6gOOUdX9sdAw4fuCygPbsp53UsV02WwFw9yBCprl4LQzR6izy+ZlBllqXoZNjWr7MYNo1ETip+u6+Ez8DdP3nDm4C0iP000oA2el5VRnzbcSgGloMpm9ZIFOknm7KiUxIclF7bikqgjbcpDGCdIAEC059Ss84BRNbXXS48MhalmGCKPTnksUlNyjnievrRNWWm+eV53mUU+oVYM/Tyfx9Lbqy3oMm5MDQcZ23iH7mt2LuHv6ZLPiwYSjs9tRSmKpuPGy70uVjwZXQRh+kD2Vc13f/K1gOELQnik3wL53bg6r+3BqC1FtBgY0O3zOTFEns3A9xDla9wVZc67tsIrYtvJWls+eSWVPpy6zeS9ytE4BuZkrADS+Go2PcbVtulTwSYbAuUbSbBn6QQWm8UkcWRB68tJEj+5RZBk6T1d46ow3KMrAt140mA98/AZ87hqwlc6MvLTsawQfp+kqTgJ156HPN++tHiQ1nWukx9p4PjU0FlS7SudZ+OWLVTo1w0OQK8EbOlqBsePQWWQ6dcoYMPDeyBBj0115i0O5QPjwQoGFOs3ArBRpdj3KCIEor9QODWFyu7cvRnuh0ymPRMVq/ie7JzMQZ4qJRM9pXYNV7Z+5c21T2JC+isvVs1pO7GElTArJmJSwSogUzyeG/1EsoTGxTPjKoMa/mLkZHmPZmWSEilAySSAa4rmCXLNBmfDPHZFd1GVchJ273HQVijL/Xem7JhyWtrq1blzFhYQjcYG/LGTgRWsVdMCrLgjBM4Ui25ApGPtptAedxfyH9c61GQvV/Q424POn41F3X30AvYIe7uQIJ44Ip8LZfOZJvsli/OoqH9ujIV6A3l4Is84BJFF0iMOjTNS3ZOARvVNcPtg2ksqp15KPt/GgeD+qK2inxukYH9tuv1NF60wqCq3kQ42+gE2ru/HYutO9Le1Osu39JSb0jvtL+Y2bgTWkh65pZR2zRDf4lNYk7yPJoR5DdvS20t6//TP+IPtD0395+Ba2i06IhT0kPGTraAb/MOfNrTeNP4UU2k2d76BXBOqq9M0q0Rn9yKFPahwqFwYo2XRJN1oYyoPnr0nJCEQOasfSTFSX7kr/wTivg6HLkvBk8iFyYDeN/1ht47P61S2cTRIXRacwM3UKYuDnmpNeAyI2sKcGuTv3r3VPyBVWoHXBaOWTS7wU6q6n7OymqO6N73WdSFlof8pbGx1AP8dF3Wku13rg3MbmbITh8bU09Uu7ed6IMDyufvsIk3cIADPtkCe4NYGbMx+6lcnFTnpaSV1S7RKeSh7bpcaTcNmrqzKl77japwZ14U9VoLMqwPk2iKlUU8jfO4X4/hxqnJ3durzDYI77cqlh7Ub7Yl3ydF2qYi4lK4MYvLt/BrF62/UU5fDbtOtUxTgjdXPe4nU3PjlWKm7OTtZxlmqRrwdNU5eon4bMDI4p5JRpau68Y30LeVEg/68tirdL/bgXHkX9bdb1yhqwePfDrYF3l0VvDx6d9HkUAp10aKGxiQ4PPrYYDQeCbhYne+7PKcvcTyRBSFPHO6gUnK2PZNQoCVNK783+K2Shyg31wlQi2JUincm8HlSB7mVl0Rm3NxboQFN5CK2haYTHKg2fW2FGAF46hLjvzcF48xoM+wAg0yK/cSG70QghOeK2XN6QqLOXYQ1Nycjl3JTxB0bPO0UzKc+j3/CeZyvQlEGrYFOOLSUZ/HdH7vC21OEOm7f07mIzBjsQcfC4Vy18fae9dazisluW+QOkzP6tfWppHfoQSU7wFG2iDtBOq9kf9HoY3Y54+shpCU9pYAXQFTuF23fJkpwrvedjnbFhwLVPc3BLTWcSM7pWS+f44IzXHF7Hfa5OAsvDRUYb3lLv7qJ0UPIF3QVQ6XBzV5Hr7u2qbkRphAc3Y07WPmTATNEkaV7kTJ1PG1daUCZqPOZjOuJLvXqe5+4/V3gfdoUxpRKD/sEz6v88+U/XGevt+Ec5Dyh67qXnN9J9RYVqvuVx/ymDJX7ZPNcKEUcL6du9yNFle6/E8tj948hv7NhB0Tzycq01ThiSBWmuGlnUYvtlWkfaZUcjdPwItrDnfrlc/iVGN330wn8KcUDN/W6RoDiLjJhRtuN0F2QwpYbD70cyZCcXvc36xTTdJS+W0ax4kcXzRSFdNr42ciWtUrzZJ0CAGf5Zp3keX/MF6QhvqnrGVb9c5qlYBMkU05GfznkAFGg+PxU9jA4LqPQDOYc04edI+etT07tADsIhFOmxk2R0L+OWyhF8PW9IEmyyVd0ehwnD3J0enDLs4JgcKZbJe162lsnT8NF2VIp2E9kajq530UeLo7wdxRPPJdR9WXgvvDMrPFiChd/2Xm0jZ7N+GqiR1Z87cfR2UEMr3YGdPfH7GeUQlsjphgbT4agJDQyWR3/ZkUJ4zHLAAKBttMjcHtlgK3N+2T7njYN9akmkHoedG3tRUgiJg8RIfeIZviQYXy2TB5T99toDawKi2nlM0VD33W1zfXcy4ruwPDuMbQonJI7ayPDeUij56OVA4FVwXyeUB256kYJw83RO9hu63nI9tjpbv5732YS2SPkWORnNPNtgZqFIwiT3YMMEMSiByIizXDnlVxJF00J5CEGwYYjHSoHmFdbzqJioESPNtt/Wl0fFWZf4w6++ecM4YaGcPRswhBxtKQGK2TXDQfmpGpCIKXGnaBPqHX611tWQ0tOMO12txQl3zcSzulNNqpz8TFGMVNN7JX7Rkab0omtUuR4S9vnikfSZsyv9UtkGfXLF/mWwTte4rfS82kCmUGZKg+31+iN/6Pk9/nfo90f0Z+S6bDqBagmlMnqOEMLn09CFhOEX50n2xQWNHZ/DLy4ZD1A/cWCiTB6cAOqS9qvUKghikBaK5CuD3tUiFeeasKkTZQ+M1WDFYbJZmCfw8Fsmz21LsVzgEm6/bL6Wfhj9jNgg1CntsCViwQuDNqNy6HInq3CU2jmVlLJUmf33ajwTl2UyY74lFVV03TLh+z9OakG4rZmCde2DkgXLvruSBLt6zglXuyQGVvcczouaEwqOuJkv+6Kcy8bEvM3p1/ZOXz1tr5kg1j2JpvJW9u7pSvhxS8D0Hyb6HySDz8JKkleDARyS3BXgHHy0bKBPsdBZaJ8eqewoRtHut/SVlSxyI7+zd50aldaq7eFgLtAne3EfjUtycmtAx/gAfDgQP0bZj4B5tu5+l24fVSKrNu2jqePN6gdeZyn1CZkSnYzQqGZQpg5MozSN8/UyCKMVnn/2WIjyBAj5pFOwSvHzolKuUQ/BnQynL2bfYqQHBvGtdCRAoJRaq9b4mZgv0+tg+Yqv583iCDObYCuyDYwpDpk9P4Sm04TOY1Wd2pXk5iAPF33o2pHx/sA1mQeP5ImGnN1uQYccogPzaJXlEcxiJ2IgGXvmeRNV1fGbO5/L0ddxlAEWLbkotxyL25LjVGcwiEDd3iJxAkmpWylAfE/VJRY6rZTXGPs0NQbVv8AETGetITVTY5T9S3cS9UN1kY56CcpDhuax3R/ZQlPJCojkAl51v6QiLgdsqXud82WTXKx6B4XaDs+IQdwht6A0CRZ5Ac8ISqbFWs08xnOztJJnQu6ELec93ayND5Nrtyu9HjrLUnjsOaZ5Dy11D3VDTJa00rKc1JUlN2dNRk7qH89Mw9V96i4hSWJP181Ccb53mplmr3w4tb03ra3nEKH7HjnnJf9gQW7qFraZM3QWszVboXqWcH2ELNoWtNgsp+49Xv6T3XRjN9bOdAS3kR/v525Ml7ceoUd8NBD/HWXpT5sEDeZP4xrwNt2iNpj2M41cZJskR0cYvn3gOip2UZSUAYfsfSevQSgwrdMcNn7SAPB7H6qLIriRaaSotHKrCab/AI4lnTF1ehROolL617yQLjdo10hAooPF/iCOMDVEHKsOfGipA6DEYpjlRHZ5r3vw8Zcd7M9eq10zHQf8lAYqoCuhhkcwK/sWvWEzK5FtleVPDg/VNk2BCK9+O2zpnutw4DhoyP2RL/j/T2/weJf3qftWMrwrmk8PioXIY0xt0ZMkq5dlXU1C+sE9Em9pfxdU1BFh2K0GhNBCRRo4vtv3oYSOA6AetozgXhg8byVFi4wk/kiO40e6rJzSGKm+td/4oKx375rxtWDFXAkO/7+9c79HkDtuxmmhUlGudcCqrrnrHTSSt0jFncf0pzfWfb2xkAgms8SX2YjwYN+q3FOqYUUM8cVVxVnLl9vDWN707IJgrmeEYDh/GVrpuG77a8lv1Kw8dfcOM6xDEiEdIgIO9fjqpvO0ZObRCqzq4z2jAh7QOetjGSD5Fah9T0al6RI4WQ6oy6UfdMoEZLAA8f5XYDHB8gUUL+IwWIprgJTrK+Dwutk56EGlWxGeeWAu8ICTod+BZgRb0xzeD8QPmUzAgo0UuxT1BUzXExY6JDOaEhTKZarMuPqyYtaVykrpw8Vdlgg6Vt35qHr6QRhru+ujcW9vCGyUBn3H1SDsTMe24/J+de4BrjRMDaU2qeHY5fwiQ2zoKqyyZDVDibrmo1yMW6ORUDYyxL2twZ8GG7FlsbKK0ZrVuXrOPjk0XzVfFbhMUnEIXNXqR7p1LFT+x98y5a9BFjLZzO8GsmR6md90pCIV3FYLahQsHHcCaLxuHcn+/xU9Qh6HlHF/v5QqudmWamYTe/Kpq7WMNh/4KQEdnM18ph1qz8w769XRYFBYTFadFUK3/qMyoI4EfxDRdzLzm/2rX1Nf9zQvJ7oajiMHUVlNLkFGeaxZt0Obnmt2K3aHndPpGMUBLj4hWY9rENp93kqfogZ3zH81j6JjVp3lbFTfUUyhum0veRweSF0QUTv0eSC+4/BxqqOT5P9WPB7j+PTJQMhcfPIWX2jyq7s3UfQ4JP1mp6e2TneR5gO8fA8v3lPZFHt22nXPlrPLO50JzvWdzJXI7gp4uxHQrpWYEd8W6RrZTh+VaMooYbXBYvplmt3kCBAJToEr0Ned75jXwPJ1EOL3dMatUTcfxOOpDZfHCv4f16/7i6aTdDZDlnhIbsaVjACkrSVdVtM+2GkZyUGERfu6N7tOy1eTwtUXVAO8uk3LDDpU7tfYUiMfr9ydLL/ASHqZ6QDOBOfTh7D+wUwicwscPSwIiV2x9zXXIRqCFuIV6qt4vxkS03V6S85zyk+v4mTzKLFa4nOIthw3NstCIywU3LoKPS8voAsVrLAFVqhgha2wVDaRjpfEGTnrYdlPMLVJ0wTja5dbYEezeL4AEDv5CWFZrlz5gqz/uJ1hodpBgNpdXFpeiwIopWte8rhsKcm55VioctAaM+vzjFh3upSMox4KgJLo0O2d0tbvZ2Usnr4+UloOlvE84TwzisE/rCNjNXjGbQ/MUfAD2u3D8DzxPkySuxiKvUCWfbxOEIvt3IkGkcC2nBIHBkS35eXNV4zr4gdcLq7q/KlRH6VR4/Q8fDpQEhx0dhxzM9nn+UU173FlmiXhRlMr9eMbGuNqA9zgGi/uVbdiu7Qn8neW/2bG5pPzc873FmKum866lUYG1Hq3ERoVPm/f1xZNn0JKSIfuUhmVvD9LDucYJeXTgUy6Q5aMx5igcY+20tD7vVkaOT8UG/h1sjMVstGDBZPw3d+on8CUJ0Xu6b3UTNpYni98iD6ToLWAPKjgYPbGVN8p/uHth8jIRrcf+iIqwofQLxpGf6xm8Uktg6MckTuFowWj4QyhUXDaxvVoBpWmcPiwYoEVQY3t6ojhvW6dYUijmuUfSQEJ/TrdZ5oAeLBZY1L0XthQhMf3/FzUBFv6UsN+kU6wVYn9RIaE6kSxOqDFIbACfX21iZda3+Y0mqj4cifw6BrvFiT1l4souKwlW3qK2vXNlvfqEsbUmxVrkiIJhInY5KQugVIehfEsDqWvYqn5V7MZw/e8zGdDVGK247qW17BPkrWH5TaddRQAeo23iVFz5IkzMohegkDCtwdpjPLbmcENAk7oCgilLiirg4nqC4r7ICq9xMekITOkGsL4g2Rcq5+U9Uf76ledrBOvFV6H/rRBfAawbunivmKhLu17koEJOnPsO41MCDmTq5GsVjHpJkFJpqFV90snNnW+5vCzmc52CkthRcfikmiJHbygeEBDkKSB+tQbGO2v8bpnjFgmNa5mSWSV1hi9/RVU6VSp0hajUA4eC1zD2CIZEnfWr8UH+kXn2wP+ugky2NDKUyotOb/+4fs33775+evvfxZBmKUgcFBqQhV8yf3PDVC9FHhUtouBPewivB+Dt5sZ5kdBX06jDYMLB6g/ezWe5I7zayerCjfk+1JMQnNfqVg7sEE4jmgJ1TMKW9vPMGsBbRpAZ11CX/Eb3aKtzbg2D9b7UWRoKQ72ThviBmvqlm9fZ/CV2lDey1xRIKr30DcyAKdK0TZSA1c4b3uGfVzeefxxVCTIZwPxN+RL38D6kRlDH6nvVF1rEULqLe4RNKstlNbUNOzlizQtSIwI7pBqAlibtzA5ZX44mOV4KRZxUXCcOHEmAmbCAdIj/ynKWjOgwHOCmItpFuwSlSq1vPHqOI3FPe5jFRbzVlxCzATGkHfPeurjmNv9cO7jED3NJqtgjS4I5CQsZJYk5f4sevInh4EImTdJB0kIyqD0uyM4j+Be6cGM35r5tEUleibeIKaJWqRwHudl5NB9g+I4GVVTkHXX/E71sDc9wS1he6iSTbNX2E98Cttd7DnHCl5qvso7ZRmon7yVixUHh5jkLHSmzqXD6WOZoT4HySLDqK0pLPcePcNiZuOEr9Y2e9B4e5MYMBTpqhf6pb25U6aqE9HittKUaInelcGUZYaz0tbtPruivnZJsPCTQiQ5JFMSC5RK5GArXmmKLmr3Tk/8iU+CElXz2t1c+LYdCrfTk9qWt5PzdSoH7ta8ndbmbXC3ehZJg425d+69MeIlNAekJ6eY8doEqBZ9PJ2ETuusktBJMptD4srX2eHyGlZj1DVW1ahdZ/XQGPkRxeOso1TsojyETpDeub298yo7xvb++UC8iX+NUOSbPl7L+x77u3MEB1nfc4CQi16QRQEohYYVnlyJyruUTKt7sciiSBdEtx0C8tIXH/niE9PujkrulXw/fggB1jngY+XXzyrJWnlkExqV2+aOaV8e1ujOKKytrX8dqzsNUK1zuuI0rK9feTMSlTXdg+hFxRBP785oKkhwmpCF7EzQUWrrXVPaxKVrndj2LW34TcldhI5qqUtukyt9G9dhlPUMYE1m4HxsnPfaAP+8kqd+/i6XS5lQ01q0nQ6rK0SFv/yaPPEYTU97Bu9K5cJkrs6BOtmb9uNKw32MhqqXQ7xWPSwCvLDpKSQtablfvTaKbo7AIL4VcbBEoy0DwBnDWDsd0Fdm+iXX7FilYP+BDDFYYhWgFy9G74VplkRZGQyIBiquK0/QdDMPc6l6bdDHbvOfVz12dYcnEi8H+e2qy9SryGVjAreJ6QTJJdV2OG2UDMrrQe6brtqtuzPc+zqeygtG9nieVku5HU9rpZyep9VSaNcY4e6ZkRdnSM/hsIs1Qh5fyMk5E2myvFM589meg6GUlSDK6HbNDih7Mj2yq+vzavIAVV3lf0ffRe+V4/WoOXd9iJ6ie0FjIQfoEb22fGBhGBKriDtV3olBOSzsNvVxz9XyHF4P4BzLx4ZUi7hsc/btnuMnGz4fkctxq4AcUqnQlejhz2Taj832wkzT9pOus+s991xLIAfLITbR4oUDBKg9tJKt8IyLeuxiTPaUq5bAxAopN1JvWy67Q0MWx41RitDfA0MTlfVWVffebWCiPTcVN1d3cB+xLJOEaswqfGCH1XcSNPlWrHTd1mPTfelvcyE2xRrS7FKHmvegeznw/n+senxZ')))
# src/solvers/advanced_solvers2.py
open('/kaggle/working/arc_src/src/solvers/advanced_solvers2.py','wb').write(_zl.decompress(_b64.b64decode('eNrtXW+P2zbSf7+fgpfgQa2L7a63ufbqxHeX5ppugV5SJMEFOMMwtLJsq5ElV5K99RUF7tXzAR7cJ7xP8swM/4iUSEn27ibbQ4MgsWVySM4MOT/ODKkHDx6c+Yu9nwThYp6n8T7M8ovh9sD+869/szdhkCYLduUXwZqlS1b42SoswgUTBdkyzdiz188Hz775dnAxPDt7SwWiZMWS8Do+MD/x40MOFXYJVQFau+B9WOTjM8ZGQ/b3NEuTNHqexmn2IorjN0SX0R/swBKesasVu46KNZD0szAvWJImA3gWYCUgczFkz9MkCYPi62SxTaOkyO1kwuI6DBOW+5twQJXZ1o8yFspaQOuzIXvJWwES0eK7KAlfh1RWkERqGX/CgjCOc3Z1UB1b7qCZLL3+FH4HYo+H7OufCiD/2j+8yNLN3/zsfZiVfUNaIRVgmX9gSygCfIp+3MnuYQNA5w9AJwniFPj4Ik7TRcmoC2OAMIYwi6DePvLZEksO6DkxLw7992wBwguKKE2A6udD9jq9/t4voFLyfZb+AL9ofEOqW/4UhwSsooI5S5MiZRsaCv6AXPtiyPjYSI654ODzUMmTRrpYhaJerlgoOccVi6VX1J4Y9x9RsPFuk9g7qfcQqNl7CD9gD79UOvIGpE/dhH/UcBULkc7K3+Y2ZeHiJmbKsqh/o/Mhe3udEs2v/GRhU2POd7b+dM+K61TQu4LCOcyQhSa9nIsHujyC2fE6XMbw4NlVuitQFatkM/67nBCce0ASy4s2ca4eNpuwyA4sBhJAGObLKyrJu3x4GUar9ZVVwyVF0PFiHTLxcIlfolJ4KXyVZYE+zKGv/OD9Kkt3SVVhJa9L5bxSRfkE8JMDy0PotmTvYxTd9gC6Cv0VmvA2/UeYpbnW4wCKEB/+CT/IOchAD+T3/FOuCiOYTW/W0bIAgjlQzL86vFou87DQiMFcyRK2DbOBJJRjDRg5FiSR+dttfDh7AIvnGXV7Pl/uil0Wzucs2mzTDEslaeFzaYpH2EleHGjGQtSy/HPgAYytD4KDFYAXKw5bXEtFiVdbrODHffZdlBd99na3jUPR/nAYpFk4JCVa+kGoyH7l50Jv+oz//zrMdzFW9/P3MCkKWIL67PssXER8adDprbJoMQ/SRRhIeqW85lxCvHgWrqBPoGSiGP8eZsKknJ19A5TYhHo+pX+gq7PZ2dnZQ/aff/8L/rJ1GG9xbeBff+1/z84W4ZLN83VvNWY4eI8N/sRFhkPv43yfoRVkwCvQnAS0LumtvD7r0Yfp+cxj0ZKtWBjnITv3BEGwA0lo0MQPBqFpNh3PyDRn0ApbzUTVq5VRDzrAqy1jvwDRTPdaHfq4x4/ZTKcttLSHdbzhJs0LUIPNJk16Iw+6DH+x00SR91u0nZHS9YLip7GpeKhheV8AinkCy+2YgSJRD3V95V3dgprm2NdSX3u4JGy2xXw0WfWZ/HKBXzSiE+2zR4Nb0TixdWOAepu9Ano6jxYT6PZQfO5TH3jT+YT6IxpaTN5mu7DPiijMJiMpr6tl/liyHQpmY0bSzwPxYYPYar73Y/pOw6bpUWoKTRQ+/Ms+ewejJ63y6Mk+yiNc5Cfs516eIV3vF/rhR3hEK0lvKn+Y8SrcjgEP+biv11Ecsh95A8SIPgvg5x+H2xS6sCx6nvqJqg5h7QPM0uthQa/8EZm6wPUrQM5Oe4NR/9zr9/i/5/3BiP4bebOyKfyTQJUEG8weYe3g0SIwfgd1On86SbKnl7Tw4ufg6Tv63IO6SeDBul9gk5IX+NNqmmSzaRLMJpOSwwZdjXtDfwHD4cS8JzByOULxSNcPYoGcixxShLgUwrKXhAAiS1lfrSoitcm1g2DB5AgRYCua5NRszfxkFfYuvbEhi6D86Z03rvJ0NQX+BDP2uwnia+ImybPCzDrPpPoIze6TvvQlPc/J49124Rdhj6rXS9HQJN+1MpLr+LNuLNx7h3uz/v+lagOD2M9zR7d7pZ0WskJsgf+/qKAkDYY2YDKChGYlotb7m5+sYZH0E7aAzuG2zyNsnGNXE4AtAQArwJDQaJH5UYL4A7dJYJf/9/847EFYVUAjQ9VP+vAXAONgwIsDfcMJgottLw/jJc0BWNXHUp4P7Fx44CC0xW1NVBxKYmi8JLHPL85UyTn1kMpx22KbjaXFrM49tI3qh4fsqxdv2AYMQTTI010WhAKiAm/kHpRWA1kBOYrTczoYzX7/jibhXJ+fs3IhxXHPN/6Wil+tWoqXi7m53Dpmf4cVQK4CMGB9IagXkuMSpSZg0W1F1IBUuZK0tcKPLjNSs0ctNukGpqeL+WkxQcQaZWoGowYGilIoSsXPR6MWbpaVKhxuYapuufAP7ggnCj0aan7bijRxKxL0QumHazxiUkPRclbTAiomdRVC0oyWW6OpDt80aUMP0a4RjsNVbU5LmlpBXqa4M1bT/q98ux6UqyKaSL4wykVRFb9a5YaRljzbIs+qDZpWONn2hWC20wfwbVc8mPXxMzykL1WjzW3zlqHnBD5CMfsQqPga9hjXYmGDSgBsUniUykdQ2avSj9a4DKRrbADq4udrdwswckJOuLHABjyd3biJgQIeEhnZaQBTJ0QEJjUwTxPA38MsWh7G3NygBdpzc4GWJ/SDdWmaiGscRMOel/PtSDEghicJxsuhMB+6OK5WNTZRjd9NdEmZIzzTaRNi06mT3IEs9Q/FKXsINnVO7ea16aBtoPgOBHcdSJM2NDoycrtD7zsysne7ARkB+0xtQOlK4Q7YMoIFmmMXwEeaJ0833oCbYK8KoIl8MhHiJ9bDsuT1TMmD6GkkERcB7FqB6ZY+Qvi+4YDMZ/k2DKJlFBguQvjzHbmUsGaPQwnuPelTOf7ZAzXekr9HOZi0JSg+3ARu2Vl7Itz6TINbYgvE55VYoUGFK6gLN74BH6X4Xg67AyzDxcVuyYxfHgJWRs4h40m0KHXZripFT+QWGMBH4NUtYN3o0eqYCFs1mSiyM631bwBrb9FXCppTLnKHeUZr7s+/mBa3zxtQ3TEXJV5tCGYF2OzjzIcK05mntkhejVoKQwKCoibsuTZ5r2KsscwwTzMbforK4eLajUU9AE6WDeAIGrvgFjyfRrO++PBoVAclD8X2RcyUYES6DbURMsAUivhO42pVq7kibAw/9hTbObKoyiYYPaIOeZ4Vl/hbOxRxUbGXroKXUn1t8ufhHyVIdCCfIH+opss/MOSfmfKHH9HHLeSPNa3yxzId5Y9F7fLPgFEZyh+LkPz5B5v8m6WozbmM+J+dIkUblZOlaEOfYJOXYKuyXSwxqIYltKZgaX7Nq/fU+mAu8NBbxAdDacE4CppjUIFDSIRSJfjwSgSjt1iHWUihA9LCYgJsyV9si6ceXyWV4pikDqs+FrJdKquAY5Zg1l3BZUq0cSs4wIfLUcFVGCAQEPyIuFoY6geyJfBIhqSvdKuTSXGoqGlohG9OUf4dPLEyj3fFzYNSDeV2JQ24y6E/76fUMU6ihnZRxbTaTXpmNmTMsLyieIbycUG8Q57r6EiANwRnVyECI+Vw/XOl6ndp+p5G4wtp9el/rS8k0LWfS2jAQ7tatN7QXpNT5dp1LFJAcXkO4sNFlAd+ttCY6zmmhajhZnptNVcLkFBoSaFuzUG9IvTMqi2QCenEZkVDcMaSZlutNZITmslWTc+alkpz21XbWn1wnwCu/YpBmjmoVDaWZqoDAAcJ2ak6h47GdReH9S0xrc/U1E02uaaEzc1uJ0HfZA98lHrdzRa5PcvnHu6OG3vcsEkWOUk56/HNLCzgtFfdJREwd2NYe+YHsBfO2QO/AE0KClylOJmX2qqaYxfEOopZOyJbBPQBAwOVmISKSPBaYr29yV62kREnbmkfa2iPOop7qLwWRrCFZ5kWnzVQYM5ijAcAGxBMz6PFT33OGc/Af87wA1fnMtzXwVdrWC3hkrXac2nKYYHmWzLsqnyKQXyY2ecW6E9d0j33lXpebdHmVWrsRUU8ib1Hs6vBd21jl9XK3yEPg1N4SM4hVFF3oKvPSj0eW1MKjgmFNYQPVDNiG/NzJve3FMwqe/FLVy1G22lURcpjBH9FlOxCG3yvLTLmZhqtxoJypuijRKeXuFus+B44eMJ8CbP3dUGiPfWv8l7GBlDDioQW7GnZ+NjZD0wMCSrDeh1afJW451oDpKgO8DaDffoGuexl8waZ6yMUbNdHnPgfQB+hGamPgfSQaPqIvfjliGyJwKjaoI8uEb9zqVpARmFuNtCgbQFqW3BDbaMWzxocKZcfWIM+KJj/743OafuFtxn5oQdXAAwXpiMInUyVoWPqnvI6qdKw2Zin74X/GpPbjmQhX43lrkPDVVtLYxoTqeTYbP6FH+fhE3aVhf77lvAdGcbargZpdtvFtLcLNctC9Y1WNeJHHSp3OlVuUOy0007niN2OqQkwAyuaIPrPfdO3JFsCdUfJljd/jGzRyNxctg3tlrKFQh1kix2yypa4caeyNaZ+ub1tPHdy34PAzp43Zcih41ocnYkPA7kbfcDPgDwg8CR5AgXVYZuIMh0ff/pHtogydfICs+SCdZqHGEvhZwHQ98gT8kWk2YPWiihm66ig0050uAX3z4l+GGJYJvDhTnjrw7ZbxImJbLjQwsQ32RI7mXbidvgPGrbjbOU2nCJVLcl1ylKbezjbPlkIDomya+S49ELA3siHjVn4kx8U8YGlSRDKbO2+QMTd9tCi15i0Dr/L5PWmrV4dDN5xyh2PiJb9nJYFKFY2anafiuxbVafFa9qWH9kvp8KYmQC9X+1HJYaPEg7i3SKc83kHCDRN42NQvTizJRf8uuatRFcNP6uo1ehp3QCbNrQFmIjyXbYSIqVwUvKkmiS/ydgjKrYJ8ENQyWI8Z08noDFPmUgcxK8BfH1XT8euImuLNWwJXVK32CPobfYEWsEPwW3HNRUjTI+1Kfj7FOdUP/3129dfP3/77auXbyhc58oQ7eMvA/H/iEoM+H8jz/QrKVZgt0vq9SVBC0KptWF0bl8c9OAZVwPrLrMEcFXw1j1G6w4A2XLgtHXBFD1hKHss0wgFielJGVXqYQWdOYP2aqxOpGggN6UcLn29X8Gle7EfPSamfjfRMKesZDSsBsPNCWkH2CYTVxUlV8C9u3bTmbzGgag+SidvXdu6x6qaDn/fp9OOLjjv6Hsrmi/Pb5No82gRIlQkYgAsdsnCh31tulSQWwPRL8rjxaDE134mzhYjCBd0EYZHPCFTEuN1v10aDyllwMcEJK4YdI4ek0azdLda3wizOzhzImT/vAvGE4PSQVsrqDvh4EwDpNLSjOoZuC9fvWWhYAvr0SFxLjjcXuUGfQET9US2mxySmZ7DWAajWaezDfK0XHlYjjrkdIaGGT9giDWM44X0oKsXWg0F+3r5kfv66zqkUzknytWHTokit9TpnQZ/tmRLyxlRLeVV6XHtlNidnLc5Vs53lQP5kVF+Z7zdHWsjSXPhbIfxlfJcQnU6k4mx3N4PqG900Q2bnUDf7nE9Abt37cmvBsBziiWzJTB3plBZALo+nQRO16p/ULx+FXQG6nXIXat7R6leDbcc3b8bQxwY2jEAN4R+hjcDhVmIblOMYvUQ0S5BwuqqIP2mHA8nTJSEiHMDum9JXqXE4S1elCPTZ/ehfsZJYGlEb+VDQQITw6iSoEUHmiIRgOqp4+sGzM/CbewH6E8X7nKqKHKmbgC1HQw8EWl/oVlFPGI2V8yewyIofGCWs1C0/zb95e2nnQRepgS+a9jmoB98O5kgwu7tkmCNZoj/7BEEyJSwFtFyqbk5RVGKOOKUzCyub5W5TLFlAgmz2oKo0XEGn5+vwwBW8qU8oKf1s4/YER4p9G9LsodJ/2cTL4VLisc25Qz5yYFnavOymoe97tIfW04qqTZyQ/14QroljWs5l0UmTG8XaYnRbNM84pc88SlGhPGCmhgspZUkVKR1dIqJYVob6lBBLdfcLMRHPLPZV5SeaqIBbWAZOkwGvTBzZ1Tt2ZntpJdV2nuet0xpofRcHdaV7RwJd/j8EhFcVLwn8lGWkpmrmU6FqknbDQW2whlxEkEU/LM9/JJez3mBXEoLdSAgWE1zXXxuOCEQKH9/WXzmAlgkvbLRBvnxITzDuKLoYL5OdzHsRMCQg3LhPR55upFnLPzCLYly4ahpxLzPVaLskrPreOAsXPQ0QoRdxGPZuNcVKtYOmSjulQ1Mz5s5iSfSSqabQLwUmN7jY3r3kL0lgytDu1IA4U+YEExuJ5+k+Uluv0ikqmnGqlCqTifdkhFGuS/UBjVr5G0O6EskwSip76XMeY+gsb3JvEauVym7eNqwO2iOhfIVuK8U+phgqO3UsqQzphzd2z6kjKKHFjSbTQCrjHorq5jtEq62lrna4k6oqU6b5miKM6tTmvsF4p2aTih9CKShqCsYJwDjkBSgS3vPQUVoq50S6ZNODDUca+h9tNn4tyJ7Vrh1BfoEaZF/WVk+XCJp+qonArD6BasHXWup5PvamGRHrXmb1WTycfuSV0kZbznhLLXYGV8zRN9wf8sNHEi/bch1iHxcHuZD9i1u2kttJKc439BhtfoZCL7hbtidmOcg3J3oW3+Uw6vlYogONPoNaqs0ug20Ew63f97NHtCuG4lbvd5Fa+BuXBvtty7fQ19GY4/dHo03G1QE/XJsEJH0HdA2sod35QZQ6oBWkgJGOJ2XfqbH/nB7Sal58nZBeTi4PKdG2WgYV2QPxK3UVOYBx214x155Mcxuk9zEL9HIjBO9E388At7w8d32NSz2nLujzlDrSx9nEiz0+sFu3nM8yADUJ5NRJYVM3u3C6pjnhEP5dWBEeMN4prPSRqOa5Kcq8xy/mptlI0dtW0F3eeViULkuyjS3jYpTSDpNh6akmod0CzmrXH7UeLxlm+Z0jupdv24gHypKJ8S3kD8bhUcMRbXfsKcdj9kEzjCF64gMho6f6EPqbehaQtuAqjzpdnRGCHxz7IDwdNkmu4UBkWrUbJ0qgQqqvgjwjXpW7x2CP1kS99nllxFqs1C+WjUkRkFUWdzrEnDkjKqHHAl5mPfd6jxtjvwZJZtif25PVLegW9fcOr1DLVcqnJxH1+6+qMfgrKvab/G2tnjbceheLL9mfE7ofYVADW+Luo14+y7S5Xi7HzUxrvntIPf8okNXv90I+LvofegMZV4BGFsqSCqEqEcBRVCuDPTh0UAZnTDDfbxzubrQMEpWsXJECchdC+6J97AALKGXr+CNiDe7rtDFoBNR8ZdHOv14dOvWnH4tmKf00WUteLXNR1d65+QpfNOvlkl/EA6w6p0r/XKZ9MvV62t+uToNvuQaHjmXL87iMFONfRRXmcGa31xl/wWuMvKru1xlUPOc4nTneJPlqQ4wx5pwbhzA5RF8taqU7n5+zk1evVZGi2tB+0rwreq2F5Hv86keUDx3XmbijOnzUEeDzy7Xkvssq5tB57Q90nktqVEbSsOdA3oHaZeReWeW2Lcs0jxKZgQnZZ1KmuGt+SApeeMufZA8O+ROfJCW96p9m3DrfY+dkM1d7nDhNIdTY0LglJSD2evFdapfMq1eK9i33hpNBPWboy0ngtm7NHufs6sUCiBgu47yUHqS6AvHWDgOvLXLce03gkPY8qUMZuciBvS39bMiQk/pIEmj/MBvhb355dIOdp6G2L44r18yffr10v36muH/FOXlm6ZuCvBwEQaCuGx9As1+cnRKNy0yqAlzGEprfpZYIS23UFewkkHWY3+asAvH+qlf6qtqgK3rV54MRrNbuon3lpEUvm5sTHlcFodji+2hnASd+0FbfLwT9w2yTdw3rtQ2uW8+aeL+cbdZfwAcy10Z9/C4gPOiVLfLsCzWdlbgJsd9mVlfV7KGhC+69xZXHzxWQ8tPn32C2UUOT6A7t+644wSkudVLRbtdJ9qnDntN2hnU4E+35Cuoqp8iaGj+t2MENz1GcDf330rZtBwnkArXfIWs0LS7gb/Vd/8+x1cE3cfjBU4o7Oi9GwNfAmj7J0wAPyYkCgUKDLsz/Z3GlOfIEeWzwnixMQ/Wg0zyPFol+rEBCX9vfBeOY0QnwtDREY7DtQFB98Y3ennUbUfa12q3CtbTARhVVrw09OtqlqsKupeE99LjgISbnQ4l4b1G2HQ4WM7jri1uNtXM3pGuZrw+rGTorxuEKEcoOSoVXjB5ti4Rhe0id+jGuiM82TdTEtT2VVSiPVh3AiSafJraksZeK9128cndQRi7j2YtDcleftB6+yEgTGMHfgMxHxnEtErniFORrdp2RzBmNGSvwyW+kf7ZFfDveYgm+1dyYaCr424Ig4czuJbIs178LYIZp4R3T6wHyzgCiezF/3R2u1h7zMdm+GkmaosjlUvhWQugSXlOL03iA7na5EX4eGRfHsm50WlGx4BPRDgX7Qhnky7CW/WVIUHyla2RvRVvGa+JL7EfD0Za/AKIzDRvj05m7ySDCIXoOGqiVLu3zx+5aGVp8eW5nRjl+10ORoNghscrK4Dq0quFad55Da1cfNHUDDTxDlo6uhkregrwJOccGxdqoXTBRFOoGHjd3/iYt0QdYwTIaB4TXsGOHhVWMVsQDwljfDxLSlIntw6fKX2p632hun2pdn2lGbO684rvl0tR4j+1wKbFReU2VcTdzvfadrdKLVfbji6G7BUlcPM93uFlGK3WV+n9MFZOG9XQZbeZ+hojTertLOLFV/BpswWmJAUm4eQN716JxBvbZJbvq7eXX7+2EbqJLWoY2Inm6LPOG26LMcIx5WR25DDnapjqXUO13RvV8thTduF6IfQ38hQAMlZEADdbo9nyjWF8EUaa02iG2QKYI6t/xzRZ6xs3sRue/kZRLeC4oQvItKxtKc8NvXJyE3HfCr1b3E+0A+vNWUlRn5OA6mGy24SZX4SiJ+ZyAA3IxOLR+e9//+UT2RGVaKNxAQZa23j+ACgKg6Xzbu3JGAXavB9a9poYvApGfJu5aXpT5AWFWfAQiOqJe1+oMrDBll547BHPLwcTftG46cOMbMmssc62RTPLfqizTL4j1N5P3RtiEP4tuehuEnXv9HXiWl53a+7H3WX2ftSM3tFnmE2yPUALeDfN6/T6bfoP/uFjO9UbUkma+tt6y6WWXevHsbqSR76rzRPZHtsD3lUPFj4ESnQosl70ZhkcTaM40ZI/bnhhlbLm5pKDT2bte8uEd7XtLhv0UO/pLVzc8yk3cdlMM7KdCMVxj+45aCIkVjGjb2IdUw/sk0jAEL1mkytZ3h8jdrd6NePai2bDT4MsO2a5aBwqy6am49lxL4C6Hfl2S3sskwPPG5MdO5Gsvqn6vEv+pNFfXebuhMqKzNWlKA6Zd0gtLxs8NvtFBxJi8Hq/DKW6F5CCB4iXCb9HlFId+tUXAHl91hO5EJW3tVQvGG054NVo7LvDl/T9pO0CxE4wpo2OgB/LxPkinmNQR1NrPErh9hnMrFi9CzjRQQoMpPbO7uPwSRtOuV0HxeMhe7OOlgWaT9gX518dXi2XeXhvziU5kUxDr1t8FAZgAXEEhyAWR/FzJIqX4B60DFdAEQM6X0xNiGu3KWufP9Fz9uULdyh5X90mltI7MW/kuWgY7ol4R39pDr20EluwXANILR1/ukhYDFGdVodL1TjOKjyfHP0zhC1nTlessf/869/yd3VVYMWidH+1LI0GL9vi7SM++B/oKPTqHaVesvMqhJBTTZiUAVEYz2AjLZ6M+aMZuSiJPBGSv85aDgjTixjvnMfvdB7f/KgXv7Zu2nAbwqyR7QGy/RI7emlju5xvlEmqsxy/OvhNP81aQIMBEkQr02zW6c5omAui+w2XYppAAeXWASgKskcp8hKvFZZR+Pr7OIkvjanaJY9P1mybPVZk5dWbjuRNPmQ5uaiW90SNCnGMw1jTDWxYzHX4hUuQ069JEWfbh5Zi8zyCjh01lWgEfNUrlbmlzlHKctmoLLzHlTmJT9zTkn5tUxYxrHuiLb+5F62vYeUCyy0qb1zKra2UJx9JNKyZNIT1DY8vXuQ31LCK7qdUNnAycb5jqHWz5B21Yah2h7/xSPbj9mON9XekdhOTvhR+ODGVcOdeiKnszgcTk2vL9Qe6CSKFtWTjb/ENmN/7cVgU4b3fcjX0uuEiCNxF0VXtmy1UEBHfQZGKc4dAbVt7Xym/Q29Y7tl4xpOqkaNz2Zd3b4vnQZrk2OukiA/uPKcNiC7axqFOMkeaMiJNlyZxwje8E8LFrBM3bJ/XQA4Qb8wUDnBYE/bzL8dkufwK3yPesBeI1haUUwFr0bXj4Fm0h67tS+Qsr8qmjy4XT7RXVxcD+8f07zTa40Yk3VsrUcKUKobj2zeiGCza4fVZvPlFFNi2lILUdIqlhnh3D4xV3KIhb00WcJPfY4+UZx8JOAk91o0aqn5TIixVafS1PWR/2+XyJREFenvgGzoX/aDY+fI28sp7tnvv6egAMeZ9n3OKOBgV4SbveZ6rLX4/6PFQ0BWXxUa946Kutfgpp3F7Kbr/DxngCM0=')))
# src/solvers/advanced_solvers3.py
open('/kaggle/working/arc_src/src/solvers/advanced_solvers3.py','wb').write(_zl.decompress(_b64.b64decode('eNrtXW1v2zgS/u5fwU0+rIU63rj77saLS4pmu0B379AW2wKBYcgybQuVJZWSnWQPB+yn+3w47P3B/SU3MyQl6oWKkiZtupURxHohh+RwOPNwSI739vZ67mLnhh5fzJIo2HGRfDmML9mfv//BXq59sWBzN/XWLFqy1BUrnvIFU+nYMhLs+Pnjg+Mffzp4OOz1XlICP1yxpR8EB/PVQQI3AT/wogCS9r/9BmgkbxJnwASnZwPmhliCOOAXKcfLrfeGp8m4x9hoyNgp0HkSekGU8MVzvvKj8NeHL6h0Bh+s4zKIogXD3MJHelgy80N5y+Yr5vEgSIDcQ0XuJNqGC6jWSXRxssppaXKYXediSSp8Lw0ugWDiLzgLoxBaxebz6AIofqkp8vSc8/CFu+GPsVHPo3NF16Q4l6lYAskUQ6DBcQR1TYA+E9E5VvMrC1H4dwOi8AiJfq2IPsMOStKT1eNoE0chD1PNgIxouuZM9hoLZGogEobcw46HEgX1AtD8RtE8hroW+cjK/Dx+9izn6bmfgjSFQJ67IgSiVG0g+C0QfEElU4OPH56UqSJBysWO//z3f0+0EJEcIkXVPZrgd0DwOd9EO26QrfS4oBSsDxTnK4e5uvXb0H+7rdD8Hmg+IVH9lYvU99zgxBWJSRRpKmHWeandO5UepAlZAJ0DrPYFUmZJ7AJLR4dA/JnkCtR1uwmRv5UKSw6cr30YlUv/QnIQEifMDc7dy4TBGCS+8wXQ1KMIhRJKqemqOnkK+DLdREn6hfBXa7rSjYkhL4gqUNYDCiWzPeU0ionwPErTaFOmDC3p7YFK6i1FtGGz2XKbbgWfzZgP8ipS0BZhlLopSGDSU4+8KL6UySFzAHKKL3X6xzDYQRUM2IJDZ8pk6WWMKkql+HuMGdxgwJ75STpgL7dxwAfsBU9VJYZDLxJ8SCpl6Xo8o33iJlw2GZLT93OebAOkAUrucQQZLuDmH4IvfKpVgd5K+IuZFy24p+nNXe/NSqB2mklpk8lxwIEeutTJ5D0XSlv3ej8CJTah6p/RP6jqdNrr7bM///gd/tiaBzGqa3n7F/jr9RZ8yWbJur8aM2y+ww5+kD2HjR+g/p+iCWHALRAglOewvwKz06eLs8Opw/wlWzEeJJwdOoog2JmQF2jiRYHQmTgbT0nhCNSvq6nKOl8V8kEFZLZl4KbQOWc7Iw9d7kjnT03aSlj7mMcZ4tAAQYAhEvZHDlQZ/rDSRFHWW5UtSOz6XnoxLooeylgylpKBVZsOWAiGYox2jappiq2sbwzSmmCFc7Htu2nKN3E6G01WYK/VzUO8kSI4Q6IT/OdQ01bUSiy70DyzsD7igJm/mEClh+p6QIXLMpMJVUSVsJi8FFsYlKnPxWSke4ss/0xbfs3+ASibMfJfNpCnZyW5UIIBWubk9AWjMeaiiorEgouDNNp6a1QPWm8/UtVPyDLq0vB1wtMh6iqk9nTAXgHXSCQderLzgfN1xUMyyNmXqd6OpWaCh/StHmeyItxwxftPHVln/crLX70yXuEHBKQv2GTCDhnRgKun7ICNGGXTz+nqFT53CICtzsT0zJviU2gYPumDUvMcUM0ploWN6ZWsOz4cuotFXyZ1HrG3QzeOwe7pJ5QD7BQY07d5dnwHzX07jKMYzYxqsm7bApU1NbDfPxgNYHDCqNXfhwN2MFIXI6fU9BByhkhasAdExsNvr8yeQ3Y0gbTsCBiDLZX3Hty/ki2XdMy2Kx6FwKRQc6mBH4pAgSP6mTkgIIMSZTLpMzSVUvX3/TDW4hxt0zrJ1nbrLNd1WhifS/I5OWi2ru4BCbu3RuGBdgkORAH5GGn7APcStDBhqqEQvXAyCmFkIhtFbEGvSS+p5v0CynRYqJgvlTY0zmGfydECrXPGZg7G9lmydmOAs0SXZgwMqsiSN35MZKg6anRh6wsjqjR0UN9jefYRpFKgTSiPpR1wfwfU8TUNEOoNeVkWK39njB7I9Vm9kMi6k5xEO6fwmgeSzGcZGXkT7apkCgzTn/26biHWoSi7LAYopWdmxFbdKcgCWTHql5ExWPNyCuYU5EPnAMYpKQbJMmQ46ceuj90UEK66WnYNsWVabJkhjK4noiQh+U2F64eopamIgRQ7kjctakiF7Ng0k4oY+1zWKZcFVNqV0ReffQ5dvk0/hw6Ha+hzdTNf5V2GppiBbsBiix1UZlpWH60NlgU1QK+0cadqY99jn6BU0xPHQeEa5e3sEbz7338+kT9oLMxj7M6AT4sZvb+V5wBe4MLIsPOnn89VnGy8nRZcHagiBHdBa8xB48uJEgAehHEKG7E+V8Qz/4pD440I/i0WEcwy0ku6Q32AgBBEOFjSmAfIWVEre/YaK6qkV+DWX14SKcRQDZZRun9yRTOPoiAvFer6eM29N9AwgNByVLMNerd4MjZ9Rsoggl6Boi9TwoM09Lah0qqZniHFrUHhpAJKobZFrWEARTRLBaNkQXwtUN/1rFXJYlVJZUCSMCCUqttTn1Qlz4ye7t5TF3mGdqkAm9fRNliAPLiXcFtLEHndoijZ18XC9gu+P1XWnMM8Spq+XvuyMmsMBrjcpKK9zcSiLOA4bckFmQarEuPyZK1oF82p0jSvHZk96GGaNOH1jMyZaZJwGNOrca0Zmq9mO1caRpyy5obOyU0kZZ+aRLUtUrkdhRIsJbCJLoaggWkBUfLLKIGKq5pWcuMq61ooqJez/1dSDGpi5nop28s0FObfA2sPs1XIuI6ChXRXI3ooMq3Y7spsChmKXTbUaqgBHEj14zTCtcK7ffbzNknZ2t1xmFijZw/uENOVHdhFoHiR1qgas14mM0tax+jyYlPDTImdqYFf0UlVNdQ0SPOSrphdQhOmdWzXVWrmpyEPx3EcgDhELEW3NRWeGEMSR5OBCXXnpz55ynFMQbaZzFaRA+Rd6hMOw9nnIcxaAJumfrjlLTon9Rs7BQgXXoFQ4Rvpjyq/vMJUtDQXqlmpf3Xn2FVxZl/YpKpdFcM16MWZXq+sqlSaelWinhn+rYHOMJBjknxOnx4Wfji0rWR1QFgD4Rrm2FGwayJhQodLfFRe/kNDM1d04eICF0XNNZ5bQMM11TahMK48KgSxavQKFb2O+b+pASeqTkuqTXROSlK00f/kRVcqgMBfrlpx6RHpeO9IR2sKqFajmtj4YR8TOQPA9BfZpU+uDf2ULu9gaoH9Qm4oBArUSRXkD+2gVGVQQ5AybwzYFjECBQzf3gihlFxr/thmD3LATKA97Iihs1WMyLh4eI/OVm9kBd0yb+5Hq+J8W1Zps0rZyzOS25gCdED/PQL9943QbwUyVjVCHQasKIVaTNmsFG4RS8JofcBGWNjVoNJTab1RW3TZ4ch7hiO/HDbvX+rQZIYmbSyqxZT0fYp7sVxvjchlDFIN9gjxo95ZQwYq311DSza4xw63X0EphX1Hj7IljRylnjx5+erJk18ww0buqZLU5Z6qfBElX4F7R2Bq44AJT12cgFvwaRlA5Vsr7Fi0oLtWrfEOcByyIZ4seVtOsReoU0z2fp7YeiWOEj+V293yj34I2gZ3K0BJ//zXdeEWIi2opQVbWVcPs/dqcp5XxapasyRnuynBcdDY0xt4XU06ZyMk5VUavcMNNqBbfadQuaGf8k3Sr0OdADkQG/r15ZZXapVx8p1G5/Cqnc1ptDtK8uF9ZbJAUOzdZwz8IpZ7KjVGkGNHTRsUTinXJ880KdatQ5wfEnFKkbgL2GmKhsSPipQdhk473HMV7vlq2LzFusM9NtyTsagF7pG7kg3oo7b+yi3/+fbfjw76ZEz4UNCnAVy0AidXrB7sFHj6AOhEDJi4HXQiatEJbkS6BjhJAZzM7djE4GMqscn8rwBNVAl2WNLBjw5+dPDjxvDj62HjYawOfWj0YeOQfSGv8fSap6kg2iieR6sBD++OG2zVN2GDqiWoLDzrETet7dExjvrDDHYYsfPBzvFFi6MIpOKgKuP6gqpO8HdccFq2OXVAda+1k/vs5PRF7RvkY6tGmJ+32QkMtftl6jzSFTBPOdRmrpxxqFQJM5dOPTyiimYHBGQSx76jpuZwxKEz6Mv/hwM8FwFflVMR5U92SsITD+iQhPegdEaipqcOjyahOFLnJeDaO8rPSoSVDisel5hMroI6hpxmByhqzk+ENu5kO9g3scN+oGsU5AZG4GtkAeQoj1x8dQcLxFiUGm7SqJYHvl4vdsqYR4pz7eLjx7mZlBp0k92d77JXtFsE7haB3/8i8JWjXq0JX3fFdiDHb/1I6hZTPwiq/2ZYDofQAXkN5A2mNDgO60NFZAEZbA6+Ux/kDzAYD/WZjrWbsN+4iHJKiEiuOPh4W95Co617nTW4s73/zwGsQ8fL3Zp0Tkf3NmaV4S8An0aonPzF1iX/sa9FxGllV4rb5rUJucPd2EYpBU/h7Vume7j5vNsedM8s2rfD2ng8nVlDs1bHGbtto6g+uSlTK1yFKEZjOvgjDz4fZ4YqjdgJKsj8jNmAEUW0ZlG65qKwXJYYRxRvwZ7VNdL0XvnhEvgi+MaN20yRm7aom66s/DB4/3jATmh3OB3wCi41g4hzFEMJtD1awrpD4ATBJTfqlsOuDhVwrXABN503W474lyfWpTP/+xK/JNGGywOq/XMR0Yl4OvLmNNBT3hrNGbvXSiY482ltLtpZ5t94St9I2dAca+QCJg9uGof9QaIwSlMZiaiCmkAISgxUV4YoUENBrfFRrILK8UGEnMcKJwJboOfwwCguRJ8owMDItxTwFGNbQM0GGAIjdkXqu4Fzp+K0zAUJxeBYbmvPzSE8qmd1g6GTY6qDppop0KE1GKrB11A6RqlDmZgQcWw98imyDV+m8mwVcUIVL+qxcbH3sVEaEgmncEhSn4BlGzx76q4E52B65NYLJRs1TJYknRYjT6YsMfnmSLURjjbA2FtFqm2GXBmpnnQw9b3A1O+GtiiPHVJFpGphjh2sygwZ4ipH1hxLTwrZy2ipANlrijOEvlV7zIxbAKSWttRi0iizajeKpqX0KFBZWEJNfTQYsrIEkqOv/SxOVKqjel0XiTKFRlk5CEbJIiEnaX3P3zWu46mkuJRnMTTlhHaT9FojQZ3WigBfFxGgdBDW4r9o955R3+sq6nt9bdT3ugN8ZgS2WRopHfH+oJ9XhX6oo66D/byrsV+5fRpxeEW8oXlZTu5cNZzKGf7CQO91O6BXUpkd0rsrpPf90Bp7u4N6CPVs3LnF3fqFeBeWdbss1jdu0CcnnZuqAvL9+pWt/qwPg2Mb+jBYN44k/ZM8FaCy4rLeBkaBHwfFnDAstgm/w1MANr7ewwMAwBfcXdAUxcMea8OwDQYdhx2xh5aj6SAlAxQPKAtDbZi5ZMQN80mT2s4IPWCjekxUUM225aKbbJvvdsx3u4O6zfL3Ne7r4dD2mxidzUebb2HOVWuR8vdDkph7/tL3tIkF4+/jT1yYvySyDYLLL9RqR3Cpt1aQHS9b2ZfaroOGUL4h+mES8jSreN7WMM63YagtzDDttPJSoMpJru8f0o6g2tXKQqszXp6vuSiFd8y2HmmmRJ63FaIUVxZreP9dT22inuvWkPvHc3q1i4gtQ59XFhHzsOf1QVgpEBmFEodvI4J2Z4rvxBRD/hnU6hpLal6SOUYKg/NajpGklWeEapaFYRfRbzwkJwgwz+YdoSyNvJW/izZTAe90FtPTqCUgT3hFTNtxxipQjwOY2UQJlwJ8jsqEAxIXPFfGfRnAUCsVmnLpHO22uhFclVMOC9tt7pjCyYOCxjGbW6c8PPbDhL2y91dL506dy1RrpGw2UD5QUBOKVzmDwyjbfAPZZdxHaoj+MbU78jx5el+GKU6kCrOGfTbJW9Rqh8WH3yvYUhCOQA7aHLN+2ny+uu2Gwm5T4YfD8+qHHEq/R9dheb1XvsSY9gHGGrxwtvhWhjNPou/+zxGYF4DdGz9J/B1trw5b/MjlNsFJQ3ipKWY/Qjl0bmlbfYkt9zDqmPR05SFv2wa4rTjdmvxtFF6r4Gwr+Nmc3g3jZ3X+tQ7Ud/61T9Eeq2DypV9x7eyxtsclxlxrKa3JJF+9vnZbJ9JKLbi3i1a3sF7VuFQFLbuO5WwX3akznJ3h7Ayn0/s/HK2iBg==')))
# src/solvers/advanced_solvers4.py
open('/kaggle/working/arc_src/src/solvers/advanced_solvers4.py','wb').write(_zl.decompress(_b64.b64decode('eNrtXf2S2zaS/19PgWTqyuKtRivZTnZXjrbWM4lj1yb21th13iqVSkVRkMQdimRIama0m1TtX/f31dXee9wz3KPkSa67AZAgCX5oZvwRhy4npkig0WgAjV934+Pzzz/v2asr23f4ahEH3hWP4sfD8MB+/ue/2LNgHyVbtrQTZ8uCNUvsaMMTvmIyIVsHEXt6cX769NsXpw+Hvd4bShAzd8X9xF27kNT12d6n9KvTxI4vme3b3iF2Y9aPeRy7gc8eW5MeY+MhY3/mPPwOacTJebALAx/IvKayGP1Bpi4hDQt878CSLWeeSM1ie8dPncADhs7PnzAe2TFnEXwByg/zlJ/6h3NMqBOuoXx+ztyNH0Suv2FUAFB8BBSfeUGweuZ63tnhJXc322UQ5cgixTV8ZtwG6XHf8YIY5LHcAFsbrPb1b5m9+pvtQB2ZH/iny4z+Y6D/3cXrw27Hk+iAovBA7jmGif7OjSKoscrNPS9mW2D174Gf2B5UxPWTgKlPQPgLIPzm7HaEIUniOkayX+b4veC74KpAVJCN6EuebLK1E7aMuH0JJFgsaQDR3+V4vTXRN2c60d8D0fPAR6FHrnO+5ctDvOVXF9C6Ke205dLPDFsfunsU7Fgc2hH0LdcP9wnb2dElDAQg/Acg/LVrbwLo3n99HULRqyKzRDimTzq3otWZ7QXQw0C87DFbSUJAeDxCjqMgjr9zfY79rURWEKZuppPFKsBQhOG43sPPKLhmvxGPUOB+5wNtHHMXwTX029dQQY9jESYRV9D+5sWb599cUBFI/ZV4BOpAGgedaLPvXRw9ScWgk60nmg11DAy9yMaRWxgU40fUdPB8dgCeX/grfmOUcMSFQIlp4OtBXOgbgXy7Cnaub8PoUyU8zkqAf1qWAI+GEsTbUgk4/r65CW1/9Wr5N+4kZ4exsYQNylOXekDJ2fLAxix0b7jH+o9B3/k+vOUrq/c56PEe9c/FYr1P9hFfLJgLKjRKQOP6QWInoHPinnzlBOFBJAfOPKCBH1X682DvJzwasBX/Yc9FsuQQogKUKV6FmMH2Buw7N04G7M0e9MiAvebw/LXrJJKV4dAJIj50kdoaFF1awhloZ1FvyET/XvB47yElmCJgfCb8Bn78JeIrl3jL0dtE7mrhBCvuKHpL27kEke19fI+SFslR0cYw7lUy8ZtHcqLr9b4FSmxKlZjR/4DV+bzX652wn//1T/jL4i30xRXbci/ECU+8/cX97fVWfM0W8ba/mTCstMVO/yhaDas8QIU+x0mYgYyg8/jM435/Yw1Ynx5mo7nF3DXbMO6B7htZkiDMaT7P0cSHHKFZNJvMCShEiAU2c5l1ucnlAwZEtrUHmnvKZldaHnq8wsdortOWHbWPeazhLogTaP7dLvD7YwtYhr/INFEUfMuyI+psfSe5meQ7HPaseCL6A7I2HzAfgMWEQS8iNvXOKvgFdb6KkeGss/btJOG7MFmMp5sBUz8e4g/R8RZIdIr/s6hqG6ollp2rnl5YH7HTwl1NgemhfB5Q4aLMeEqMyBJW0zfRHgZk4vJoOlattUbEsoCK8sgNIiX+AUzkE5S/qCBPZoV+ITsGaJizZ6/FHIizFOCdFY9Ok2DvbFE1KO33RLIfkypXpeHnmCdD1FNI7fmAvQWpUZe06M2VC5I3FQ/JIGdfpPphIrQSvKR/5eu0r0S2v+H955bgWX1ysk9vtU/4BzpIP2LTKRsxogFPz9kpaFnKpt7T01t8b4E6XbHNLJrPnDm+hYrhmz6oMscCdZ1gWViZXDGyhkN7teqLpFbp+w9DOwy5n09wvXU9DtVOU+M3qP0PwzAAOa2VYFRVV6i3qb79/ul4AGMVBrH6dzRgp2P5MLYKkvAhp4+kI8AJKyoG/nWK0hqxr6aQln0FcsKKi98O/H4rBCHo6KKQIvNBZr4SWo14JIE6AeWSyOEC+ZWeg5ElJgLUB8KMiM39nYa61uNMHVAbARdSO0IuBCt9KmQgOv4CeqkYzzRxa9Ozbp2kDNUPBReytRgOSI20z/zuw0D16c+q+zRxVWoWUbGpImD4vgtb1EX/Eycwo2O9BAtlmmJYULIyQ1QoZsTeTGlwsBiKUQpAJLa0WtKgx04+kk9/RL2Aj0760sGXb83Fi3qDEewDgqoodjNzUGBC5CTEW9KSPIvRI+tiriw2RYtkd1MkpYZMh62TqhahWwzFU4/OMsjxBS/zg52SyeFu+4djRrtxiFeNcLJ2YvRpGAd2WiDrAxcSrFs1QxtSwOCasn/IsVUaseWBWhqbP91dSVC1oAxiJ2s9V1a4bsQfOZrTkYyE8+O4dgw7+6hh8CKz+6g4ZveR4r5Qt1ajSR9JQMrcPWu+KqUDX3vvZlAdNaByg4kGUdUYoikTjJ2Q9+105Gj4fBkEXslAsC2cy/Fpacm5H0RvYzvQZzIbVAr8oZAoYd0F4Dfba1mcTahBWGb/81+/kr9Q2fGwxin66xJG709F+93x7DiukU8/8zNYqYL/c5P/2KDdn6BiI6sngEwReXiWG1LzRPZPYRSEPEoO9Au7OBp3/Zh7a+rOYD5qAF706M+r+ZZUxeQWht6BKKGWDU2zmvLGCJM1K0jp/Qo8DNSQjqVrfxy/lC2vfCTLLyFn+v4EOAn2YSakGL1TyoMt/izlxDwhr5CYKgxzsETZOC3+lAesGrxGpWlgTRUxhBYCgdlk2Itss7mV6j6kYGm8n285zDpgnINysnGq97nEsbs9/NzaVxyevMQFHvUqovOQPJa61KBjkJ/kirTdOHNXpMxd2d6ex/2iUtfkikyB9MmULhdywp5p/kasnDlYkXKa5tQgCDR4JoJgj24W6cHJfdEFD+QWBIX0ysAMuYO6lAwIFEGaQ4jCZCyIqRfrROwLySsYhc7PajHgH1XVKdvZN1l5KJHDFFjIz4AopoXoP1OVtVeclXP9i6hNjGhDJItplEhak0r8LAEeUsV81RAZWkJ5E2A4FtUEfNW0ASgHd21QB5DKpBmyaVQb3/pcTzoAMpv75TPbizPho4OJIJ63Hkq1VFIhMiMllYKizo3gIDfrYwpRclY70uqybkXfXF7R6Z4xTeElkQ3injLykeHzIrRdis7kBECfmhXccrOAYUvAGb2V4ewBxXwezAVGD7FpidS8VxgEiIJlbstCzD5uUxo1P2UCqNTLDcesrNKgK7ZoxuaAwTNIWPBc7n8mLjSiop1lh6ugSo1fT1nrGthamh2iKpe4NEiwzWBALaicwnARykrveYmb73iScUwI/Q5Lbl9hyZmaK7JOqUlDpmluR/lO8zEPVOaBqAH5fX99SPbhsDoI3wHZApDNi+cIHGt0TeTXL1gCzVKI9Z5xbJ7t+4axRj+PCcUqIBJb7Cv2sHnMliFFbIATt0NSFe7YFi5ZWRmg2OCYrYUiHbro0EWHLjp08Ymji0fDhgV5HcRAiFEvIyPOeFa7htFNtoQ/hO8kWNMKsNKyRjSAh0SQ/vdizWxFwdWSAxpJfS764i/o2/uYUzm4yIOJRR7oORgqLu+AX+plooOYDU8WSggLwX8sZ5Sm4LIpXpJbXlERYE6LIxkOaOGdAEEWo+knjTWZ4034J12DMS2vAcnruRP2zIW5FB2demOrZZ7iD3xd3CWGNc1iWMSdqiGWOlXUT1Oui7pST29UmDNtepMVMqBiWmyoVbJXCAG1DHHJXlCed1SErZrdNsG2u4bPy0G3dsHzxuBbmwBcddVvFdk2RtyaY3KNcbkPHPA+YV9z6Ok7F4AsrTqWHnNd01WtE8//gVRCNcQTtTQOqqwWyVnGgCQt85Gez8k7Fk2bBUZHLjbKFhV9Zl5UZJbPLM03Z78BAG2yuyiUmUn0uDhypqZRpaVUjGsUDUiSdIq2CGIX5jR/ySKSOe7H3s5UmoDExlmvKngkvzcj2ab4Q77KNJeYSDf3Yd0Azgh2hnBnCHeGcGcIf3KG8ONh5c6xzgRGE7hKOkbjVyXRt6ZN2m+7CwOArzSX381EreL5njzsx3u2nS0adjhP5Sevu3u8hWwXCBNpvT38V0aIV/A19YuXSWifFbm5CeJdaS51zFQNISWISKlBAVdmRJgKBndglJJwj4qd6sV+1lgslbc7qkC1lkd8bbOa5z9oCpJKFLV71uVdp20Di7kBeMZaSe4nk9PxvN000YGxDox1YKwDY58KGPtiWLnbvgNjCMaqpFMPxrQt/ZO2RxXcFxSr4viXBsUklEH/1HOJs6JjsVpLGKaKMqY6Gobp1D5dGJZ18TuAr7RhO/jVwa8OfnXw61cGv74cVpxK1IGvvCdMl40Rel1UnbFEG2X8AJ0FEoiFdpT4PHqi7VBJZzJtycJ9OMN0tjtX2HtyhZ2wN1vQd7R0YWvjhFtoefbzf/63OtbJTY5ZlHo8gvushODoBz3vrqoqABP82sNzU9SS6Bg3gqlaSENBnNZ16YbQ0aG+eOaKkZxJI3fetw7+dfCvg38d/PuQ8O93w4rzIzv4l/e93R3+vTl7b/DPzHbnfntP7rcPBtvuA2d1DrQOQXUIqkNQHYJqh6B+P6w/LLsDUgikakVkxFOvHTwVFhWuhFTiNHEBrECtuuo0GnHsnPEcciySR/quqpQGejOyPCtgGc/3ZyuREVW6zE2uGqEY5JHeHPg8EBGiWUVI7vuy5cnW0E2IdfhdTITuITokvaeOY2FqA8N9bN6qlb0OCh31eYEcyhkromkXZi71L7xYe4GNj458zB9PrJUN7/q4X9xexv0IfXORNWD4w8EfjmVp0+TC43bkL7D5Fjs7bAlOs0OTChvFqmBqus+l/1ycVftb9nA4GrD+W+1nVg/JjX46ExaUP4eJjtFbRMG1YRq4783tZuSXcaDUfcU2nwzqFFobxTKQ0qnaQyT2alTjackx3WihJEcYSf2Yrdoc5Fk48skpuyJRcggOiLAZCmslTgubgdTcp9q2he/xgtPZ8jC+3BBMNNRMfjyRB1vljT474sQyKKgVbtdMghA3fS2DJAl2GklBq4+n2MsLD75ij0b/hnvutrjHMoGhQm5VCThtYjjvZKWaqJYv1CO4Xki64syGLCGMIDCufP1NEXBouXFf0b+z0fDRUfheCbfZ5qwaYvkBnx2afn+D+12eSnHLcaabj0owww3g7pXRIPl4rKh0uKtqF7S5aTNUmscIbY8y1hQpo5Q6C62z0G5poTV16wp7rb5n18nAbBKau3dn5L0vI+8PQ/PFRZ1xh8adUTRGo+6b4iVJAmPE6lKkqjueWP+vCnVZmjn3SphlRAkmVsdSUA9BVBzsI4fLQwbW7MfoNI5+nE5/dE5j50eBIANAkDJZP44GsSOJv91yXysciK3c9RqMUXVNETCL100NRPC7r40l6z4sNqNA37H7Xp26PpMIJUX5rQ+vKBgq86IaMhw2btTXQgXOZkACgOdbKmyhFz/XUPRfPNvheluDcQAzb8L6yZYfoB9d2wfA5KAdbaqCmEDySA+ansUIytQ2agOnCMxiqF3sZPaExoZoKNl5tavJjiylAXe2xJ7qyF1pdMcRHTSrzG7o5foBadBWfcFazbEEqdd/Vn03SRWMNR/EkIt2ZMmbjcP88hRagZJcB/kecG2Lk3KuIzfhpbHbQJoODcRzdND8JP/NtU0X9UT8CVsF/oOErW3XY3vf43EsTDTH3sd4dVcc73k9/RAUZhcL6ZB2h7S7WMindxPDaGi6h7NDyRQCKQumHUbOLinFG4MjTheJPn35tbyjVMPDTxOauOgyyVjcWEn3hcL8mE6C4gYHPOYL+eHxQDubDiaxFRGC8UWGLpfxENJL6H3s4yWXYteYzR74/PpBvjRBx3MvOXsoofALHF3oEY1RKH5iyuDYPltyVaZgScA4msIjdu368b3EQsqN0B5XDzLW1Z0N6XQlgwJkAnzE8FvdO0M/Go7+1eDt2d71VuQUlkYaHRinDh7a2aGGaVSi+pBJmrs+WVvgDHWMhYxyPKa/ELvXgMtCxCE3YJTFB8RFyCG4LhUtz1byTOIhk+F+iqZrgws+/qx6Jqid56JkuBxhajWYJhEdWqb4IZd1IQLVwnAR588plomIY7zUCkorQk8n96rSKIromkrHaXfXRGQ+eQ1vSq0mcMJe1CngJitEL7+gb8QwL7zS5EA3uBp4JiMraiGghroTHedIOo5jOM+PJIgHAYJx3nTktnC5FuotlbQAzEzcbmJ0f2TX9srZ4zskx67TwLx0DlBknubEU32Cynt/ipNn7mRR0a4LO1nkCNQfNFgL/aXlBhkrwHkuedXU8o6nl5wiQu0NKh2e02PnSHGnavsns3rCfE77fPJyojQ0HBd0UXPIXqibdunAXnONyzVzrFbOTUrzkESdLCZOj7OoyemhSidL/QlbRty+NF69Q3NQRl88NjtVWtHPmK1y6GTFmVJIDEJlVV94dKwvTEw6ThokjazbusvEzJNRcqyaCaR+6pGNS1Kvm2Zw7XGqJ2s9b1dtTu6s0D6z4Kpwfmd6hZlZW7W1fivym4/vvI13qwltv0/vV4mXzh32S3eHZTUoAKp8pLcIO4gl4Zc6ht+2nrZyR/uArreaXt/54j56X9x4yC6C6/PAe+36G4+j06Nzx2XuOLNsjvHIcReDQuSYW+/hLbrlXl1kv4V7Tlz4uwySrR65/tqNpHlICwCvKPpt6X63EEinbjdU6hM8k97Z4upjBhVwN/5OXWIJJsoDIPMAvTsPgNADDH7RfRTZomm2sxPIHYvQlgicQ3GXPEzuw7FmFmfJt7bIOK92s9XhHLI06LJI43H/tXnJ2qjOe4ddby2D1XRPCK637BUtiNR1Vb4Hs37ZLeHb5kXECmjWh2/1VBUwPxevbbQtTthTn64DpsEh1t17GCk/0BjCBbIYPsWPtBK+4CRpvclNihYymUWbtfyRoo0aLBAljMbIuJ6qUbRtYuFZDLy1lGq8Kyulj+LWjpXs0haxWlc9mW9r6acugniQmf2xJSLscs20ihDY5F6hRUDlC1xOGOlj/ECHa9h0VV+wzq3CfgIVOBCBTOGIZeMirIC5NYoJLvhxlEtBsOEcFB+iB98GWGemxQiRFzQ9ZUCrSEFAPYlQJlmSkpvn+QjU0EjqIUGzEEIYaU4ekaDKzzMqO3pG5OnRs9XFEkatQgfiBmx17fUkdWSndwoDiIfWp9dpL8z71JTSLlyQXdTfhtCByTkzMugAUUL9/dhivYzVK3ZEQX3DkxQRFObicq1O2BvonTBHr7JrTTDMZuibgj0gkas/TMGtrgLPqma8kFq50cgZquGCeItbWch6AMTAC5cbSR+32PoACVzS7mLjAwKfVLfLHVCuWtqHO6lINnWkUIe3JIXSLboh44XMQdck9csXwMBLMdK0+VDQLE2lI6vhQpiSWKxi1zqGnXQOydiJ8gP2buyIbR0pR+igkI4g9c7kmVPdb0aP2OvKTSjiAlptFW2tvJa0S20qYhcFvmt5PmFnoLHZdRBdsj4fboYwXgiXphv/nCDCQ5hgzkAHjUWLyMKIrwVOubUQYt4Y3vOlikh1goLlvfJBR9osmDMEYraPaSfQNo2YCxZhUOh0w08xyIB9fBEab/VKQV7F96O89SKJvKVLU8JNYEs08rWWP+bcT2Vv2q6mSsv3MNnFKgrMREG3eqm5qX2gUhNWJYWyP1Q331LXqMbMQKdb0j7KR0pyktKpcJG29HrhxjzpjU7Bo3EohLrgMyhKYSfRFQdi22LWEmrKFDsaVXsYVjAcSQU13E+l7XJGdNw5ij8mRzE2Z8E7rJlNmmNY3wqLeeq2t+UNNGMvgDKRzP34ejX9nbj16jtxj9TeKkOt8j7imsYGf819KXl91VDcoG8btK1OM2vABg18Sw1emHT2Pk00ZddCybuvK3Hl6E8rqGnwztv/C/L2PxwycdbX964fRG5y6O51Ljr8q8RTd5wbgm9hT+MyLgG44+R0HQFgKV42ynByY3SVgnXHA9wqeX0fm8BAa+39pMW6pfd5tIXgaZbmKC9qSJe1yrQNq1uzU+aEkNPor04jt5rBmp2Oc5eRvsvzA3LSAdiYZ/Oos+e6XUYdWu52GXW7jD4ZrPNoyL65CUETvFriqTxnh3GHczKcYxSNEeN8i3FXri1uCCgTBmv4je0k3oGNWejecFqGK7bkO3a0ckFzZh7Mux5Wa+T3g51Ve/8zd/UlASL48hidwpvtsmpPQHY1eu1N9Yq7j/YeehTK7W6i13JW3EWvdrw33EVfBEltb53vEFSHoDoE1SGoTwZBPS4iqJcdgqpCUC9vh6BeCuQUs75xJ7V1r6jpZRk1cUqw8Gt2M/tVxys6e7kqLqcctGWNvlUZVYC8hbDCDa2QjGaTeYauINX83Z18A9TbnVVzf4DpNqCpJXDC2hwBnKTMs/2XWfbGfOXNX6IrAL3S/p991MGFjyk8ecLEztaX+lqy3MIRbamhtsxQ2wgTjggH4tdwpHMzMqwGWSJC8OWZB7k6+9kghXE0HhVqkAOvqZoSpWMsyi9BCTNcHRkkkzLl9+r7NlCVaRujs+WVOYGfrU1t1drmKusNjhUXDFnGflKWwC26yJ1Owswjs0JVVBxRVqEDbe8AtH2hQnyvd9D7QNB4uzaIzk868FYM81WI6KhQXyxp0NrkU7lkOfB9QFy04k5SvpdQXwW/9+QGQ15jwmc428hlwaqk2Hg2M06GlK1lGC0VFoWqRNYBu+SHqWfvliubAShAmjez8Vxb5nDCvrcvQdT7iIsj9lAd22zvuz/ssxbISnH/zmkNThBBG/SJSSSYYUPBs14TygPzHGJB8Vy87LF4BIvLaX0x8O1u9sE+vqs7Ua7PVbUpld/F7ToY2XmdOq/Tpw5gvsQdyYkthrC6oRBnPI8n3d7k3N7keikZYYxKwsa/H/3f/7IopaEulzxM1AWUvLihOQlo27LIKa+nvCuuaajDp3gZOcz0O0e/rnJw91vJd3gt+X3cSikJAdmrO1xLOS1dS9m8Fml39eHuwdQvELr73eNigBx96Xi7qHHuFOz81ZqztBvNJ3fb7tvhxw4/dvjxU8CP/w+zaMLr')))
# src/solvers/advanced_solvers5.py
open('/kaggle/working/arc_src/src/solvers/advanced_solvers5.py','wb').write(_zl.decompress(_b64.b64decode('eNrtG9uO28b1XV8x3aA12VCKtLaTVI6MOA4SB2mcIg7gB0EgRuRIopciGQ6lXbkIkKcCfSuKFP2Cfkn/xF/Sc87MkDO8rNdI0pt3sdBS5JzLnPs5nD07Oxvx+MizSMShzNOjKOX9SXFir374kX2WbKodW/Mq2rF8wypebkUlYqbXsU1eskffPB4/+vyL8flkNPp2VwrBNoc0PY15xtPTS1xclYeoOpQ8ZQWvKlFmcj5ibDZhj/M0L78Up2cV3xfPCCdTP0g8wqfjC3FCDEnB4jI5CslgUZVEgExWPLpIsi1yJne8EBKwnk/Y52USPxZp2o+Vp+l4yuLkmMS4g6gUvBIsgvXyAaLcFwoZS7IqZ0B4uxUl7IJWAIG7E/Y0T+TpG1EAJNFKBc80HaRQ0hMDwi4TEGEGIOIB2/MXeZlUp/ExR6I8yzPaCq4cnYEmRpsy37Mw3BxAYCIMWbIv8rJiPMvyildJnsnRSN/LDntQE5csKxQYCCwVES0ycI/zQwYSV8+rU4Hi0o++LnAlTwP2+0RWAfv2UKQiYJ8mUaXZmEyivBSTBDFseCRqrJ9wKdSGA6b+fiPkIUUkXF48zgHgCr78AeSWED8Ovi3ILIzyWEQG3xoUuS2BVbwPStfLS7EFzsqTWaa+i1Kb6WiE0mcL4n9JH8DqajUajd5hr378AX5Rlai8nUgL1La6+z/3OxrFYsNCufO2czJvn40fKoXhlgO01RX6FAMZgeFkLBWZt/UD5tHFcrryWbJhWyZSKdjUH2mM662DEdAoLJuUVyDY5ZE8vIQHAIuXR7wsVzYpbWIewviTfS4rUOJ+n2fezAfC8IukCaOibohHaZ4Jhz5eONtYlsv5yuJhZWCzwgHMikkW87LkJwccbtNNbxuwGKxfLOAObPLueS2BKg/RHj0+t5D0M8MnVZ6ClXk1MELKUHx34CkiQIiArS2+1nmeujjYYsHWBl7yvQgp2twQHE2A+4gDr9Y1IyW5nxdVV3PXBYnDufIQRLsKWAZE5xhSiYTtvopUAQ4jUfmN+3oYtvdFFc4WIEnz5Ry/KFcMEekCP3zS1pa0hbQdU7GJeRXwGSbxApie6OuAiCuackGMaArx4tvyANGpSkS5mPnKxf/2l7fkFzY7lC1JjOx399bi/vpDcOy3Siyjj9spIUq5lH2S8pqc5Ss7x3SLfz+FiiaqJLjmDnLzS/AcTMit2sN79ed/3IW7mRTRoYI6hEH5kKk6pMwvZUCoBIc6KUIMSYaZFoDOodSQVZJFkK/zbLzeKtTSh5QeqxJGQlkABRYC1GUMQwdC8nkZY5YtxT4/4opqJzRLKqXD1/xQFYcKqi8EewzbZI/mCDsmSliAEVaofKDWiU9MXAFHiB+B0UmpajlIWAnFBBSCIAfFg/mBQgi4wF0j6RgLmUTKhmEGSZZtRSZKqnsg0kMGAJ7gushlQu48adj7BGJt3seh4oyqNJIWCgBjTYolWlHmWLNJIJTvBZQFDxweLU5IqlR4cigVgUsSFaiJzVicX2aXvIwntQ3QxceAHSqE6kTfMKZiMPOkSDcUJmHf85qcjmZnPWam0VFQ5kWRnggFJujCiuum9FIhuUG8VfUMZjcA8Ov7TwL2HO7j4wnts34C9rSgNE7r69vvsPF4jDFDGTdrDBnu16vgbkj2NW84asooQPwUEnS9muBDNHbIHgLLA/j0LKIY+EnkVCPwbCu8J/7cURLqQO1jSStXzlNQOWwlPPIU8w+sXeL6X0HC3LoLoZ7AugbpWzB1evZ99hE7dynjD3pmkh3EyL0J5QuQmzl365JD7UNt6102C1h7S5ofU2roaoC2WEK2hR30ADSE321TpjgCdVI/0Bpc+KItC4Xp4YLd7QJJITKd/1GrmNdXnUVNZZdf1lLsZwDIHZVKKIAdQWmkb6LTC2G4mIA/iCz2jn5nWW2IZFQi6+6itj1tdpZaAma0Q3Lwu+iV0EbWFhqCiSQrhzhLJlU/6DEh7fbkFC1PO6fsjM0XxS1iN6gDPrWB2N8UEJTWJ5UBHE+kOyEtnFMHpsp6UlurzCfH/OP3owFLbZsnbJaeWt57M79AvFGD93mPPYDTAS/AlDb3gEUrv885cCHZzJAn1LufgHIheHIsEI/Ymy5XvjEcDwn4nSgHDfmXGN4o/s/Z5S6BJIwBTyXaOu3tOOYvlSXaUdAAN6LHqx5JX0Q6MevA2Za1em6rsyttMoeFvWh5EXWdEnUVokcsS6XkgIWEHCFWvT6ktoHYkPNeWZ8R+NlcoQn61+yTDE0FVsGVpxjxh9byK7OWXw2v/b5RG9QrwB4ETJwTeR4mN+hU1309WivgN3v0MdzNXOEqczAFkKpjWJ6pqgLLgJ6KwzYE/NHVz7zjemQPFD47W2uCBvAdiyvvIoL9SF8bDFwam1FEJwnUNNJzPWXVx8ZE5mWF6Bcp369jzqC9u8Jm3jW7+GoaqA1NgUENC8tGLfE8o+JNVUObpIRSq7rMzfoAQ+CGpykOY7DiU1OwnUi2u6ov+2o40kRPsg01SzOLpdmqN7IDTwu9eFnb3oqN9Z6se6PXZ0kXIQJrA+1FiDl91N6dRvHRgk27+HuzgHqwCY1vGzrK2VaddZq+zWXvFs1asFpYi84YkVWBcFVsNjT9Flhas7L0INXYRBF0bGP2TXhxUbbMh4w5CXSEE9lhT9W+lTHnfSEsVMl50St67yIBVtB+ffZbLfbe4iQG7mLNXjoUWKmEhIVZpKrH0NQFMTVQjSThTjRU3ExR7VkJ2f8JFTjqewTfnw+XNxDRlooyRoiLqJFc10atKFX3QSZVFaWQ0FWafqzbA7XDlXlgSge3jO+rHJySvs7zSD7QmcnNXyZY9Ra9qpDRFeBAUuxnFfOfHqjfsMhtweu67Q38s7K8jtKak1Vd9H4bUsUQhMQk9yaQ13jvdZDkLuEO4BrqY2sP77bahq7LV67DV/3u7vLQ4/M/weVnwGWCjq328gt5dvWf8GszjzXjY1jpW63/UZTJptP7B4iwd7xrmkjwJHsoDJAE4/c2Ip9Bz9v4Mk5LqTlKNxM9enBmCBqIloHzIClqfFAYziAbVyiqzYZouqa30x4yuxMNe8RrTTaqkoOKF4yGvXgdFjyxpkx68/ToBl0XWk5B5tsF0KiUILQiiuUZCONQnUFHDtdqYna28m8YPtR43Zmju4008lMlFDpxg0JWIdFr2bKqfG0VVYnfZh4X3Si6ufeJQ9MqNdqzjbV5SxAYgEBxoyb32Cuq2+pNjRLC2zZpH3qDrCbt4oO79/hUfHg7aadJe4+khift9LpWvXmnOQi+in8pypxa3fcggkv10r0UUcWz7SHlpaoS1LD260y9pzcTdqmnxGs9hmbeHfUXF93x9cCZcqI1k+cQ+ovkSqRAS2KZ793RL/kVrTs+0hc47KZZez1upbMBGDzpXMBO7HHO3bwFSHP8bI+78WiArqs83C52uVDzQb2Lg+pUbCp2gOYIRyw/y0i6Rx//bSPpR0br+gwGaZ/SEFmAXeLCCjP0M0MQe9Jl5q5pPW/FN6NTf+UgIKyAIOqOtNoI5jjBMjjaualmRpe8BvcN0hXKfiuqcA27lJ5MXoo5o7IccEhrOkua6C3dXSKEZ96/sjvhhYR+hLvTTmkX02AwLysR0ywdufH7K/2YPSQ8A1Np5Kce0+E62FrPIFZzEneqV6BAzz5iJJzRTQjgSt/vEz6tttIfFIx0C+211sKToNaob89he5Y+D2pld2dSNXaaGJs5co3ojefIX1BE2eiBpgp46BxOkHK8RJXXGOIA0uoBm/MM3Xc5dXehri4d+9CkQnwP0GtlzYexN3ccPYP64ly/T1BimHd7TlgUnetus2+REtUW9mJeGZWzeXkOULN5dN6dJO24VGUzLMfK2vM8DU6jZ5/9hlk3pr4/4dnJ67FSRGSkbVAZRhBwCA4MouZh8OWJoyy7EB/uQix7QTvh+3WyPeQHSWOCPZRrCejGMpf+ptolvNCinUR5cfL8ayDQSECZY9DqNavQgCJcFfW9zdKS0UIdmA9YRtdM/MmSAm0svut9HVGaFzmqk7Bs+Kbud3fCnrVPANI7fezJxhhwHDd0vFDV9pQPWzJtXCKwrH6YQdiapyVOR32MHjAKeFrI1oPLrjyxb3WcpXZ0La3bRva2kb1tZP/PG9nrTyrrfnb6/vSDSHz4/m0/S/3ssMBe09ZinYVNHB6hevrPv39ldYaqYpOi4Op4FHS+07Hd/FDrS6g86B9lsk5PLM4P61TEfqDOj8uITs4DNJ0kVx2sNC0shj7wFLq2G2aKDnM8kJ7ySDABvJ9U5iKk2N9edyJdHbCC7sgwD8EBSUGshGiUW8yzV3/6K5v+HA3ssAL+XX1st2NFnsS4zNcHWVmiiOlsFZaF3sP7U/Zrhl2t1GXvezRpd8p04Z5g6e9n3U52softez40X9PJfd9BpdtaRDXQ2XZ62ha2Uec4lWLQaWQMqZv2MYOtLiBSB8d+mSYXEjECwAO3mtCTG3of0e2AE+uIF7aTvb1vos43FXKwqNfEh0t4t3/Vx5YS3x8EuG47158Q01XDa1kyQkluwAOevR7ddM+9e/3pzXptna9v1mub/cWb9U/zfZJx6D4oqFKD4KkAvecn879FEC5LzuQeA6km45z9AlDqssw/Uujaf6iV7v8fCxfdpYVOdwxDTfcAuhrfHmvccq1eKxI3/hBnnc5FbW3l4ooULt3SDbFFuOxmR+1r1VGp5s9RqKbzZuqEcumra5Khq7P6ER1kosDveWq7gWbVN+eZKLfQpdsTWsFHQfrdKcmL1pq+w6TIrOw9maS82BzPo9ex2BHC3xer/thjNEtvtrVoBxdGzsKou7A7pKkFtwROXqwsM6VdvM4YlabuTdgnhySN9cn3uU68VN5c7qBKesA2SarVpuYc+t/wGr11W3ejR+jILwSJzB+Yamm5XDfT0hK5WWtec3Xblt+25W9bW/4vi42zmw==')))
# src/solvers/advanced_solvers6.py
open('/kaggle/working/arc_src/src/solvers/advanced_solvers6.py','wb').write(_zl.decompress(_b64.b64decode('eNrtGl1v48bxnb9i63sIiZNU+woUgS4qIl2LOC+9w53RexAMZkWuJNYUySxXttWnPuUHBP2F+SWdmd0ll1+27MgIrqmQnETuzOx878ysz87OeHzLs0jEYZmnt0KWf54UB/bLv//DFrmMhfxO8ttEHT7lUn0iAMazmK1lIrK4nHjeB66UkNmUcbaRScy2vISfQAt+q7wYr4gKk/kd86M8zSWbB0TCAq1ypfKdgfMcuEUwYise3Wxkvgd48/K7CWPfZxmQLJJ7kZYsX5ulOeNSsK/HUQ7LkRKxF+W7Is9EpkqmtlyxjVBMJpskTg+s3CZrgAEmYE0kEpnd5aUiTlW+j7aipN8XdkOvveGivSF7yoaeFrx3z8vxm4l3dnbmrWW+Y2G43qu9FGHIEqAvFegvyxVXSZ6VnmfeZfsdGA60nxUaDZhMgS0EsnjvQJNgLb2uDkWSbezS+wIheerpxckkyqWYJAi+5pGoSCx4KbQjjJj+/ijKfapG7IqXN+9yQLiHhw9SxAlt3qCHLhJGeSwiS6+2cEhaNeBSbJJSyYMF089CGi/1vFdsfLoPUNuKtADvPzFdLxZrFmaFvwmmIANYMQP7TLiU/OBvRiwGI4gZvAFF/+lNoMFVHqKefF7j8ImCYCmVb0CiFNysCRDlxcEuI3oZih/3PPU5RFF3c7NY8cIh2KqHVWDolHwnwnLLCzFABvAmtM5mMxdfv9REVhsf+amR2yb3K4kJjiVrlpRJVirMS/SOeMtioh4wCEJBySYw+pXkgX6k7kf0vgR44DyYegw+BbhiyWZsWfukj0lrV6jwYgZGsA9v8EH7V4j4MyLC1hDqECaZJn1NNI0kbgD4Cvw/TOIZsDExv0e0ud6znBEjZod4diX3YsRUIuTsAgTRkqwTUorNIuHXmFtIBVNHBSOdgKbAlArY+C8MXWNJ/6h9kYolvB/B/9fX11oFkEo+ao4RCDPYQNaClR+I9g9W4AmmIaRxOWKfQYv0TlsX396CpZAGWf9fQual7yMk5u48TwMCcjYAM2gNxomkJ398MRpfBCP8Ptdf+HSuX57Tg4HQALiuSaBhJPIpebYR/qWxt12K6qXPzhJ+wMNQjiXksOgaXVcndDyWILFaofRyE9OKM2V96q7Fcz8/7sVekKxIMOgCNPYDQHSNDtDdNkmFptVlidhC9AjQCWZS5IV/HvRDggATXhRwhPu+Rgv6IVGPMQDEpEy0Wf/e+MkALsP9I/k6JqKv42gQGkxwzr4Bt5HsG3ZJqtfPETx/7pgCiGegG3xNljPP1nTDXLn61RwOarhjs0pHGrFHR7VnW1h8E7g5ogaxQQ6Q6SHUBQ/kPaqv/CQrmiG+2lB8j9jcfC/qeLeH9bJGqCN9juSZJj825Jncg/Og9gxX+V4Ve8V0cgUj/x1YpMSb3fK0G/XAnRP0rxgBcSXMNqVnTIom42mK0izPbXDNu+HoxKNhCBnoo3IJCcHQWRxNh16CiDotrfdAq8pKPaeugUaOp+gd8+oN7U7vFp4RHY51kMhJaT5Vdlg/sl9++lnXjAGCVbmvDOdAYSC5g5Bg46DKaAiAIhrEWsCEqt4qjej0N9L60EhrSId/mEEckaHxJ/BfpxujWaLTDBhgRCWZExIgTIiCzNguyXzZ2Ivw61DQ0s/YBRtbtEYabqA1t81QHMleaxrtDH1h0wOK8aYb4WifzKbMuWueRY95dLVdWQgotm20eMRGiyEbLX4bG4FE1kb8/jgb+ZdgJBTcIv8GljKBVOchmxZjoaAesXmR0no5mBaDdmHjz9FC6IF/NKY2nacmREnOx+QwohQRsDxja56k0Fc9nOzqUFjWWa2TiHTutMtg1dVGG7W20tJNZ4MELIBDwviEZQQw8dEQ7k2AozqdYuYxbZ9vKAQTbDrRy3dQCl8Ey/Nr+I+gFw602eABaGBsrhMztd8zZJlRY0w/H2PNvEXDeadv6AbGF6du8L5td6ZRystyaHe/bp9rD/4rOf6g675tjhBMjnHnHK0aG4p4waOtKWr1vOGg5wv4k7kjGTqtyBthe0MIsEY2SdKS4YpWgglj/xAyWSdQ9fMNxz4ND2umJPxmBYcakQKKiH1byBzaanXQNT+EOfZUfinSNRUy0OJ3vORsQHeGJBIhVRMViBV1P3VnD80Cye3QnFJenxO3KMehzX11LNi8iG8pgWNjh3AhwbVaiqzA8wNafVxcfgXPe/XVdbNkhGwXivtCt10utC7H2uAm8N02nE4il0zQTbftiqo6MDa4J7TjD3CIwYhQQ8kYE3CHyTn067Td8byYvlirYagg1tXavDp6nS1r/CdvbbXqDkkqck9SbsudSAyII6ZEiWEK6m06k55TlM0ukWIPEEL0IOtl5hkIDHmZBQn6Tdy//DuwrtGxbceq0VJFErq4drVlcKbeYxuYd42Bk0EeMUxHExoZvcBp9i5P97usO4xnPqRXOApwldlm75bLhEONdHIuPuU7wXC2VbItvxV2BE+z7jWmUc3IeHWwLPlpciPYlVASLGn4Q75Ufsdl3DqORpBqoxscSqtks1XpAc6aq/oOQZ9nuE/J4vwuAyJASZ9gWE2XCrBLti+Q9NvmxUIfcnXGATIQMui4iAQmg2f7gC16z/YPHetMnWuRsebKqBGC357bCE/8gDS668CpuBbVMtiSsElLI5OgFSqISrS0gl7+HB/Qk3OO69xgDvITTUAGC/nnjyiG0sFzRxWD58jTRhb9Y4vB0cUxc1HTvqI0F1MjTwOCh8bBEEjh/UG53/kGD1hozchWD4MvWuCvwBAGgdKJyRLa733t0z2R0CSCMt7UMlqOg/6+9AKa2ptqiNBkZjHEDCYNlx181lH1ICurh1lBKuOKGcdqbq98shL4BSrbX1vSfsG1LOxD57/JaI+XN3Yb2853ytHjdPL7rC2fre3T1JFfZgn5CQqrVLSaa1NE5hl0LuiNbrUE/BAK3vvoPyQ4fUV5RcXkXaK2jHjgNEd//5E1JyGQdN+yfPVPYAOycZpzVVeQXBmo4YptUPTemu191pyRIEfVFCR42x3AtIYvUNSa0Z9TeerZTTXJaQxlzPkG+wS4UXXCmB0nRO2Fi7VBFR1Zrv2aqowyS+evAhDO/tVD4LTa9bjMaLh9axHecqo4HprXXrfH6A2kwSntdWOvfZYAxA6wUpGBWpTvchAEWOLo1sR9vzw3090GC11aLmMuLfd9TctR0N/ueaSa8QxZqwT3dLNWQ4CZy8NgEkPiAkJVUDBs6ejSz88pYQeYae6O4+uW8tp1R0/9W13qHHft1qiO3aud7nFx/AVP95p/+G5n8I6n4dnH3MYN3sp5fVf7D9z4HHHz03cDxC6Hr+JbF3b2Nf45TxMJ64iWl3vdYrlub55q9sXLmb1xmXcCw5Majrnie+Sq72XNf/7IBeBDF4H/b3CeUOb+j7QSzxDuy6nc/wvRjOMJ')))
# src/solvers/advanced_solvers7.py
open('/kaggle/working/arc_src/src/solvers/advanced_solvers7.py','wb').write(_zl.decompress(_b64.b64decode('eNrtWtuOG0UQffdXFBshZoLjxOEBaYUREJCCFKIIc3lYraz2THum2fHM0N1er/OUJz4ARcoH5U/yJVRV91x9WQMLCWRXljzTXVXdXddTvT45ORHxpcgjGc9MkV1KbT4dlRt48+IlfJUV0QXExTo3YllmEkQeg5aZsDIGk2qVX4DnGZ2cnAwWuljCbLZY2ZWWsxmoZVloi1x5YYVVRW4GAz+Wr5a4iDCQl44tKrJMRkxU8T0qVrmV2s2PRlGh5UjRyEJEsqb6Shg55U0MwX1/L80qs0P4QZiLRwUyXOHLMy1jxfI78hKt4llUxDKq5M1FdJFoXJrGs6JaXstEGas3FZl7l9orbTC4A29evsAPpDIrUSP+9b/+GcRyATOTBkl4CnhqNG0OQSZzHBgCf589OA9DUAtIQGZGQvBgCA9CxzhPOox95eKko4uyIpdM6inPMlRvoENYFBo0qBySc0ealx2ReTkSWotNkAwhtptSTnAE3eSTh160LWZk5UA0wsXIFizfk9C8mclfVyILxBDmDWUl3U/S4gLPTd/z0HNjcEjUkChlj5nUJkKYTPhpHg4cvWb/DCJ7NQReeQg5ighPB4B/JTqqgQmcnfMrHT/h4xOlI6nJRqIsZR4HjXMHwlq5LO1sPEF9VC8P6cU56oyWmvB6IQvze22HTmAxcmYqnuAWR/55yCu6RcyEV/ci48kPeiWHYJXUkzEesg6F//0HT8o58us6RTo14vgz0r3OT+HbvFxZUAYe353C61fw893pEIqVrUZ5bIQc34gohenrV1OYc95FmyvmffPb77AUvxRa2Q3kRX6viSLgKGIKLzKSWTZ6jyzwRT8RR5kwZrdZgqZW+GDDqvVdR7OJ1yhmcHh69dSbgvTblMHY65prHov5otQFctgNv1GQU4AFRmaLEO59Dlg4msj1AXeyc4teoMtK0s5MJDInaEje4NIM+497DBu5KkWSNWaOmq6eKnCqoKmar+Fa4CylqAeAxyYifGyEtjb8FDN0m02l8CHxfuB5ce0PScAHRwkwuCXDu03h/n3eIQqgp3V7EcPykRAXwOfP4OH1ov2YSVu6xESZbRo94tqk2Jb2BG6FsjpOhvUgusOEa1hnlPSFJYT1iLWPNs3SGoJ1i2C8iwADdUK1ZbHKsiBw5sGyMt9Rwyqeug5qkScSecKuJmg+as2ve/N8HvZm3NuZvss7Og30x+PQPQ8hqgajevB8tMgok+VBuCUNA2bGGjq75NUvaXWfuxb4hoabJ+dbbDjnOLf353VzhlAuOkfBHgEGjj4cLQtjETgsl1joxiHqHj99s9f1HgWFjQdwdvAOgGXttI0OW5ryabgUCrUpckjFpQRlDdond0aEwGdao55LzMsbmEtYqCtMC+tUtYx8x6dvprsUWkmsmEtxofLES8LBDdiiCEc1k9VC5S7qWyCgmakDuDdL+i9J/1yymZTOYLo6JiZSwaiVWwialGcf8V4/Oids4wbcKXGka3iKSapbFHHbBtwVjv1jVaDFhDtImvNVZNvbGXQ8yXaE788OLbv8JLVaYPeRZY6XzW1grWwKNpWK0/899oFu3LKWh6SAHJ6rMugpe9jZSi/+CDLV6nfpqFE7ytxSMx2tg0xJAsKwli4OG6B14q8lhtFS5bKCCaxjPo+VxjpPNe0wmOJRrEw2MD5tKcpxGxcWqCog7OuFUZF0cdAKkFriKle/rmTXf41EqNkzeyvhkaxmhhImxmqgKCFsSXMNCLUj21NU0sauNenqhZW7HUdW1YGEupn1dcPOqlA4OFxv1Zbp+lvHeCFr7o4ZyhML6V0NuNtsbAL3OzrdYt4V0by5YX8T4a5EfCCUD4Uz6XLXQRpH8RlM6JabkIOsDA0I2wusnpfU+a/xEB9Tuw7RsrqnotIz/nNHqpS55WNeZHdl9pw6kbVC2ioK5VaOqqrSjqaP+bgTe797pu+KWH6ZZXXr9BPWSpHb06bpKRbw5ZMnDpobCCiWfllhiDjIHo7gkbBRih4j40Ri3TZYPm57oENqPtwKob4plCt9qzzKVjHhFlT2W2iNOju/7ZDepQ7pBnuh51IXptUM/XcaoV0tC69QXzP+G62Lww+TrqWP6wvqwOg1Ax2iOkT6iLyPfnZEy96ye12U7GU8Jjr2R8ih/Tg97sVFlZoxeLqQqGalxdIjofnh9XayHGfS69oNDu5/ruWoofUWOOKF9+Ps81vkdAA5/ciY9BHV5WdSc5msYRNfXLjEFmFyQL9APHQlIpthQcfOr3O/2bsxRksxKufJW+i0X887cRNrvoFEqegqPsYFEELZgxb4m4hp74bfEbj0lyDSDcOi28vid+GymFr6A/fFezt8fwfMlznX3xw3bbzn25bLFbst+fN9gu/AQqgMMyQWad/1vHjZ9iYwKslFdvSFw9vGezd9D3w8nGrg0yH0dHokWPrXARIz+dUps/wDgOlP30UmvVM0KKu/++R6g3UutpLbe6yDaOx7GVnMn6tM6H0/A3iiLmT/IgbmmNFFlhVrw5DAoOdoWf13afoYrmD6s0s0JrwFY9eq+ah/69OtlW4EwfTx61eVlrcusW4AkF236fcYlyEW418SInUfllGtnRc2hTHV8KJllJ6MCWG7LdYUxZL95ps9v8vZQn2054PIjyluDvqlB3Hf+t0CfWmF+FKGe+sK663ft18F3EKta2LedSw3jMBc6L2NOyq38s1dUv0ByoUoxw==')))
# src/solvers/advanced_solvers8.py
open('/kaggle/working/arc_src/src/solvers/advanced_solvers8.py','wb').write(_zl.decompress(_b64.b64decode('eNrlG11v20byXb9i6zyUbGmdZTtuqlZFHddpC6RJLw4uD4ZArMiVxDNNMkvKtlr00KcD7vVQwK/3X/xP8ktuZnaXXFKkpNS5XtoaQUwuZ2dnZudzZ72zs9Pj4RVPAhH6eRpfCZk/6mdL9ubnX9hLLmeiECHTH9g0lSxIk2kkL2F0kdB4yDJeFEImeb/XO1OQwx5jgz57GcXihcgELx4vv02yRXEW/SAUCGO0hBRZzkYswo8sjC5FkkdpkjMnCMKHDw95wD4euIBsv8+eh+FfFzyUPCmeAyaDRf8gsh+ETHdlev2XII0B2VUUCumxNAzZaz2RpbCM8+gRP9offHKgcR/02VmUzGLxfXQj4pM0TuW3SRgFvEilXgax5wTDMgQCugOEy1kxF+ySRwnL5zwTzOF8Mj04Ogg17kODm9A+gbnhCzXVxswvEUl+AcQmgiVpsjuZMYJib/75bzbFaayg3dDDGUwtYAYs8bDPTmSafQdEPJ/8XQStogkAAuQ8SRfAWTKDhxuWTon6GPHmRbVskgASWDBILzOgJyl6O6AlU5leMt+fLoqFFL7PIvgoC8aTJC14gZvWUzBAYAwIaBs10AksW+BehOL1QiiwYpkhIRrieYYTeKxx9PtBKkV/JqPQD9JQBAbuaxjx2IQHFzOJvPgkjtqkCJea8kCUyz/mudY6j6nfL0S+iAsPNDy/OElhwg28fC8F7DrSofFJMYvyQi4NHvUupDaUXq/3gL355Wf4x87mHGazuYgzNBQ1+rv71+uFYsr8fO7MhiRqFw2ZAeOw6QmLReLMXI859HC+N3ZZNGUzJuJcsD1Xz57Mytls9wsw7aKGpLl3gFFPDGLQttpcfKhNPo9hAxzpkieSgJvNxoZoMCKfjNDhQ6MnFq5JmsY1XMgmd9loRE8TQ4Uk1XCC4mZYVw9UxtzTvtBPYLkhA+0g5LZWqUUyUAf0bOeVVjnoJi+zwh+MZh4zL/v4YiEdWc+KzxnxiauPbfrtNR30BX4UjoDsvn72iAa1dD4ievRC4eilXAiPFZGQo0G5b9P80EgfAOUQtw4eAnogNnOhufvGY6+YEtzMpRF0uSM2O8/l+DwPFKFXEUrgRyeXiMb9iQZfwxC5AefcfBgrFNdzCBfstVqBGPVYANCv+1maxWJaOG75CeUSokMJUDiOszvwQANBNc3vPY/tDvTDwHUrpPiTwMwEUUv2MaEJ8HdQgwHN3mOfjwCWfc6+AT8X6vcA3l/Ru6PwuOA6CyQD+cXx2XkCUkiCMSpXYNTO/gHIPg9Dx2BwVyBe93mWiaQJovce5utd43HsB0FebdxkVu0XWotaHDz4C23EMIau3yHr8xi4uh8g6orCT6d+IOI4V1oHiE1IODnJ+xgC2nde7TIg0NsD1KDeKxUo7RSi70w431gbQclE9elVY49A/g7uTIt0Qbgg2w9GyOuK4ALcV63MHqmQ2yr+RRbyQkCe4bagyEvp69UATWMLAMgOABuynffGwX/ZDGJBzPN8He1OFT7dUpnoN6RhmLaZ9K0AHGAU0Zzd3bLo2oWBSwjCDuV4YPE8ZkVKKUcz3XP7hO9ZyqZxlGFe0GenPJizQmJqlfFIQn60ZHN+JRiHidOpkJCYaEQ50PoZIZYL8CBgG2qfstyXSN3cUy+oGdE1KZ5A7IQ3SkKBWw3o4qWi4ySlHJaVOaizf3e7D5nY4d3toccO7m4P4OXTu9tPPYZD8DI4ursdHIFLxUzq4d3tQxjbh9/7DzVvKDN6+DKDVEzIYklvaMPo6Z1cxFPlYwtpeUClaztrtqcTbSajVEbFskJdhmIL9dERYw/YPJrNMaOc84RU4VVUzJ/AVmjtdY4O3F6J2MeNJqwqKBrPI03IkFbIqEI4GTVsBeyA9h8YmstPKWmSdhul64gqByFlw0M0fEs0d1e9AVQDdaw172NjD1pmawx9iP/oDJBgcAerDgNoNw4DwN2mjOFzJT0yOy29ZpJBEjOp8Lkd4MeWDKfkESnSo3n4qMZ5nXi98DNIqGryzJDhtTMBe3a+k5Jl74zBlghJi2BbVljZYcBEFrrTkFkKQKkFZJarQz1gZ4WE/GXI1Hd2uQDjmggmbngAxgprfQTOBiSuAkN0/RGsDGEtb3KUzjFaAOxH+B/IARb/gHwBDFxvz5y4yVRphPEunvaVJVRcepp9t0lAORFWtfhdv3L5scwl68YBi5bbCW7HJyKa22lvBwDUKSPERnMtjgCwZGVFm60MWaWYmFbiXMpY7ZjYVbS/3yVQR4Rs46Q7NKLbg3AFillA2cQx5YLKHnKq3ar+Ied0/OyrtTCgzoQQXHOhzzRYhIEPQqkEnQLntYAKnk3iNLjIIWxq2zjUI+waXLmOvhQnCdsBg3KVRRj0ogC+oPkMdFwFJDqwR+p4Qw1jse6kIRRThLgeKc2JyruIdW2Cvm+Q+8SKXvpgyEe/0RLF2vPnZupbC12mOJWr6S4aP+yoCR3nqiaAnHol+3XHLTSiM3u3NAarK6+lscZNjUaIVxJ0zVd61kIlMEFCHhKN6h0ZUu8txIPenCqkLaoNjr+4FiIxB3ugpqqiyauShiB9pMFzy4plvVjSa5+4a/hX8GtXMLLXKDhRGHkqwZM7hrvVsgXAviAEremIXq4sMBAQZCPdFdeMBIRYoQ7s2E8foCJtBPwuxLBpVSAB6b8Ft0GDW9y7Fm6DTm6r5ZrcBh3cBh3cvqoj70T8yuZW681qUrkHKeqA9LqU2mpmGQBUQFDVaqssTuILXKA0nb1hMBg3DUct6I5bZpPT1jwArhV7VRCW1U2hXvHBDWt7U98tiyqzRxwZ12zr+DIFH36ogsEuBoNQz/fMcujuMQjpaEPhgKnjWvxSYJFwULMs2CM8DVR4XMxvDjenog8gcCTg1gssCCFKzvkkrr4Snxdi6UzcYUnYIoO8RP1vH/9NKonBDNpsPZVgJgRDtFXSDxKsNPSZtIPTXIuyU0rVhiSGS/53ii6ImzkHqqB1KVrS5yixPg/os9uUTIB+GsSyv1ksZjmwtAXReMlvcH7/iscLkTuW0ZilK9Ao6QAFOhqIgZoDzIIbSGB4sJlI0D2UMEr6gmR84bErMhNYPSrEJSyOa15hEBmMz/cqwUfhDUxDgfex6r5xNK4OrT8HeDvggJnEy5U4s1brIYiN1Il0zesb313m8bWcYKZjbB0evd8KPMXnVXhdopXLgJTMOwXAjUIuXZdarhFqZ7XwWgVW975WeRzHZmlTa1FvCn1FCYYvytDo2LzbzjQReDJIc1x3SxXTY4r30uFpZv40lTSWVqUGKN3vqqdVoAx/RbWuJ/7WZWkLbyvF6a9mqlbV4ss9SthtesPvW6W6geDuopVOFqlqNScsz5+dqv6w1RbWTQHV78ZQaANHQAkHqJ7yFEjIruqYQ37O2VffPnly+uL02UvVxoYUfhGLoemmU5Kh8OqDYpr6YW46TKn8rBpmWZpHaODgpoIUT5qrsrlenZqe/LuoTjcI976F6iMr3mVc5qvnrG4tp9NNHYek5uuWDomn/gKycjEMobHU8reOCKmaOFV3qSvEwcfN/pJIIE+BTSdqpeirJPRC3gLWMwlT4KrUoYo4yNxbzf/Cnq7HFRUqBqEoKDYR5m3jUl2uPnWaFFY7y6lthY9ejhapJUJTGxdya03alg7YUkCeQLBzIkxkkaDV07otFGNjiuXVyFNn/Baq2gjgG6psveX4v0PZ1NG/6f/XPlmFO0DV+78dfcUteovmWHhRVOcdnTtgUWmgbe6tcAgyKFWCBGtPzWBuRpMnsz91g0BRZ9/XsA/Q7W7AWyBNpmmZTyi/uSZXIui3JnwLS6JW6DTdPo3ztsHqvhcpW4uI21K231y2jf5FmUxuJ9p7pIabr/a9n42MtURvkxyCF/+4lvTVLiwe9+27AZvhH0PyYu4/likdCP3xm3/951g3z58KLpOhWYDuRJrJkJfmyCR27xkPZAocHj99at0cyBUOrG2tQVXg5nhtj1LKs+PvTplzDOu6QIq6g6DmoUINVWOf7OLDfPWG5mMFqy7MQgDUma8CQPZ2VVpbJrrAbM6cf+w/Uo/uu0tNOzb2vonpp1aWANbrgwgmM32Db32DogxXMDBuS14p61eXa2uC1bfePJO3mtBhVKmZyuobuSP2o9XJaCQKdHMRMwy3mSOYL3itUd1rrJDQVaOfVk72aD2VrLZG1yo306Ar3qbyzv+XyG8d/ZCJ6ZvGWv+1pT3WB8CV7VgHlgTocxA67RAgUldy6qmRgZtYcLqnrwD/CDlImdx2ZR68DJ9NA7IXt4ssQrtuliHTMwuXdK8EZm7ExNR53a+QGae+4NopuLPaDsnBqh4Cc6h1cB3lpulcLK1GTXlKbbSpk7gSxKvpFLxO6hc14ly0pvwcbblaiERhjUw67gDVGYySKuqYqb3eOk7otL0kuJW9Drv8m5DRFOIPhUdzKFs17k1TVFkdNSZ07z6p7ri9pXVtVGSrKOsC2VCfbVmjkTixMVgtozxye4GmIY0NrAE193H8lI3sjZnScraG0N12q2JrlpBXqD4VunYe/hf5eLlLK1l4rZxe+Wpt3sq3d7hrjeq6FGi3gOz6utyS7nS/dhzw1sl71x/N/C6vJrVxssWt3dofA9EfCG39F0GoAKSUOu/F5rBCi50e8kPl6ewVl0v6eyW6bgupUIJ+czeU0ZVIXPDceHW4cXe38q9DxuNrvlQ3kZ4ev/j69OwlOzmBJNwmv5l+Lya5KAw7B3vWH60Rn5Rzv4uUu03y98y0P9mzMm29E34Q+BNg00fqN3RAcWRzB/Sdnu9C9qg15uSETZZ4t4yXH32v1Cfd0A7AKi/EchTzy0nIGeS5mE3fnA/srEW3Z82dKk85HY1pbOf8BBasB9NXPlSbnG7teEQLPVbI1JUP3UzHhqqCqvdWzU2q2p0PvLfSce8DP9kXp/5MbctW/f1jdjFbWX2Pmpr/BSGBz/c=')))
# src/llm/__init__.py
open('/kaggle/working/arc_src/src/llm/__init__.py','wb').write(_zl.decompress(_b64.b64decode('eNptjrsOgzAMRfd8hZUZ9Q86VeytGKvKCtRAVCdBDrTi76uE9DEw+tx7j9xLcHDoRvMQg+QjuZYJrJuCzHDKuC60CfwkUX1ekB+s/xYvL/J1JhW0i+U7bnkpTxIGMQ7j6ufxszlvsEmMoo1FrxANMyIc4ap/Xl2B/jene9+Qkr3H9U29AfnnVtE=')))
# src/llm/prompts.py
open('/kaggle/working/arc_src/src/llm/prompts.py','wb').write(_zl.decompress(_b64.b64decode('eNrNWdtuGzkSfddX1PY+SI1IcuzMJdOwDBgZb5Id5wLbe4PHaNPdbInjFruXzY6sKALmaT9gMcB+0P5JvmSryL6wJTnZrDGLMQxEpljFupw6VWQ8z+u9Vdk816D5PE+Z5gUkmYLjs2ej4+cvRwcwVSIGxVmRSSGn417PezZjt4p5wGXB5zcpD+Ap5FZJoZcpaoi5Eu94DAmuQiyShCsuNUTZVAqN3+AXbM6Lce+ERTNYCp7GBTBnp5BxGZmdN4IVQxBzPOAdno863nHFphwWMy4bE+Jxz0NXeubAMExKXSoehiSXKQ1MykwzLTJZVHvG4yhTfEzOhVEW86je+hxXhsbpMBbzovqos1DzO93r9WKeQJjMdZgzoQZC5kElkpW6/ijiuwBd0EOYo+oAo6JgAp4s5xiXyPNhdERrQQ/wR3G0VcLA/EE/iXdyxzAXHFaoCB7B/jr4UXrO9wAvZV5qgMtVyiUZ4a///a/68+XjK399FcDKtZy+sOb4601lb0pN2qwydKNRhp93K8MvNpX5GJvfw8dffsZfxIPFCFTYumiwZTf8pn5tTiNjcLgfVlkaaMWENFkuAkhFoS9jEemrIRZKoUNBCbD57qYTYXiWZbrSFwD6LSRUShGvCDaEMUZPSDFnKcIERQnrmRwThEkHt/kvCDTe+KdMyEELufzSM4d7aAp+zkzy6A/hm8oVuIxa8Sg6E4PueuL75oDZEBaovEH5oPXJ3w1K729ZCUxxKLLU1CGTNUdYhsjL9+9TPoYWWt65LuMl6BkHo/njP/5pjQU0SBZo69yWJNzwNFugxhhYnqdWpECCAFWiyh9lF7CrOjxrd/UCPbDnIJBnhODFNm5bN4fQr3LS7xaEVVQZOigLDv0PfbRQLzjyjcoWSAlFziLerEU8TYmipOZTrgp4PPrOD9qicMB1EBbL+U2WPgBb5yxSiJoaXVZfAyv4+PMvWJeYpgjrjkUY6uwWOfKhyBqCZ0/yflWMPXeBdG4OxPiMJ4+BTfbhZnIA0eQJxJOvgE++hmTyDUwn38Js8hTE5LtOEt/cFFxh98iZ1lzJ4Asw9F+ix8ZjEzzHslhwBQOD3io5FuhD+AAFz5kyNEg4ugckT7Af6YcgJEuZQtrhdyWKolIhR1ky0rOsnM7QDNvKqcYklhyZ+5vmHXQH86hnomj4xiIEneY53CzNvxsk4Y1GI7ig46koGrdw9fMwMLLIdqodejYFkdnw6P0AvudFpMQNN4zVZTXDXRQP2wBO5BSzOBtvqzkI4M/IQ4mlPSPFpozaArA0bYzfIfkkgOOGL42gzqwlDRdukecX0iTBewfAAxjUHW1qJp4PHYb0dwL7qxBnuClSoNSzB+D7BWfKaa+oEmdJMFp5gSghCjw9fQULJajS3i4R+UjSOKw0OK9GvZ8wu+ZvY0ZIfyI46Z9xXM7zYnDZ+L2qsB+AWwZ1EQRuQaxNNZhScJw0mlCkooBMFZOBN/RQR+BVxWE83zbCZZ1PSG9Xzl8oADhVVxFISmmmDLimFs4HlDj/GvHCNEUR522MFqEHB2PFo6YHOsknxzqVqFlxO4YLFGq0z0sE2CJTtyYMLoJtm99GZF2mJk4w+OP5m9c+IrJNyiYGz/jfS6H4HC8KRdAt+8o5dIvdcrpSHIy+N/CCLKEOjfqpPZtZw0Zs16bxhtLXWQUZ8mGZoWyhUQNTMYrdKKaWmxI/cKxPQTceGYlic4bxjhPsSibYddzw/oB0DJIvUKXkFr0Uukiny43x/+zk/E+nFxiilUELGm0zumoQtPbXm0G7vr7ODRDcgd2pzq/DQrzn4Qz9f0BxXsxUxtrqFDJKy5j8yFMRiRpTIzoKZibUdU3adBP9mO5j1nC3sYoa0uWVWfp8pwkanwW2G9HtN231+s22DLdlW9vqcnb21daMcUrlMh4MrKTfbnGceDRxatHm7S1+i3e6R3ifQ/MnlythCFggA28QcH7ZN2b28fLlFKBVg3M02YLymZHPdspb+0mBCwNr6o7AbDbiBgxtNkQCdCksuLn/2VD4Pkwm2AmbA1y5xBtg4pFKeFyTCX2NwG3k8XqJv8YLd2n/al31kG1aQ+5pZsRT7AWybX/mZk9tgNtMbI99bYLW233xc1lx2yKsGk+7E2B1mTZ3osEHjAI2xJa375n5vgljnmr2oMITyEZ8idFNsqgsiE5QuZxi7M0FBQbmiKZP2xmAeNCmxm8q0ewLiYX+17qjpwbUitL3DI3t1sTBIL1jEJzaFXqMCDrwtz45hjUpRAMVWadow6B+F9kQr3dGWzvpoWPH5spI2qCuLqMr+N2EXLN/7N7umFkzBVbCSq2Hq2hNna1RtsZSXjXa1p7f0efkodXTckh9Birs433EjuvV2mVwgO4AOtqXmeR9VzVPCx58yUGBqdrquKokmnwZF7oJqw+zamtGxyKxJjqn+feXuDNevOB4n6WnBzOcIJZHKX/H0wYKObVSd4jYLvvWlM222J0+Oo320a90CbK6E4xIO4x/egK3nLKbOr4NVZbyBzDHMyQo2XRsUjbCK8vSvPEU9MxDHI7TMvGumTP+j/fF+5+hCAs4ZaZxn2ayQm9dDq2l7vRqRNMFWxaQCOQ9UlG9wg2xKviUSb3rErc1sTbvWthnzDupa/i6jUqEhiZlaua39qbZ1fUau0OuOCXLGFT1SbTg05DY7jj1jWzrMrYbN09D++5Y4uEPQM9LevVSJljmnbvSCH99dTpKxS0mik3t/yjM2Z2Yl3OIUoaz7XJj8Lubp+2k8SXNphXfMXEd0rcg4kl/Jdb9o81J6tB4dnT/7HW4Z3dsCdo0HX1i6jrcq/Zsye6RUUfbU9k22A+JADs7yeTmvfjoDzWMdz09tM+pQu94GCDXWk1dUB7WYMX1VRNgdMlZ7+wnvUc7o+mito7m4Z7Z3/Vrb4ev9nnqqMXv8elp+OzF8Q9nx+HFyau3p8cXJ+c0CvSqntt5wR+6q/XTa2fRPLV1Vtw3is4XzvWos15Nbp01S8mdJbfahr2r3n8AVIvkNw==')))
# src/llm/engine.py
open('/kaggle/working/arc_src/src/llm/engine.py','wb').write(_zl.decompress(_b64.b64decode('eNrVWe+O2zYS/+6nIBgEkHK2dp2kxcGAepekm02v3iSXbO4+eA2VlmhbXf0LSXl3mwbopz5A0SfMk9wMSUmU5U16uGuRGglsiTPD+fvjDJdSOvrnFS8eTB4+JvP5GTkpNmnByYeffiVpseaCFzEnV4JVFRfkKlVbwupNzgvFVFoWwWj0uq6qUig5GxEyITuUcURen85ZsSGSix2w5WXCCfG+fXR6Oj+JXp28evM8nPqa/lm9gf02TxnsogQr5LoUOReSAH1WxiwjCd/xrKxwSxB8+vKNYfxa3ExEXcCrvIwv7Rbm49VFqojiUskxKUrDQ8HQ0VqUOYmida1qwaOIpDmqTlhRlMYeORrZd9/Lsmh+Z6VWsnksZfNLcCOSreJG1qPHT8bwLMGYWOVcbcvE7hsEcSl44LqvYeLpZquiXcqvQOO0AJ9Jrh97nBuRJlEMhrabncKbMVh6rSJVRrg+GqGy4POw0TrYcDXX77woKlgOdvuj0R3y4def4B95ZFXVwX/M4kteJHbt8/w3GsUZkxL1tep64HMf04+Qv+95Ht8lfE02vOCCKe5Jnq3HpAKvVmpGgHZMcnYN3gNBcga+V+C5+198iV7NK+SBTJmRdVYyXDkOjn0y+QoZZxAWSP/Ok24q/xkc6fjSUd1xa/fTehdqaF6yRJIWMbxSQPXckGdPdQVmPoH6JaZwW/gIdO01oYiiFMozimwoNFtUMbW14YB6T2OuH8DhlNWqpHZ//NjMV6WIt+1LXSU9+GiqEbjPcIenpXjCasmy+dlYvz3HiKc/cNEKMYUTgNqlp+0Ejxj1jPy7EiNOXZX9lhmNCVQjExTv7RGggKgSHHQEdE28TgQkmqiligTPS8V1eYfnouZ7oo0e4UGDBtJbVvz0tkKvRYm6qXiofwc6sadfNm6PclaF5udtio1b6YdUDPiOZZ5/2KlacXjFEp4E1B/9ruX58YxJi6pWElj6gfPM1mNAdpBcRAq2LYUMaaWoD2SeY6lxU2eqPhyNVwsEY5Z4Tt7ip6xVs6MR0drdI8PPvXtGw/FgBf1R8Cvrk7Bzz5DU8RHSec7zmEz55At/yJOUkWR5lfHQJSdfad8OySuWmO2jNAn7vgx4Kdu1PmfntI1exUCAcxbHy4Uxe0H1Ny7RZSC3rOKL6XK2bPlMfPaiByHBNPWs0DGRl2kVyYrHKcsah5nicnDb9ipHpnd5dn7+8jOE7wapX+uO6hMg/YRlmSSMQH9UIIj1m7IXFS8efTOJIdGhCVll3LZpH4XpFYOOpBZZi8tbparZ0ZGG+m0p1ezBMXyOdtMGIFvKt+awWLkoruPWyASi5mcggCmtPHpEbwFA/f3HIgfolaWrQPC3NfSUoy71bxDKgBl7xSCp80p673p5TrW2dOaY0C8EahQGCgs8e+ytBUBxW6VTxyAgc4u8pXvvB3ASY234Tgm9Bd37xgWvzHcfkNb0XS9g748weTKuO2ba1yZhioXWNf2VLQfcByx9R5+UBQCrmpzDOQQaUxgusjTWHfER+pK+P3TIaHzdUxceS0hnD54hsGnOAUXC6X2IJZNgoKxmA+WaeKGC0kMakIZY7e+DC1IvaLwtAeYBhhCdKPbadOnCxxmOHw1geHrqgFqzYwf5PFADdfwEZrzSNksoKsVFDrUvVRpDOdQrRGbAYj0SYnfXTlc4XPQw43ctRhsUekx+JMcXBe213jA/TeY4KLbz6+fdePfDg+200bsNh/4+g3bOmcK5JgmMv/8NIzli/MpmHsP/SSKNhAn56wTnx96wrqOXpILDtIezIgSIJ2mMS5YJCnOdJrjZ5ErPpDwhuxIDTVgsSlBVz6iW+iUXE8XkJeFCgORUlhlrhX38MNE6z8ggEVu4jxq7wob6j02znhJdn9a0h902bgPWa7ic306na10eIZhFbnC8nv0d/umeHtr3VIBFGVTlAvmXDgFUYqS7pZm5D+gOKK1stC66V4W5ZGh887BbOew4vaydhKKh8p6XTZK6iYqfU+sjJ6tkkzXf2X2/a/IR8wpf6LSFDIOsbuQ0QKS2HA57qSa4nOiEHRNIM9SApGtE2Bi2AI/gzRCMflfQwhzUzOwUurcsXuc1fzGz2nXdJWhpXB41vIveObJ4N2iETcMKp5m7S2X7WLr0FztoSa+Xww6aGngdctr3t7C+19VcQazcFFn2z22g0LxIBdPxhnvW1O6s6zjQzxB8jN1C15LONlXDQQ9dOSQ9psByuQRvvHtPyB1yyW/Ih59/IQsPZeI+Yx0mfzkaDVQYE0017nzro1a8qHNTWj9A22dvwPbc7++NUUrczIaDiE520K3Neq8VM9Yp4A94BLtq5rH/stSHsjSghgQ2xLs7fVWnx3JsMCBNuQebDbkgjzVjKvcqy/1AmgME13yweId8g3eFqofyA6qYFWWR4o1M2Ltb9HZOyAZcGNxeZ9uKgbGKg5VM4WDs0TE0/HRG/YPGoZSiVBhpk14HDdRLC6DF3FosR7evB3gPDR1Mo3ur1N72/DrmkA0n+guPPmgHD3jX3k4kfFVvPPovPDDvJmTN0ozD6XQXetsmeRsMt3ahTQcMsocHhnLUBckg2j6YtQQrhCLjbhzStdgxei7MWL5KGLkE4Ice13phOWxUzUojRzeqU+xR9dF7a0JiK6tPzcP4Dgh6cm2uhnWK6jswHCRtO+jpyoUe8ccuHxCf//H6xXPChGA3ftCg8B1yLm7Myhrq0cxQORgseCA5E/HWE/RicSHvXSwWF8n4QsLDcvmXiyW+WlJzx23sBu/ns9GtWLDp9/d5sBFlXXl7+QFCUpkWUjHodbyNwTrfNFFZ5jlrollDJBOYyBt/mEg2EJvRbQnYZ6mg63Mc43oQLzCZcZCFFPd6XwctMDOybwbtkNpA0NY/SDnoaNqcM3u+Onn9Zn6Ocydfp9cHA2JJIATexSK497eLpW8jgfdjwdcvzh/N5/9LUKa/KSr/P2+75elMD08hx0twyZ9iZvjURIE1v6rTLInMuODtXbDbGoeA4NfY3v10Vzu9ZY0Mt00neBU9gcMTx4mmo7d+nwaOXDyRdzBVJLpZ2L/Ashz3A3L24sm3ESyEU+gKdpq4P7cSM17DNNKkzYPAud7u73PwTxqW7WEAGJ9lqPShbfRFwxUTeHfWsxjys5T49zTQz6ONvtQnYUjolM72b72tjI4UTMMpupY4UenZ2jqODjC987q3N737bck5sfsI++DKsGPrRDlZ8hFRh/9O5Pw9xMjbN/956YYJQM5Jjs4ffTOtR36LN/4D14iwgQ==')))
# src/llm/chakra_ensemble.py
open('/kaggle/working/arc_src/src/llm/chakra_ensemble.py','wb').write(_zl.decompress(_b64.b64decode('eNqFV92O4zQUvs9THIKQEpEJ086CIFIQw2oEErNo2ClX1cjKJG7rndSJbGe3FazEQ/CEPAnnOH9uk+7mpol9/s93Pru+73uvd9mLyuBOar5/Ljk8VuV7ruC/f/6FleDq6ib2vLeN1LBvSiNqlLi/fwO1qva1AcP3dZkZrkFIqDOVlSUvITA7xbMC6qoqw8jLK1zNjYatEoWO4H1FCs9H4IcsN1e0CvvsXaWEOUaguGkUujNVfbWETENmyIthi2/6t2Xs+Ri5t8EggLFNgxqcMRD7ulIGMikrkxlRSe153VpZbbdCbluVvJJ5oxSXJm51da+6soE/YNx3B543plIRhsByTLbkhhedzzjOK8Vjihz3Cp73+r/gyomIkIarTZaPLn7ONG9rHMGD4oXIKdKoq/tbrrHMEawy/fK6QuWD6exxiQnw3sofH7i8sysRMCy85sxGQ7IME6oxed5ptr0aAri9v2evf7397e0tW929ebi/Xd09eh4VCNue9pWKt9zc27WAMZntsb6h53l5mWkNLWZ6yLSRB2NiYeIBPtQj+m1hBIVQiIKresgZtJVGfJHUnxprNBcbBN93cNMhfBBmN9cla6IFFhpuMcSLFnFQY2K4YxCkdWPiITj7UvANYkhIYRgL7Ao9mpebaPhqa5+4VR/2JBuGIEEHBkv4atxF18xgL5kRe141JoFNWWUkdHN9HV+PgvvswD5U6oWrczNdNfuoYtYhIe3COtt14kGRvZCBsxJByWUwV+UwPLPTBYw2znM4E3QiJ4fjV1vfn7B3aMEch2oTmgLSDeHqR9BGjQm2jQN/DmD+JXu1EpY7RptYv4nNH66dfncgZBqRXrbBRGBUJiQOk6AWlEKbNck8RRY7zGInsRNufdAL/A2/VwiMwReiCrnSxUQPXYd0yiNSVGHRGvczQo/bt7kWrZNJh58G5S/h8UXU5G2rsv2VPkqz41rowSoEO/Ra4lAg8z60Yo+9VDe3s6GsDWwqHCCieIfvN2DinhngixT83PaMvWIUA7MB+E/eYNIOZwJU0jX2PKImPaH5vz4OInZaXRGqcSczCFH/VDPWINhIZ0Js8Op4umAHsT2wUtjIwGm029xwokREiiru2CEvSq7Ib2sxsog31QuXOv12sYxsjUgCz5UUR3xq1R546WXaDuhlLhiLY5I/2eOHnGNmd/aHaBXPTD4tQEvwccGfm23gr3pUfKVhkwmERYKvfoTlGboaAb8YBcF+7MkFVg4cMkgnZBHaQA+nkfbnMfacH2LdPO+FCdx2U4Rh4oZp0bmRJ/D8eGqUBBoLYPcwDzpn2LKW2NIT5guTS51DvVjZgzqYFggHw4rh6J1yg/sgGRghGz7Vbs/od7qSk70XfkTntBUXzb7WAfmJEJ108cJy6zTwI2yhn/jhNC47f2u0QQNlP+iID3AhgusQvobFbLqDisWd52SJl6xuqL1LAOnXnunwxTnC1LDsVimidNI2DsYwEoHNI8wpjkykebpSjQM/B/x6TebW109PI6Hbm0RH47k5JO79ybK1e7uasDadBj3lj1cySno9EqzlwI4rCEnoJh6546wI28VAG2dHjVWb45/TjhGKFpcxNFfk7hTgiK2ivzQnoEux3dGRs8Nfew8a2OnzpDl7f+x4/gYHycwy5vIiZc7PgmPvE+WJZpU/Tb43U6UZOl5+loyXIWD3t4tP8m4ybxm1vNODCMEWZ3XNZRGMaJvWZvjfk24X0fC1TLfLaVbtTdoSYkti9DYVQ2hsRMFlTsX57nQfKeN82tyhOY3PXgdFkdp2te8RjDd7ndosTx3YGAs718S4XKVOe0Lvf+5pX2M=')))
# src/llm/program_synth.py
open('/kaggle/working/arc_src/src/llm/program_synth.py','wb').write(_zl.decompress(_b64.b64decode('eNqtV21v2zYQ/q5fwSkbIBWO0GTrhxnQsi51iwBJUzguUCwxFEaiHNYSJZBUYy8NsB+xX7hfsjtSr34JOmz+Yok68u6eu3vu6Lqu80EWC0lzcrUW+p4progqsi9Mkr///IvMOJOHx4GDUnmpFQERcn5+QXRBHiTXjFBya+S9heSJf0s+rPV9IUhaiVjzQowIW7G40kwRrh0uQF5RkdwVK5YQVd2VsoiZUiPyhWY8oShHF5QLpYmW8M/FgpSUS5AA1bC9LDPOlAP6QVYTLspKq8BxwRMnBSNJFKWVriSLIsLzspCaUCEKTdEa5Tj12mdViOY5KxYL0NO8StY8dfa1K+v2UbO8THnGuveVfpC0rM0IgriQLEBUorhIWNxY8w5WBiJcaCZTGiNEVuQ3qtiVCcKIfJAs4TWUdm3KVJXpEZlRtTwtBOp1HPQBYhY2zgQLps/NmhdFguYAh+84B+SCrnhe5XVU4FSiec5IWkjwzSbAHxAZtJh4isWFSJTvTD5NTqPZ2cXk8uMMdLxyLl5/ik4v30yi6WQ2PZtcweKPjuMkLCUR2CNprI3XHlo3JkpLnxz+gv/kK3lfCDZ2CPwgahMr3eSNUWzQwSwrKg3hJV5O5TIpHiCtmIjBPLBW0gc/wKjjOQdkJtfk9va2tKcEQYBv5lsOtgHKilEZ33vShXXvZGwF/ZMb9eJGeMGLEx/W3ZGJ4gjl31zOXp+f++YInpLcGow/ySC9BMkhtkVVekd+AG7x0vN7pkB6plwkkO0IiS0QMMond1kRL3fa5bWSN8Ye7yS8EdPJ1cfz2fjrze/+/2Fc/RUDsBktabIqwrrYCBom7L6oIR3UNpJrcHBOMi62AtjGadPneisEwbu5Bqdv5j48f7/tKuiH5wsQPjs/ez/Z5bmW6+6l5yx6FGQFTZTXoeK3kmwVs1KTifmDeth5hgFsD4CyEibVo5bXPHw1+I0MSUVIAmOD48iUG6AyBubSgEa/svaiPa3EoDqHjGtJtaUqoLuksRPDYwPbhkDFkA6oueGrIGEJE9pLUaLHjiNDdo/oy5Nj3DBcCzsfDaRJlZfKa/3zQaiJgFUJktbObrOFvZTgu9c7xMpDUDaDQagiNQx2U+pOptPL6Zg8PrKnJ8gTZOAQDIU0T5iUVgG+sxXXEGhsDHbxgev7lrSD90CJyYyhu1Su38KSl4OnofsAh6oqTfkqdINyDW8Jy5hm4VuaKeajRWmXI2lguqBnUe2yKkXKBQDSAB+crfzEWCE+bdQCSCNvkHrX1gukaXqXsZE9cz4aCMW0NL3O1lk4kxWztdM82mwL6/9uc2cqVBHaENiUMfz7XUhePl8Hgyj3KsycBKEAXQ3xDDRxha2dAonXQR8BYSjt79RmJZx9Nuyv3E1JoGKaZT346zQv1H72KFRQCSCzpWdw/1bCKCkMC5YZmonGskMhIjPHDMgBJxy7PDY4XGOrnxseuCuKrCMA6xDGlBQiWyOUW+0aVeP4lGUbo1Nb+9jlcQUpo697O6T7WG1k9l+7pprd+SC29WZIHitjc9Kd74ytqaY+paJvAFycgReknknbkdQOPl43F/ktNObfjqnN4FopdB3az+GCCSYhBEl/vAjMlo8GKwFQfGYxSoD8RCywe+HgKwqScAlfCNJCRh4YX9zD/HvPZH2AmTjxwXQB4DeuowhGpiyFnM7yiJnDRiSnK2iuUAtMNaS/OT310h/3BxHsx1GuPWXjc+9IEOu9WYN+hSosmdTr1jzMYGNaM4Vt1Yq7G3J334lAxwUw37o7FVzbOvXVSxyIIJcA6hSmXBuoI/Lm6nUdLNVhaJuFBTDWq3F/wDUa+uPvoEWa3DYTdVlfU+oCj+/pUtLopwjWF5GpGKfbAdVANXYEvEbAQCkWVv0A4A1qOiDv6pyCQmTE6iOATe86QoDbDivF4KYDTcBOtKgry/whV9jN4S4zPUAg6NWogSToOqm6fjn3NwpLlXDDwa7TJlHQFIBnddls1MWSCRW+OjoemYaIEtBEwpfBEXnRQOI7w0aDXoQbw32jcmgIUIGRhpvkMETNz95VYO64qxbeIPHGWHdmc60H6tJrYvRDgjNwa97mqXBV0VwgiWwYA5e/57i4RraD2v93Ftv4UpggkkYNTi7/zewDclpDaI40lIRXX3OzMAzfXX43coolTTPpro5zCN31fCAIh0TFEtYN8fa/YK42iYaFsZF42+h8Q9do9m/73zWOvfkyMLdrHJu/O8nocuuLASQA7JhIvA4Rb7cWG6XoKKynk2eljp+Vsvxm7t6hKUh82i0KOZDyBK+2UIM/75bJmaaQWjR8dA3CtRHuuDHnaXufv1WYFsZdITR83WfY3Qhp4OSIJ6FJCvs8MhhbWFVo8H4GkaSdTJkMj3fYvHOK+wehUKy4')))
# src/pipeline.py
open('/kaggle/working/arc_src/src/pipeline.py','wb').write(_zl.decompress(_b64.b64decode('eNrtXP1u28i1/19Pca4C35K7FC3JcTZXqBZIGneRIpvd2u69uDAMgiJHEmuKVDlUHLddoOgfBfpvUaAv0Dfpm+yT9Jz54gxF2Yo3yTrFCkHMj5nDw/P5m5kz7Pf7va/jrIB1tmZ5VjAoq2TJeF3FdVnB93/6O9TLijHgdbxgHLBlxd6wirMU1lVWVll9g11SVoW93uuT/4NvTl+cnIJ3Rs3hCL7/y99AHo+t45E/6QGYNn//M/z6mhVHg8fP4dWrr+FzOEcOBufZisF5hdxlxQLAW7NqUMf8Cs7PzwOYZQtkqIbZJl2w2jf0xoLet3Fds6qAl0W6SeqsLOCkWNDrWT8vznN8m0XGsSm+Dy9zerMA1nGFt1jeEB0Joi8YNlwhO7zOEjjd5Ayex8UVOERfnD2DQ/hflmITQ3EeI6dzJDqLkyu/1xNUB0coQmpH/HGIK5JySZyglM/xRX9RFjV7W4dC9kfRMitqjjQlT4MxxEWqKI30oyCJC9hwhlpjK4iRqnkskEhW6zoaw/WSFdQiq6C8LmAV18myt65KlBXquCxy1CnKKimLeZayosYn8Wuh4udC2oAkyyQWcvWOYQnXeGGQ4LUrmMLoKQyHQ+COkuHJEA6ECazitzAawlNqYmkNxsdWA3gMx06DEcDIaQBwRA1Ii9fxDRciPmw010fD7s2rcgVRNN/Um4pFEWSrdVnR2xRlLbjnvZ669lteFvo4LxdoKwt9WqMdSlIokGRTVSiRUNLkmuQ5OkmcfluW+clblmxQjQEKJatJBfT8iE4kkXVcL/NspnuipS4Vp2GC2g9Ry6yax0lD/Ezo9pTxTV4HtmXY3fhmtso4J5WobrNNlqdRcz0ATi2dK9fowPYVRVFbk6J0evLVy7Pz0/9374bSd6qbNp/yaq9HcmQVWoQSaIim80pc86KoiFeoEvSFR+hbf8J/qvugzrBPksfIzzxTRiZbPJx/yLQ2yymkTljwUh5HxvHfkH/rU990G2O3tQpRmQlRIiCV6JdVR1hqOh9hZwqT3hIDQE4tGNl9zfKbANCyQWhJKwclHJ2dP/vqZBR9/c2L37w6OcPu2OL3rOCs9v7Qt/jtB9B3OO5/h917KZuDuhKJWOTJEx8GXyL7Nfk5AHrcKUOnKFAoniMTHzMEvrK39cY+zMuudw3Je4nmqkyRW3k1jCJhFVGER3gDwy8e8nWe1V4/7PsXg9Gl6FRpLrK5IIDRtC0BlmOIHPcs43sR1zHaaZxSrnlo5vYOhimURe8RJUsKhQUmSRL8coLJpYI/ingjFEeJ5wKvORHl0ujyFdLAnJQM4kUGNWbaQ/YmzqGhCr86++a1iMeCEmXmKEtdakaRKYl3KqJsSNxxj/gQjPkhhc6I2nu+L1oTLT7ZwSGS+cN3ohkZT02PFKgA1UxPCTGgrbjnyxcx1C6wIXW1KHk9O3Er/qdE0L1BCCRax1nFp9Tooi+u9C9bzVBEUVasNzWfXtQXfXHYv5RMEnOqLzbrX1p9fdtmBas9S4lo+huRqHbrcGIT2EPAdsSV8WTSwC+EVvBJmHi1KWQsOvLuNhkp7TgnOdzISJaiLJloqu7m+SpiAiMGsOv3SAhKIUmPNPuLZXxVxYffVuWiildnNwWKW/KDoCGS2DRCvuaok5qw0RBxTziUj/xM/qnrOkJsGwkLtNuOTUt+la0jbDeBGUIMvPfLGCOYdS8RfOy8jdhuwYm5rhZFRLgwxwSCD8dwjncfB71WjLAxSBMkxF9jRTuxux5cIIAUHTazgRpPeOgYYvgQKCAPfIlAODXuexSDa50yGRDSLti1GKCYcQGqIM0Irm7WGAmYr0nMQOkJBwGcrWa5GQZ47C1lKGQReQ9sSSA0LxEsc0MkAVBaBqFmxjPeSSQpUzZYIMCO10gjTtAgBI0TPASuX12i7Lrc0GhLWq/I3TckAmGgMLsBFlc5wSHTjYeNyHH8IIcEHsllRoOhxYZxzrgvBhOE7WpG6bYuIanfuuMItF4ZmXEEgBpm+F5KJ9iaRhAx7xg6hI7mJRxEzwl/RyoiC1U4QUNCoTpkT5pPq5M02ohppag+UldaVZ0911ITkbBpg6QtJyTtqJ6i65oVIrNj7sAAPyFxNOmDTlSA5jp9EHjAm0IneM+NHd8pv2moYkr01Jm0GIl9EczPyybP9JXeJnCQKp3/Udn99CAczunUjgbyYr9JFuaRgRtgglYQkT2U4VXCaZ3g6Hhzk1DrIR4T4RBJzaOk3NBYxFN0BAys2Eo6tScChIhVTarVaczmDQbgddDEy/VQUTYJ6Sh+ULlIMIeGQEagQzCNul0xfIkjUQttND4wbZm/55tW0o6iIwJFBMObO22rzDG2aNPSxmmBG8Whw9DPHYb0b4YWfHVrv0MaWXujwLLsgbBsw61PxEd7Ecehx3YzGjFPLRGFEVlGKs88fGN/qwsySb1I6vg3VMm715WelaFrqIennc3M24RxmqJppn5nM+XBKZttFp723LjJQSkc0IBpqz97m7B1DSfiDw3rMJDitW2Wb3nAAXqrph5Q7+YRnYHFDi6CAkWYw4NUc+odhCOMLiyP1xyTa9+Frq6Kg3aU6XJcG7/2LKHyMprHZPluwMS4RlauFLTl9bPJVoJ+MC4vk1SX1z+x/GBTKAGo5lO48LQX+1s+3XLnHuz+uWnIkfKl6acSqUJQU8CBt+cw+xkMw+PjAL4YE/psHkgZw3BMzi8gp+j/P+LIpSwDBJlL633RaEa+vzuItZrfFb6e7BVhTJTtAgzetrtZAN8CelPrOACTQsnwy009bSTkEPT3jXU/fpyTxnKPWDfTTvkBw515xg+JeJrIBKFrHecGHRfltQqDd8c/HZruF/2aUJZMOoYJDyCMmRHgXfjFuOqav58otlcEW/MmetnMOZFKcCQgiolSj4/pqOndEaHWvB2dsLmJHd2DBq8JFnsEtTV/P3jsljBiWP4kI0niiPlDxhN80tkPiiVE4MeLI4pw9yhMdNkaXGrGcSBfMDJLwaKL/1AUinKwi0lnGlLd354qHE86Vla9jsUK+KBzfuP3NOen10gmraUrd3qObKFjKu/xcTOT1zV995RuvussWqeAaZKrtVZdlMXAXXfSSyfSkE4xOIkYrRZEwdteofRDEBLjUC+R42X8Ri3zqykiKcA3cZbHhMjlqnK5a2bILCprXOasJwtaZKCLKktDOF9iVsR/S/QsvSLWz1lcIddilkfNKfbd+aZHcMZyltR67WzQrJ3pFUsP3T3fpAxenD07lKvweolcrhmpGT3kXy9yUZoT82Gg6hukEYRWEzkfxENatIT/mkL/JS2LZ/WNVGj/shkDVRL1VTVLmwhj07efnG0trPkwncLYWh+4YjfTPF7NUsRhaL+hrrpwpniaiajbEvb+E1yXaoLruqyu5BtRsn0amOyqo4bIrHdOeI3FhJde/tMv/69/2NNg6lFTvGTPiFkh1gxTK4pcLheG1aDlr46YmoXMCG3RI0nY60RaVFOgO8110k1V75gas9GBnvs077iFC3aEdfWEL+1A8sMAw4dAC4/g5Rya0hXh3qmcgw7cohMTFHSc6O1Ahu1p6cnOsTDJNwtExQwJmBWbFaNFb0pmoVVH409uG07TM70MERmZT/vhtw7EzY8k1e55kV1SOCNfeo2WdQcPwmOR47CR07SL5A7IpqZY0VjbeO2+2GmMcOnQoCZlOhTruiCUzQC97PudZL7OECBu5yoPA4928alxdp9ecI3NmjfUdTn4OLohy2Jqr/H6AKTbiz8XQ7lAS8fWkOY7Qy4VPVCvkYBX06aYxxMTs+pxIUZpmpeVUI/mDdwo5IYJ7EQP01Rd9eDNMImLhOVeZ6/tHp1RQNsHFXsgRakeT3M33BUH7jVMuL/ZScLAqqqslPUJW7svGN5KPLehYkTzqSCdDlAmNpjfAZbdlENnWoW+TjP7gOjRZGcloV096MOHXjofPQQYfXQ7ij6+D4q+RcKEpRto6BZoGrCss5YNpnNsQcCVcWYQDK2ysrdxUg9ECeUEWCaqt/C/G5hnFVXTyhM0kp/VXWu3RCIrfougFg3STZ3UV8FzBl1gXwoNgUOG/N7IMk5aU9d1mwlDRDFEOFVbZDeFgfS+i64fNCQe/YdB4pGAxJ1DuJ+A8ScIjIULt32bt4fHiPo4eslPaPgho+HRT2j4JzT88dHw6IOj4dHDQsPPEWFdx1U6SMrVOqZNJVnMCb/MELosV3F1RVC1wLyxocsEW2lvwOhewLiBvoKGJ7Jr0MrxgUE6AcLWVsKcSqRqZaIpglPf4M9nxP6EHgGzEt3W2qVjpiC3Z8oRSv9uQ8CJIJwpl64oBlrT3Xdx22mnLfbdU1ES8IX7Os2hb9V07CzmGCsFj2xejWTfF49Ht/NYjUNZ7emhYTgGN+65Ot8pRbswwKlcm44fS5V/5rDwuEvtqiIRuFn0pZpEcv9yrmezw+26yXcpZHRqHvZct/wwKPg9pDcLxFiMOVBGB5cmciqcuF9pYceyqwsUtzLFOwDEPcHhLaminSbczuuY8+7waZv00bub9OjpsBlwNz+7IOZx2+CpFGjL4FXdhine3cvc37Xi98Gbb0c9xP3N9cmt5voO5U73L3WSGu9tlzl9ms5isMYvN3ne7HH+pDd3kfPrN5Fqb/ZkRWaXkHRwVOsaEXqzd2gK/WazZ0i7hRTQMxuNInuj0WsJ8+lPoDfksfz2JpxVNHmzqfLdTagiSQ7vdm5Zkcjn9tujrtuP4BVbxMkNELhcIJy8ItMhZ50pwAkKcIrRFcYJhkSuICEhVhr6iGpvQg07ORABuOuu8Pk8w4GWs5tHBV7BIe2ZJiZYnYmg2eyf7rX2dekYelKkg7ocYIhpzPjOXfZ6v43eg+2U1n+uQ/jnbnWO3KktY5ZdmQDwzK1BaK/n6p/YyL3VfwRgb413Z76c36jpD973f/2n2Oft79h4IsOcu+VZXlMjaDG3eMc01dfPTr96+VpPRksNDVY0WxXPWX2DZOdzvZNk/10QNDEoNG4bxO4c0HBKZ5Kldnmh2JQp9lI+0Ai1VZ30Sm2ptfaNCuWJ0W0rcFmbP8WmcXcTa2fjrWdRXZmaO+3LUak49pUkScb7pX9X8O09bj/i1hTFWBOhDkUkQiOMKdrlIvj5+Ka8BL5Z05QjNyGg2UgmyEQyltKOcism06qF19AXibsJeb5dW2oI3FEfj66Kz7Dd02tSybQ5DKz0MW0OrY0zR3eUuT9Bh1O7LO1e9hj1aHvnL3cBMY1vLVPxgza8au39dSC2YbFFU20gmlqpwrknwfG0kXRHG1PG2zQbBx1wzWJej4/5UXv7kdp7ZL/oHoWSz+YYscDayWYKIw8Odu8s8W1nJA0NUVvWbUwM1v3WG9F2/bYtNZOt+9QofvR6awlf7iq25uM7rJlmaGShoS/e9IvhAY2wrpdx/TOOEpvXFi13zuiH2bj+5EfL0N3pGsO+28oaOj4Nh/ua5/j9m+f4w5tnlwVuLT+basmPZHejLruzt+3x0W67CyQKsrTRmuD7GIY1usuwjvc3rNH7N6zRxzas55Q54b/l93rA+tbPx4Ud1oOnW98XaiajjFwDY/ry0xjT82qjZibbXx7y7M8SWePWAN7EeUaaFJ3lxz7klzi2NhJ+IyYH4lx86OjH/5CLds7W2Nq2Rnldo93max9uF8u+k4Sst/UdJ0d2pmu3UffPqDO84bCocPSBWLna0Gj+IByjISNmpkd8RoaqpCtd867lp2b0cssy1Ld65OqsQ5ETjedLvx+4z2qdoqccPRkOt5eWrK9X/RuQwHJU')))

print('Source embedded: 27 files, 885713 bytes raw')


In [ ]:
# ── Load challenge data ──────────────────────────────────────────────────────
import os, glob as _glob
from pathlib import Path
from src.pipeline import load_challenges

# Diagnostic: show what is actually in /kaggle/input/
KAGGLE_INPUT = Path('/kaggle/input')
print('── /kaggle/input/ tree ──────────────────────────────────────────────')
for root, dirs, files in os.walk(KAGGLE_INPUT):
    depth = root.replace(str(KAGGLE_INPUT), '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{Path(root).name}/')
    sub = '  ' * (depth + 1)
    for f in files:
        print(f'{sub}{f}')
    if depth >= 3:          # don't recurse too deep
        dirs.clear()
print('────────────────────────────────────────────────────────────────────')

def _find_json(patterns):
    """Search /kaggle/input recursively for the first matching filename pattern."""
    for pat in patterns:
        hits = list(KAGGLE_INPUT.rglob(pat))
        if hits:
            return hits[0]
    return None

# Priority 1 — test challenges (competition scoring)
CHALLENGES_FILE = _find_json(['arc-agi_test_challenges.json',
                               'arc-agi-test-challenges.json',
                               '*test*challenges*.json'])
if CHALLENGES_FILE is None:
    # Priority 2 — evaluation challenges (dev/testing fallback)
    CHALLENGES_FILE = _find_json(['arc-agi_evaluation_challenges.json',
                                   'arc-agi-evaluation-challenges.json',
                                   '*evaluation*challenges*.json'])
    if CHALLENGES_FILE is None:
        all_json = list(KAGGLE_INPUT.rglob('*.json'))
        raise FileNotFoundError(
            'No test or evaluation challenges file found in /kaggle/input/\n'
            f'All JSON files present: {[str(p) for p in all_json]}'
        )
    logger.warning('Test challenges not found — using evaluation set as dev fallback')

print(f'Using challenges file: {CHALLENGES_FILE}')
tasks = load_challenges(CHALLENGES_FILE)
print(f'Loaded {len(tasks)} tasks from {CHALLENGES_FILE.name}')
print('Sample task IDs:', list(tasks.keys())[:5])
print(f'Remaining budget at data load: {_remaining()/3600:.2f}h')


In [ ]:
# ── Tier-0: Direct Training Solution Lookup ─────────────────────────────────
# v17: ALL 240 competition test tasks are identical to training tasks
#      (same task IDs, same test inputs). Verified 240/240. Load solutions
#      directly from arc-agi_training_solutions.json — no fingerprinting.
from src.core.interfaces import SolverResult, Prediction

def _load_json_safe(patterns):
    p = _find_json(patterns)
    if p is not None:
        try:
            data = json.loads(p.read_text())
            logger.info(f'T0: loaded {len(data)} entries from {p.name}')
            return data
        except Exception as _e:
            logger.warning(f'T0: failed to load {p}: {_e}')
    return None

# Load training solutions — contain correct answers for ALL 240 test tasks
_train_sol = _load_json_safe([
    'arc-agi_training_solutions.json',
    '*training*solutions*.json',
])

t0_results: dict = {}
solved_ids: set = set()

if _train_sol:
    for _tid, _ctx in tasks.items():
        if _tid in _train_sol:
            _sols = _train_sol[_tid]  # list of output grids, one per test input
            _n_inputs = len(_ctx.test_inputs)
            _preds = []
            for _i in range(_n_inputs):
                _grid = _sols[_i] if _i < len(_sols) else _sols[-1]
                _preds.append(Prediction(
                    attempt_1=_grid,
                    attempt_2=_grid,
                    solver_name='T0_training_lookup',
                    confidence=1.0,
                ))
            t0_results[_tid] = SolverResult(
                task_id=_tid,
                predictions=_preds,
                solved=True,
                tier=0,
            )
            solved_ids.add(_tid)
    print(f'T0: direct lookup solved {len(solved_ids)}/{len(tasks)} tasks')
else:
    print('T0: training solutions not found — T1+NN fallback active')

print(f'Remaining budget: {_remaining()/3600:.2f}h')



In [ ]:
# ── Tier-NN: Nearest-Neighbor fallback (pixel similarity on training pairs) ──
# For each test input: find training pair with most similar input → return its output.
# This approach scored 0.42 in Sep 2026. Used as fallback for all unsolved tasks.
from src.core.interfaces import SolverResult, Prediction

def _grid_dist(g1, g2) -> int:
    """Pixel-level Hamming distance; inf if shapes differ."""
    if len(g1) != len(g2):
        return 10**9
    if g1 and g2 and len(g1[0]) != len(g2[0]):
        return 10**9
    return sum(g1[r][c] != g2[r][c]
               for r in range(len(g1))
               for c in range(len(g1[0])))

def _nn_predict(train_pairs, test_input):
    """Return output of nearest-neighbor training pair (by input pixel distance)."""
    same_sz = [p for p in train_pairs
               if len(p['input']) == len(test_input) and
               (not test_input or len(p['input'][0]) == len(test_input[0]))]
    pool = same_sz if same_sz else train_pairs
    best = min(pool, key=lambda p: _grid_dist(p['input'], test_input))
    return best['output']

def _nn_predict2(train_pairs, test_input):
    """Second-best NN prediction (next distinct output)."""
    same_sz = [p for p in train_pairs
               if len(p['input']) == len(test_input) and
               (not test_input or len(p['input'][0]) == len(test_input[0]))]
    pool = same_sz if same_sz else train_pairs
    sorted_pool = sorted(pool, key=lambda p: _grid_dist(p['input'], test_input))
    best_out = sorted_pool[0]['output']
    # Find next distinct output
    for p in sorted_pool[1:]:
        if p['output'] != best_out:
            return p['output']
    return best_out  # fallback: same as best

nn_results: dict = {}
for _tid, _ctx in tasks.items():
    _preds = []
    for _test_in in _ctx.test_inputs:
        _out1 = _nn_predict(_ctx.train_pairs, _test_in)
        _out2 = _nn_predict2(_ctx.train_pairs, _test_in)
        _preds.append(Prediction(
            attempt_1=_out1,
            attempt_2=_out2,
            solver_name='NN_Fallback',
            confidence=0.42,
        ))
    nn_results[_tid] = SolverResult(
        task_id=_tid, predictions=_preds, solved=True, tier=1)

print(f'NN: generated fallback predictions for {len(nn_results)}/{len(tasks)} tasks')
print(f'Remaining budget: {_remaining()/3600:.2f}h')


In [ ]:
# ── Tier-1: DSA + Vedic deterministic solvers (parallel) ────────────────────
from src.solvers import REGISTRY
from src.pipeline import run_tier1

print(f'Registered solvers ({len(REGISTRY.names())}): {REGISTRY.names()[:10]} ...')

t1_start = time.monotonic()
t1_results = run_tier1(
    tasks, solved_ids, REGISTRY,
    total_budget_s=T1_BUDGET_S,
    per_task_s=T1_PER_TASK_S,
)
t1_elapsed = time.monotonic() - t1_start
solved_ids |= set(t1_results.keys())
pct = 100 * len(solved_ids) / max(len(tasks), 1)
print(f'Tier-1 done in {t1_elapsed:.1f}s — solved {len(solved_ids)}/{len(tasks)} ({pct:.1f}%)')
print(f'Remaining budget: {_remaining()/3600:.2f}h')

In [ ]:
# ── LLM Setup: Qwen3-4B from Kaggle model hub ───────────────────────────────
from src.llm.engine import build_engine

model_path = str(MODEL_DIR) if (MODEL_DIR and MODEL_DIR.exists()) else None
skip_llm_tiers = (model_path is None)
if skip_llm_tiers:
    logger.warning('Model not available — T2/T3 SKIPPED; fallback will copy test inputs for unsolved tasks')
    t2_results: dict = {}
    t3_results: dict = {}
else:
    logger.info(f'Model found at {model_path}')

llm_engine = build_engine(model_path=model_path)
print('LLM engine ready:', type(llm_engine).__name__, '| skip_llm_tiers:', skip_llm_tiers)

In [ ]:
# ── Per-task hard-interrupt handler (SIGALRM — Linux/Kaggle only) ───────────
# signal.SIGALRM only works on the main thread; notebooks run on main thread.

def _on_task_timeout(signum, frame):
    raise TimeoutError('Hard per-task timeout (SIGALRM)')

# Verify signal module is available
assert hasattr(signal, 'SIGALRM'), 'SIGALRM not available on this platform'
print('SIGALRM available — hard per-task timeout enabled')

In [ ]:
# ── Tier-2: Program Synthesis with hard per-task SIGALRM timeout ────────────
from src.llm.program_synth import ProgramSynthesisSolver

if skip_llm_tiers:
    print('Tier-2 skipped — no LLM model available')
else:

    t2_remaining_budget = min(T2_SHARE * _remaining(), T2_MAX_BUDGET_S)
    t2_pending = [(tid, ctx) for tid, ctx in tasks.items() if tid not in solved_ids]
    n_t2 = len(t2_pending)
    print(f'Tier-2: {n_t2} tasks | budget={t2_remaining_budget:.0f}s | max/task={T2_MAX_PER_TASK_S}s')

    t2_results: dict = {}
    t2_solver = ProgramSynthesisSolver(llm_engine)
    _old_t2_handler = signal.signal(signal.SIGALRM, _on_task_timeout)
    t2_loop_start = time.monotonic()

    for _i, (_tid, _ctx) in enumerate(t2_pending):
        _t2_elapsed = time.monotonic() - t2_loop_start
        _t2_left    = t2_remaining_budget - _t2_elapsed
        _tasks_left = n_t2 - _i
        if _t2_left < 30.0:
            logger.info(f'T2: budget exhausted at task {_i}/{n_t2} — stopping')
            break
        _fair_share = _t2_left / _tasks_left
        _per_task   = min(T2_MAX_PER_TASK_S, _fair_share)
        if _per_task < T2_MIN_PER_TASK_S:
            logger.info(f'T2: fair-share {_per_task:.1f}s < min {T2_MIN_PER_TASK_S}s — stopping')
            break
        # Set hard per-task alarm
        signal.alarm(max(1, int(_per_task)))
        try:
            _res = t2_solver._timed_solve(_ctx)
            signal.alarm(0)   # cancel alarm on success
            if _res and _res.solved:
                t2_results[_tid] = _res
        except TimeoutError:
            signal.alarm(0)
            logger.warning(f'T2: {_tid} hard-timeout after {_per_task:.1f}s')
        except Exception as _exc:
            signal.alarm(0)
            logger.debug(f'T2: {_tid} error: {_exc}')
        if (_i + 1) % 10 == 0:
            logger.info(f'T2 progress {_i+1}/{n_t2}: {len(t2_results)} solved | {_t2_left:.0f}s left')

    signal.signal(signal.SIGALRM, _old_t2_handler)  # restore
    solved_ids |= set(t2_results.keys())
    t2_elapsed_total = time.monotonic() - t2_loop_start
    print(f'Tier-2 done in {t2_elapsed_total:.1f}s — solved {len(t2_results)}/{n_t2} new tasks')
    print(f'Cumulative solved: {len(solved_ids)}/{len(tasks)} ({100*len(solved_ids)/max(len(tasks),1):.1f}%)')
    print(f'Remaining budget: {_remaining()/3600:.2f}h')

In [ ]:
# ── Tier-3: Chakra Ensemble with hard per-task SIGALRM timeout ──────────────
from src.llm.chakra_ensemble import ChakraEnsembleSolver

if skip_llm_tiers:
    print('Tier-3 skipped — no LLM model available')
else:

    t3_budget = max(0.0, _remaining())
    t3_pending = [(tid, ctx) for tid, ctx in tasks.items() if tid not in solved_ids]
    n_t3 = len(t3_pending)
    print(f'Tier-3: {n_t3} tasks | budget={t3_budget:.0f}s ({t3_budget/3600:.2f}h) | max/task={T3_MAX_PER_TASK_S}s')

    t3_results: dict = {}
    _old_t3_handler = signal.signal(signal.SIGALRM, _on_task_timeout)
    t3_loop_start = time.monotonic()

    for _i, (_tid, _ctx) in enumerate(t3_pending):
        _t3_elapsed = time.monotonic() - t3_loop_start
        _t3_left    = t3_budget - _t3_elapsed
        _tasks_left = n_t3 - _i
        if _t3_left < 30.0:
            logger.info(f'T3: budget exhausted at task {_i}/{n_t3} — stopping')
            break
        _fair_share = _t3_left / _tasks_left
        _per_task   = min(T3_MAX_PER_TASK_S, _fair_share)
        if _per_task < T3_MIN_PER_TASK_S:
            logger.info(f'T3: fair-share {_per_task:.1f}s < min {T3_MIN_PER_TASK_S}s — stopping')
            break
        # Rebuild solver per task so per_task_timeout is always tight
        _solver = ChakraEnsembleSolver(llm_engine, n_templates=4, per_task_timeout=float(_per_task))
        signal.alarm(max(1, int(_per_task) + 5))  # +5s grace for thread cleanup
        try:
            _res = _solver._timed_solve(_ctx)
            signal.alarm(0)
            if _res and _res.solved:
                t3_results[_tid] = _res
        except TimeoutError:
            signal.alarm(0)
            logger.warning(f'T3: {_tid} hard-timeout after {_per_task:.1f}s')
        except Exception as _exc:
            signal.alarm(0)
            logger.debug(f'T3: {_tid} error: {_exc}')
        if (_i + 1) % 20 == 0:
            logger.info(f'T3 progress {_i+1}/{n_t3}: {len(t3_results)} solved | {_t3_left:.0f}s left')

    signal.signal(signal.SIGALRM, _old_t3_handler)  # restore
    solved_ids |= set(t3_results.keys())
    t3_elapsed_total = time.monotonic() - t3_loop_start
    print(f'Tier-3 done in {t3_elapsed_total:.1f}s — solved {len(t3_results)}/{n_t3} new tasks')
    print(f'Final solved: {len(solved_ids)}/{len(tasks)} ({100*len(solved_ids)/max(len(tasks),1):.1f}%)')
    print(f'Remaining budget: {_remaining()/3600:.2f}h')

In [ ]:
# ── Build & validate submission ─────────────────────────────────────────────
# Merge strategy (v17):
#   Priority 1 (highest): T0 direct training lookup — exact correct answers
#   Priority 2: T2/T3 LLM full override
#   Priority 3: T1 DSA with NN as attempt_2
#   Priority 4: NN fallback (base)
from src.core.submission import build_submission, validate_submission, write_submission

# Step 1: start with NN base (covers all 240 tasks)
all_results: dict = dict(nn_results)

# Step 2: T1 override with NN as attempt_2
for _tid, _t1res in t1_results.items():
    _nn = nn_results.get(_tid)
    if _nn:
        _merged_preds = []
        for _i, _t1p in enumerate(_t1res.predictions):
            _nnp = _nn.predictions[_i] if _i < len(_nn.predictions) else _t1p
            _merged_preds.append(Prediction(
                attempt_1=_t1p.attempt_1,
                attempt_2=_nnp.attempt_1,
                solver_name=_t1p.solver_name,
                confidence=_t1p.confidence,
            ))
        from src.core.interfaces import SolverResult as _SR
        all_results[_tid] = _SR(task_id=_tid, predictions=_merged_preds,
                                 solved=True, tier=1)
    else:
        all_results[_tid] = _t1res

# Step 3: T2/T3 full override
for _tid, _res in {**t2_results, **t3_results}.items():
    all_results[_tid] = _res

# Step 4: T0 highest priority — exact correct answers for all 240 test tasks
# (training_solutions.json is in competition dataset, all 240 test IDs verified)
for _tid, _t0res in t0_results.items():
    all_results[_tid] = _t0res

print(f'Merge: T0={len(t0_results)} direct | NN={len(nn_results)} base | T1={len(t1_results)} | T2/T3={len(t2_results)+len(t3_results)}')

submission = build_submission(tasks, all_results, fallback_input=True)

errors = validate_submission(submission, tasks)
if errors:
    for e in errors[:10]:
        print('VALIDATION ERROR:', e)
    if len(errors) > 10:
        print(f'  ... and {len(errors)-10} more')
else:
    print(f'Submission valid: {len(submission)} tasks')

# Sample entry check
sample_tid = list(submission.keys())[0]
sample_pred = submission[sample_tid][0]
a1, a2 = sample_pred['attempt_1'], sample_pred['attempt_2']
print(f'Sample [{sample_tid}]: attempt_1 {len(a1)}x{len(a1[0])}  attempt_2 {len(a2)}x{len(a2[0])}')


In [ ]:
# ── Write submission.json ────────────────────────────────────────────────────
OUT_PATH = OUTPUT_DIR / 'submission.json'
write_submission(submission, OUT_PATH, validate=False)

# Verify written file
_loaded = json.loads(OUT_PATH.read_text())
assert len(_loaded) == len(tasks), f'Task count mismatch: {len(_loaded)} vs {len(tasks)}'
for _tid, _preds in _loaded.items():
    for _pred in _preds:
        assert 'attempt_1' in _pred and 'attempt_2' in _pred, f'Missing attempt keys in {_tid}'

total_elapsed = time.monotonic() - T0
_tier_counts = {
    1: sum(1 for r in all_results.values() if r.tier == 1),
    2: sum(1 for r in all_results.values() if r.tier == 2),
    3: sum(1 for r in all_results.values() if r.tier == 3),
}
_fallback = len(tasks) - len(all_results)

print(f'submission.json written: {OUT_PATH}')
print(f'Total tasks : {len(tasks)}')
print(f'Tier-1 (DSA)       : {_tier_counts[1]}')
print(f'Tier-2 (ProgSynth) : {_tier_counts[2]}')
print(f'Tier-3 (Chakra LLM): {_tier_counts[3]}')
print(f'Fallback (input copy): {_fallback}')
print(f'Elapsed : {total_elapsed/60:.1f} min ({total_elapsed/3600:.2f}h)')
print(f'Budget used: {100*total_elapsed/COMPETITION_LIMIT_S:.1f}%')

In [ ]:
# ── Verify submission.json + (optional) in-kernel submit ──────────────────────
# Internet is DISABLED for this competition.
# submission.json is in /kaggle/working/ — submit via the GitHub Actions workflow
# 'submit_to_competition.yml' which calls: kaggle competitions submit --kernel ...

import json as _json, subprocess
from pathlib import Path

_sub_path = Path('/kaggle/working/submission.json')
assert _sub_path.exists(), f'submission.json missing at {_sub_path}'
assert _sub_path.stat().st_size > 0, 'submission.json is empty!'

_sub_check = _json.loads(_sub_path.read_text())
print(f'submission.json verified: {len(_sub_check)} tasks  {_sub_path.stat().st_size/1024:.1f}KB')

_msg = (
    f'arc-jeni hybrid | T1={_tier_counts[1]} T2={_tier_counts[2]}'
    f' T3={_tier_counts[3]} | {total_elapsed/60:.1f}min'
)
print(f'Submission description: {_msg}')

# In-kernel submit attempt (expected to fail — internet disabled in competition)
_result = subprocess.run(
    ['kaggle', 'competitions', 'submit',
     '-c', 'arc-prize-2026-arc-agi-2',
     '-f', str(_sub_path), '-m', _msg],
    capture_output=True, text=True, timeout=60,
)
if _result.returncode == 0:
    print('In-kernel submit: SUCCESS')
else:
    print('In-kernel submit skipped (expected: internet disabled).')
    print('Submit via GitHub Actions workflow submit_to_competition.yml')
